# 📄➡️📝 Document → Markdown **box**: Marker + vision LLMs on Kaggle's 2×T4, driven from your PC through Cloudflare

Converts **PDF (digital and scanned), images, Word (.doc/.docx/.rtf/.odt), PowerPoint (.ppt/.pptx/.odp), Excel (.xls/.xlsx/.ods), HTML, EPUB, DjVu, TXT/MD/CSV**, in any language, into Markdown that keeps **text, headings, tables, equations (LaTeX), code, figures and the information inside figures**.

| Step | What runs | Where |
|---|---|---|
| Layout, OCR, reading order | [Marker](https://github.com/datalab-to/marker) + Surya 2.5 VLM | Surya on both T4s, Marker workers on the 4 CPUs |
| Tables (incl. across pages), equations, inline math, forms, headings, handwriting | Marker's LLM passes with **Qwen3-VL-8B** | one llama-server per T4, next to Surya |
| Figures | re-cropped from the page (complete labels, joined pieces, 300 DPI); images the layout model missed are recovered from the PDF; the LLM writes out each figure's content under it | workers + LLM |
| Office / DjVu | LibreOffice → PDF (keeps Word equations and layouts), ddjvu → PDF | workers |
| Check | every word of each PDF page's text layer is looked up in the Markdown; pages under 90% are reported | workers |

```
your PC                                 Cloudflare quick tunnel             Kaggle (this notebook, 2×T4, 4 vCPUs)
md_client.py ./docs ─────────────────▶  https://xxxx.trycloudflare.com ──▶  API ─▶ task queue (long PDFs split in page chunks)
  biggest files to the box,                                                  │
  smallest + stolen page chunks to                                           ▼
  Marker workers on your PC (optional) ◀── /vlm /llm proxies ──  4 warm Marker workers (CPU) ──┬─▶ Surya VLM  (llama-server GPU0, GPU1)
        ◀──── <name>/<name>.md + images + report.json ─────────────────────────────────────────┴─▶ vision LLM (llama-server GPU0, GPU1)
```

### One-time setup (Kaggle)
1. **Settings → Accelerator → `GPU T4 ×2`**, **Settings → Internet → On**.
2. **Add-ons → Secrets** (tick *attached*): `BOX_TOKEN` (16+ characters; the same one goes into `.box_token` on your PC), optional `HF_TOKEN`, optional `GOOGLE_API_KEY` (Gemini; only used if the local LLM cannot start, and it sends page content to Google).
3. **Save Version → Save & Run All (Commit)** and wait for **BOX READY** (about 10 minutes: llama.cpp's released CUDA build and the models are downloaded, LibreOffice is installed, every GPU server must pass a test request).
4. The box stops by itself after `IDLE_STOP_MIN` minutes without jobs, to save the 30 GPU-hours/week.

### On your PC
Put `md_client.py` (with the `dev/` folder next to it) and a file `.box_token` in one folder, then:
```
python md_client.py book.pdf                  # -> markdown_out/book/book.md + images + report.json
python md_client.py ./folder --out ./md       # every document in the folder (sub-folders kept)
python md_client.py big.pdf                   # long documents: batches of pages, each saved on your PC before the next (re-run resumes)
python md_client.py scan.pdf --mode vllm      # scans / illustrated books: Surya reads every page and finds every figure
python md_client.py big.pdf --no-llm          # much faster, less exact tables/equations, no figure text
python md_client.py --health                  # GPUs, workers, models, LibreOffice, idle time
```
Settings are in section 1. The `docs/` folder of the repository explains every option (`docs/configuration.md`) and what each log line means (`docs/troubleshooting.md`).

## 1. Configuration

In [ ]:
BOX_NAME      = "markdown"
SESSION_HOURS = 8.5         # hard stop (Kaggle kills sessions at 12 h; GPU quota is 30 h/week)
IDLE_STOP_MIN = 60          # stop when no job has been active for this long, to save GPU quota (0 = never)
API_PORT      = 8000
ANNOUNCE      = True        # post the temporary tunnel address (signed, no secrets) to a private ntfy.sh topic so your PC finds it
MAX_UPLOAD_GB = 3
RESULTS_DIR   = "/kaggle/tmp/jobs"   # "jobs" would keep every result in the saved notebook output

# ---- conversion engine
MARKER_SOURCE = "git+https://github.com/datalab-to/marker.git@b578dc88412984b7531bd4a5b4886eb2d2c58da1"   # your PC must use the same version
OFFICE_VIA_LIBREOFFICE = True   # LibreOffice: .doc/.ppt/.xls/.odt/.rtf... and .docx/.pptx with their equations and layouts

# ---- GPU programs (llama.cpp)
LLAMA_PREBUILT = True       # download llama.cpp's released CUDA build (~30 s) instead of compiling it (~15 min)
LLAMA_CPP_REF = "master"    # llama.cpp release / git ref (pin one if a new release ever breaks)
USE_VLM       = True        # Surya VLM on the GPUs; False = CPU-only (fast / no_ocr modes)
VLM_SLOTS_PER_GPU = 8       # parallel Surya requests per GPU (halved automatically if a GPU runs out of memory)
VLM_LB_PORT   = 8090        # load balancer in front of the Surya servers

# ---- vision LLM for Marker's quality passes (tables, equations, forms, headings) and figure content
LLM_LOCAL     = True        # False = only Gemini (GOOGLE_API_KEY secret), or no LLM passes
LLM_NAME      = "qwen3-vl-8b"
LLM_REPO      = "unsloth/Qwen3-VL-8B-Instruct-GGUF"   # one server per T4, next to Surya
LLM_FILES     = ["Qwen3-VL-8B-Instruct-Q6_K.gguf", "Qwen3-VL-8B-Instruct-Q4_K_M.gguf"]   # best first; Q4 only if Q6 cannot start
LLM_MMPROJ    = "mmproj-F16.gguf"   # vision encoder (T4s have no bf16)
_FAST  = "-fa on -ctk q8_0 -ctv q8_0"            # flash attention + 8-bit KV cache: half the context memory, more slots
_NGRAM = "--spec-type ngram-mod"                 # speculative decoding from the prompt itself: table/equation fixes mostly copy it
LLM_VARIANTS  = [(f"{_FAST} {_NGRAM}", 5), (f"{_FAST} {_NGRAM}", 4), (_FAST, 4), ("", 2)]   # (llama-server flags, parallel slots), fastest first:
                            # more slots = more requests decoded together (more throughput), if the T4 still has room (LLM_MIN_FREE_MB).
                            # Measured on a T4 next to Surya (8 slots): 4 slots leave 1731 MiB, 6 leave 431 (~650 MiB a slot) -> 5 leave ~1080
LLM_MIN_FREE_MB = 700       # a variant must leave this much GPU memory free (Surya shares the GPU), else the next one is tried
LLM_HOST_CACHE_MB = 2048    # host RAM each vision-LLM server may use to keep processed prompts (page images) a slot gave up: a page's
                            # review, audit and second looks reuse its image (2.8 s of GPU time saved per call). Bounded: the default
                            # (8 GB per server) once filled the box. 0 = off. Surya's servers keep none (their crops never repeat)
LLM_MIN_TOK_S = 5.0         # a variant must answer the test exactly and generate at least this fast once warm
LLM_CTX_PER_SLOT     = 8192
LLM_IMAGE_MAX_TOKENS = 1536 # ~1250x1250 px per image: enough for table/equation crops, keeps requests fast
LLM_LB_PORT   = 8190        # load balancer in front of the two LLM servers

# ---- defaults for every job (the client can override most of them per run)
JOB_DEFAULTS = dict(
    WORKERS=0,              # conversion processes; 0 = up to WORKERS_PER_CPU per vCPU, as many as the free RAM holds (MEM_PER_WORKER_GB each)
    WORKERS_PER_CPU=1.5,    # a worker waiting on the GPU models leaves its CPU free: with spare RAM, 4 vCPUs run 6 workers
    MEM_START_GB=2.5,       # RAM a piece grows by in its first minute: new pieces start only while free RAM covers them above MEM_PAUSE_FREE_GB
    MEM_PER_WORKER_GB=3.0,  # RAM one Marker worker may need (measured ~2 GB at its peak in the suite runs; 3.5 left 1 worker unused)
    MEM_PAUSE_FREE_GB=4.0,  # less free RAM than this: no new pieces, except one if nothing is running (the session restarts if RAM is full)
    MEM_RESUME_FREE_GB=6.0, # ... more than this: all workers again
    MEM_GPU_SERVER_MAX_GB=5.0,  # ... and a GPU server (llama-server) bigger than this is restarted instead (the watchdog brings it back);
                                #     an LLM server is ~2.3 GB + up to LLM_HOST_CACHE_MB of prompt cache
    MEM_KILL_FREE_GB=2.0,   # ... less than this: the biggest busy worker is stopped and its piece converted again (checked every 0.5 s)
    MODE="auto",            # auto = balanced when Surya runs, else fast (client: --mode)
    OFFICE="pdf",           # pdf = LibreOffice renders office files to PDF first; native = Marker reads .docx/.pptx itself (--office)
    USE_LLM=True,           # the LLM passes on every job when an LLM runs (client: --no-llm = faster)
    REDO_INLINE_MATH=None,  # None = as the quality says (accurate: yes); True/False forces Marker's inline-maths rewrite
    QUALITY="balanced",     # fast / balanced / accurate (client: --quality). fast: no page review, light figure text, no Marker LLM steps;
                            # balanced: + Marker's LLM steps, page review, figures cropped to themselves, flow diagrams as Mermaid;
                            # accurate: + review section by section, line-by-line audit with every fix double-checked (slowest)
    DESCRIBE_FIGURES=None,  # None = as the quality says; True/False forces it (client: --no-describe)
    PAGE_REVIEW=None,       # None = as the quality says; True/False forces it (client: --no-review)
    SPLIT_MIN_PAGES=8,      # PDFs with at least this many pages are split into page chunks shared by all workers (and your PC); 0 = never
    CHUNK_MAX_PAGES=24,     # pages per chunk at most: smaller chunks = less RAM per worker
    DOC_TIMEOUT_MIN=90,     # a document (or chunk) that takes longer is stopped and reported as failed
    TIME_BUDGET_H=7.0,
)

## 2. Install Marker

In [ ]:
import subprocess, sys, os, shutil
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode: raise RuntimeError(f"{cmd}\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}")
    return r
# weasyprint (HTML / EPUB / DOCX rendering) needs pango
sh("apt-get update -qq", check=False)
sh("apt-get install -y -qq libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz-subset0 fonts-dejavu-core", check=False)
# fonts for every script (Bangla, Devanagari, Arabic, Thai, CJK, Ethiopic, emoji...): rendering HTML/EPUB/DOCX or office
# files to PDF without them turns that text into empty boxes
sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends fonts-noto-core fonts-noto-extra fonts-noto-cjk "
   "fonts-noto-color-emoji", check=False)
if OFFICE_VIA_LIBREOFFICE:
    # Carlito/Caladea are metric clones of Calibri/Cambria (same line breaks as Word)
    r = sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends libreoffice-writer libreoffice-impress "
           "libreoffice-calc fonts-crosextra-carlito fonts-crosextra-caladea", check=False)
sh("apt-get install -y -qq --no-install-recommends djvulibre-bin", check=False)      # ddjvu: DjVu -> PDF
print("ddjvu (DjVu):", shutil.which("ddjvu") or "NOT installed; .djvu files will fail")
if OFFICE_VIA_LIBREOFFICE:
    print("LibreOffice:", shutil.which("soffice") or "NOT installed; only pdf/images/docx/pptx/xlsx/html/epub/txt will work\n" + r.stderr[-800:])
sh(f'{sys.executable} -m pip install -q --no-warn-conflicts "marker-pdf[full] @ {MARKER_SOURCE}" fastapi uvicorn python-multipart httpx' if MARKER_SOURCE.startswith("git+")
   else f'{sys.executable} -m pip install -q --no-warn-conflicts "{MARKER_SOURCE}" fastapi uvicorn python-multipart httpx')
sh("mkdir -p mdbox logs cache")
print(sh(f"{sys.executable} -m pip show marker-pdf | head -2").stdout)

## 3. GPU programs: llama.cpp (released CUDA build; compiled only as a fallback) and the Surya 2.5 model
The vision LLM starts downloading here too, in the background. If anything fails, the box still runs CPU-only (`fast` mode).

In [ ]:
import glob, json, re, threading, time
def _prefetch_llm():                 # the vision LLM downloads while LibreOffice installs and Surya starts
    try:
        from huggingface_hub import hf_hub_download
        for x in (LLM_FILES[0], LLM_MMPROJ): hf_hub_download(LLM_REPO, x)
    except Exception as e:
        print("LLM prefetch:", str(e)[:200])
if LLM_LOCAL: threading.Thread(target=_prefetch_llm, daemon=True).start()
from pathlib import Path
VLM_BIN, GGUF = None, None
def cuda_major(text, pat):
    m = re.search(pat, text or "")
    return int(m.group(1)) if m else None
DRIVER_CUDA = cuda_major(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout, r"CUDA Version:\s*(\d+)")
def nvcc_major(nvcc):
    return cuda_major(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout, r"release (\d+)")
def find_nvcc():
    found = [c for c in dict.fromkeys([shutil.which("nvcc")] + sorted(glob.glob("/usr/local/cuda*/bin/nvcc"), reverse=True)) if c and os.path.exists(c)]
    ok = [c for c in found if DRIVER_CUDA is None or (nvcc_major(c) or 0) <= DRIVER_CUDA]   # the driver must be able to run what we build
    print("driver CUDA:", DRIVER_CUDA, "| toolkits:", {c: nvcc_major(c) for c in found})
    return ok[0] if ok else None
def find_libcuda(nvcc):
    root = os.path.dirname(os.path.dirname(os.path.realpath(nvcc)))
    for pat in ("/usr/lib/x86_64-linux-gnu/libcuda.so*", "/usr/lib64-nvidia/libcuda.so*", "/usr/local/nvidia/lib64/libcuda.so*",
                f"{root}/lib64/stubs/libcuda.so", f"{root}/targets/x86_64-linux/lib/stubs/libcuda.so"):
        hits = sorted(glob.glob(pat))
        if hits: return hits[0]
m = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)
DRIVER_CUDA_V = (int(m.group(1)), int(m.group(2))) if m else None
TMPD = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else Path("/tmp")
VLM_ENV = f"CUDA_CACHE_PATH={TMPD}/cuda-cache CUDA_CACHE_MAXSIZE=4294967296 CUDA_MODULE_LOADING=LAZY"
def prebuilt_llama():
    """llama.cpp's released ubuntu CUDA 12.8 x64 build + the CUDA 12 runtime libraries it links (from pip if present)"""
    if not DRIVER_CUDA_V or DRIVER_CUDA_V < (12, 8):
        print("driver CUDA", DRIVER_CUDA_V, "is older than 12.8: cannot run the prebuilt build"); return None
    import urllib.request
    api = "https://api.github.com/repos/ggml-org/llama.cpp/releases" + ("?per_page=15" if LLAMA_CPP_REF == "master" else f"/tags/{LLAMA_CPP_REF}")
    rels = json.load(urllib.request.urlopen(urllib.request.Request(api, headers={"User-Agent": "box"}), timeout=30))
    for r in (rels if isinstance(rels, list) else [rels]):
        assets = {a["name"]: a["browser_download_url"] for a in r.get("assets", [])}
        main = f"llama-{r['tag_name']}-bin-ubuntu-cuda-12.8-x64.tar.gz"
        if main in assets:
            break
    else:
        print("no ubuntu-cuda-12.8-x64 asset in the recent llama.cpp releases"); return None
    d = TMPD / "llama-bin"
    sh(f"rm -rf {d} && mkdir -p {d} && curl -fsSL {assets[main]} | tar xz -C {d} --strip-components=1")
    need = ["libcudart.so.12", "libcublas.so.12", "libcublasLt.so.12"]
    import site
    for lib in need:                                  # torch's pip CUDA 12 libraries, if Kaggle has them
        hits = [h for sp in site.getsitepackages() for h in glob.glob(f"{sp}/nvidia/*/lib/{lib}")]
        if hits and not (d / lib).exists(): os.symlink(hits[0], d / lib)
    if any(not (d / l).exists() for l in need) and "cudart-" + main in assets:
        sh(f"curl -fsSL {assets['cudart-' + main]} | tar xz -C {d} --strip-components=1")
    missing = [l for l in need if not (d / l).exists()]
    if missing or not (d / "llama-server").exists(): raise RuntimeError(f"prebuilt llama.cpp incomplete, missing {missing or ['llama-server']}")
    print("prebuilt llama.cpp", r["tag_name"])
    return str(d / "llama-server")
def build_llama():
    cached = [p for p in glob.glob("/kaggle/input/*/llama-server") + glob.glob("/kaggle/input/*/*/llama-server") + ["cache/llama-server"] if os.path.exists(p)]
    if cached:
        os.chmod(cached[0], 0o755); return cached[0]
    if LLAMA_PREBUILT:
        try:
            p = prebuilt_llama()
            if p: return p
        except Exception as e:
            print("prebuilt llama.cpp failed, compiling instead:", str(e)[:500])
    nvcc = find_nvcc()
    if not nvcc:
        v = f"{DRIVER_CUDA}-{8 if DRIVER_CUDA == 12 else 0}" if DRIVER_CUDA else "12-4"
        print(f"no CUDA compiler the driver can run; installing cuda-nvcc-{v}")
        sh(f"apt-get install -y -qq cuda-nvcc-{v} cuda-cudart-dev-{v} libcublas-dev-{v}", check=False)
        nvcc = find_nvcc()
    if not nvcc: raise RuntimeError("no CUDA compiler available")
    t0 = time.time(); print("building llama-server with", nvcc, "(about 10 minutes the first time)")
    sh("rm -rf /tmp/llama.cpp && git clone -q --depth 1 " + ("" if LLAMA_CPP_REF == "master" else f"--branch {LLAMA_CPP_REF} ") + "https://github.com/ggml-org/llama.cpp /tmp/llama.cpp")
    flags = f"-DGGML_CUDA=ON -DCMAKE_CUDA_COMPILER={nvcc} -DCMAKE_CUDA_ARCHITECTURES=75 -DBUILD_SHARED_LIBS=OFF -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DCMAKE_BUILD_TYPE=Release"
    libcuda = find_libcuda(nvcc)
    if libcuda: flags += f" -DCUDA_cuda_driver_LIBRARY={libcuda}"
    print("libcuda:", libcuda)
    r = sh(f"cd /tmp/llama.cpp && cmake -B build {flags} > /tmp/cmake.log 2>&1 && cmake --build build --config Release -j{os.cpu_count()} --target llama-server >> /tmp/cmake.log 2>&1", check=False)
    if r.returncode or not os.path.exists("/tmp/llama.cpp/build/bin/llama-server"):
        raise RuntimeError("llama.cpp build failed:\n" + Path("/tmp/cmake.log").read_text()[-2500:])
    shutil.copy("/tmp/llama.cpp/build/bin/llama-server", "cache/llama-server"); os.chmod("cache/llama-server", 0o755)
    print(f"built in {time.time()-t0:.0f}s")
    return "cache/llama-server"
if USE_VLM:
    try:
        VLM_BIN = os.path.abspath(build_llama())
        from huggingface_hub import hf_hub_download
        from surya.settings import settings as _ss
        GGUF = [hf_hub_download(_ss.SURYA_GGUF_REPO, f) for f in (_ss.SURYA_GGUF_MODEL_FILE, _ss.SURYA_GGUF_MMPROJ_FILE)]
        VLM_ALIAS = _ss.SURYA_MODEL_CHECKPOINT
        print("llama-server:", VLM_BIN, "| model files:", [os.path.basename(g) for g in GGUF])
    except Exception as e:
        VLM_BIN = GGUF = None
        print("!!! VLM unavailable, continuing CPU-only (fast mode):", str(e)[:2500])

## 4. Pre-download Marker's CPU models and font (so the first job is not slow and the workers do not race)

In [ ]:
r = subprocess.run([sys.executable, "-c", """
from marker.util import download_font; download_font()
from marker.models import create_model_dict
m = create_model_dict(); m['indra_model'].engine; print('Indra ONNX models + font ready')
"""], capture_output=True, text=True)
print(r.stdout[-300:] or r.stderr[-1500:])
if r.returncode: raise RuntimeError("could not prepare Marker models (internet/HF_TOKEN?)")

## 5. Box files (these cells only write files)
`convert.py` routes each file type (LibreOffice first for office files, direct conversion for txt/md/csv); `lb.py` balances VLM requests over the GPU servers; `worker.py` is a warm Marker worker; `pipeline.py` runs one job; `server.py` is the API.

In [ ]:
%%writefile mdbox/convert.py
"""Which path each file type takes, and the conversions that need no model (standard library only).
Imported by worker.py (does the work) and pipeline.py (the list of accepted extensions); tested in dev/test_pipeline.py.

  marker  : Marker reads the file itself (PDF, images, HTML, EPUB, and DOCX/PPTX/XLSX in OFFICE="native" mode)
  soffice : LibreOffice converts it first, then Marker reads the result
            - documents and slides -> PDF: Word equations (OMML), slide layouts, text boxes, SmartArt and embedded figures
              survive as rendered pages, which the layout model and the VLM turn into LaTeX, tables and figure images.
              (Marker's own DOCX path goes through mammoth, which drops equations and floating shapes.)
            - spreadsheets -> XLSX: Marker's spreadsheet path keeps every sheet as a table
  ddjvu   : DjVu is rendered to an image PDF by djvulibre's ddjvu, then Marker OCRs it like a scan
  text    : plain text, Markdown, CSV/TSV are turned into Markdown directly"""
import csv, io, json, os, re, shutil, signal, subprocess
from pathlib import Path

MARKER_EXT = (".pdf", ".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff", ".gif", ".bmp", ".html", ".htm", ".epub")
OOXML_NATIVE = (".docx", ".pptx", ".xlsx")                  # Marker can read these without LibreOffice
OFFICE_TO_PDF = (".doc", ".docx", ".docm", ".dot", ".dotx", ".odt", ".ott", ".fodt", ".rtf", ".wpd", ".wps",
                 ".ppt", ".pptx", ".pptm", ".pps", ".ppsx", ".pot", ".potx", ".odp", ".otp", ".fodp", ".odg")
SHEET_TO_XLSX = (".xls", ".xlsm", ".xlsb", ".ods", ".ots", ".fods")
TEXT_EXT = (".txt", ".md", ".markdown", ".csv", ".tsv")
DJVU_EXT = (".djvu", ".djv")
ALL_EXT = tuple(sorted(set(MARKER_EXT + OOXML_NATIVE + OFFICE_TO_PDF + SHEET_TO_XLSX + TEXT_EXT + DJVU_EXT)))
OFFICE_MODES = ("pdf", "native")

SOFFICE = shutil.which("soffice") or shutil.which("libreoffice")
DDJVU = shutil.which("ddjvu")


class Unsupported(Exception):
    pass


def route(name, office_mode="pdf", has_soffice=None, has_ddjvu=None):
    """-> (action, target_format): ("marker", None) | ("soffice", "pdf"|"xlsx") | ("ddjvu", "pdf") | ("text", None)"""
    has_soffice = bool(SOFFICE) if has_soffice is None else has_soffice
    has_ddjvu = bool(DDJVU) if has_ddjvu is None else has_ddjvu
    ext = Path(name).suffix.lower()
    if ext in TEXT_EXT:
        return "text", None
    if ext in DJVU_EXT:
        if not has_ddjvu:
            raise Unsupported(f"{ext} files need ddjvu (apt package djvulibre-bin), which is not installed on this box")
        return "ddjvu", "pdf"
    if ext in MARKER_EXT:
        return "marker", None
    if ext == ".xlsx":
        return "marker", None                                # Marker's own XLSX path is already table-faithful
    if ext in OOXML_NATIVE and (office_mode == "native" or not has_soffice):
        return "marker", None
    if ext in OFFICE_TO_PDF + SHEET_TO_XLSX:
        if not has_soffice:
            raise Unsupported(f"{ext} files need LibreOffice, which is not installed on this box (OFFICE_VIA_LIBREOFFICE in the notebook)")
        return "soffice", "xlsx" if ext in SHEET_TO_XLSX else "pdf"
    raise Unsupported(f"unsupported file type {ext!r}")


def run_tool(cmd, out, what, timeout):
    """runs a converter in its own process group (killed whole on timeout) and checks that it wrote `out`"""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
    try:
        log = p.communicate(timeout=timeout)[0].decode("utf-8", "replace")
    except subprocess.TimeoutExpired:
        os.killpg(p.pid, signal.SIGKILL); p.wait()            # soffice forks soffice.bin: kill the whole group
        raise RuntimeError(f"{what} took longer than {timeout} s")
    if not Path(out).is_file() or Path(out).stat().st_size == 0:
        raise RuntimeError(f"{what} failed (exit {p.returncode}): {log.strip()[-400:]}")
    return Path(out)


def soffice(src, fmt, outdir, profile_dir, timeout=600):
    """converts src with headless LibreOffice; every worker passes its own profile_dir so several can run at once"""
    if not SOFFICE:
        raise Unsupported("LibreOffice is not installed on this box")
    src, outdir = Path(src), Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    cmd = [SOFFICE, "--headless", "--norestore", "--nolockcheck", f"-env:UserInstallation={Path(profile_dir).resolve().as_uri()}",
           "--convert-to", fmt, "--outdir", str(outdir), str(src)]
    return run_tool(cmd, outdir / f"{src.stem}.{fmt}", f"LibreOffice converting {src.name} to {fmt}", timeout)


def djvu_to_pdf(src, outdir, timeout=1800):
    """DjVu -> image PDF (ddjvu drops the hidden text layer; Marker OCRs the pages)"""
    if not DDJVU:
        raise Unsupported("ddjvu (djvulibre-bin) is not installed on this box")
    src, outdir = Path(src), Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    out = outdir / f"{src.stem}.pdf"
    return run_tool([DDJVU, "-format=pdf", "-quality=85", str(src), str(out)], out, f"ddjvu converting {src.name}", timeout)


CODEPAGES = ("gbk", "shift_jis", "big5", "euc_kr", "cp1251", "cp1256", "cp1253", "cp1255", "cp874", "cp1250", "cp1252")


def _common(c):
    """a character people write often: full-width kana, Hangul, frequently used hanzi (GB2312 level 1, Big5 A440-C67E),
    other letters; half-width katakana and rare ideographs are what a wrong code page produces"""
    o = ord(c)
    if 0xFF61 <= o <= 0xFF9F:
        return False
    if 0x3040 <= o <= 0x30FF or 0xAC00 <= o <= 0xD7A3 or not (0x3400 <= o <= 0x9FFF or 0xF900 <= o <= 0xFAFF):
        return True
    for enc, lo, hi in (("gb2312", 0xB0A1, 0xD7FE), ("big5", 0xA440, 0xC67E)):
        try:
            b = c.encode(enc)
            if len(b) == 2 and lo <= int.from_bytes(b, "big") <= hi:
                return True
        except UnicodeEncodeError:
            pass
    return False


def _plausible(text):
    """how much a decoded text looks like real writing (higher is better), None if it can't be"""
    import unicodedata
    if any(unicodedata.category(c) in ("Cc", "Co", "Cn") and c not in "\t\n\r\f" for c in text):
        return None
    letters = [c for c in text if c.isalpha() and not c.isascii()]
    if not letters:
        return 0.0
    scripts = {unicodedata.name(c, "?").split()[0] for c in letters}
    scripts = {"CJK" if x in ("CJK", "HIRAGANA", "KATAKANA", "HALFWIDTH") else x for x in scripts}
    if len(scripts) > 2:                                   # real text mixes at most two (Japanese, Korean with Hanja...)
        return None
    flips = sum(1 for x, y in zip(text, text[1:]) if x.islower() and y.isupper() and not x.isascii())   # "КЮСнС"
    phonetic = sum(1 for c in letters if 0x3040 <= ord(c) <= 0x30FF or 0xAC00 <= ord(c) <= 0xD7A3)       # kana, Hangul
    words = re.findall(r"\w+", text)
    mixed = sum(1 for w in words if re.search(r"[a-zA-Z]", w) and any(c.isalpha() and not c.isascii() and
                                                                    not unicodedata.name(c, "").startswith("LATIN") for c in w))   # "cafй"
    bonus = 0.5 * phonetic if len(scripts) == 1 else 0     # kana / Hangul text: a wrong code page rarely gives whole words of it
    return (sum(_common(c) for c in letters) - flips - mixed + bonus) / len(letters) - 0.5 * (len(scripts) - 1)


def guess_decode(raw, sample=20000):
    """bytes in an unknown legacy code page -> text: every candidate that decodes without error is scored on its start"""
    best, score = None, None
    for k, enc in enumerate((("utf-16-le", "utf-16-be") if len(raw) % 2 == 0 else ()) + CODEPAGES):   # UTF-16 without a BOM too
        try:
            text = raw.decode(enc)
        except UnicodeDecodeError:
            continue
        if enc.startswith("utf-16") and not re.search(r"[ \n]", text):
            continue                                       # real UTF-16 text has spaces or line breaks; 4 bytes of "café" don't
        sc = _plausible(text[:sample])
        if sc is not None and (score is None or sc - k * 1e-3 > score):
            best, score = text, sc - k * 1e-3
    return best


def read_text(path):
    """text in any encoding: UTF-8/16/32 (with or without BOM), else the most plausible legacy code page (GBK, Big5,
    Shift-JIS, EUC-KR, Windows-1251/1256/...). Decoding as cp1252 alone turned a Chinese GBK file into "ÖÐÎÄ" """
    raw = Path(path).read_bytes()
    for bom, enc in ((b"\xff\xfe\x00\x00", "utf-32"), (b"\x00\x00\xfe\xff", "utf-32"), (b"\xff\xfe", "utf-16"), (b"\xfe\xff", "utf-16")):
        if raw.startswith(bom):
            return raw.decode(enc, errors="replace")
    try:
        return raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        pass
    return guess_decode(raw) or raw.decode("cp1252", errors="replace")


def md_cell(s):
    return s.replace("\\", "\\\\").replace("|", "\\|").replace("\r\n", "\n").replace("\n", "<br>").strip()


def table_to_md(text, delimiter=None):
    """CSV/TSV text -> one Markdown table (first row = header). delimiter None = sniff among , ; | TAB"""
    if not text.strip():
        return ""
    if delimiter is None:
        try:
            delimiter = csv.Sniffer().sniff(text[:20000], delimiters=",;|\t").delimiter
        except csv.Error:
            delimiter = ","
    rows = [r for r in csv.reader(io.StringIO(text), delimiter=delimiter) if any(c.strip() for c in r)]
    if not rows:
        return ""
    width = max(len(r) for r in rows)
    rows = [[md_cell(c) for c in r] + [""] * (width - len(r)) for r in rows]
    lines = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * width]
    lines += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(lines) + "\n"


def text_to_md(src, out_dir, stem):
    """writes <out_dir>/<stem>.md for .txt/.md/.csv/.tsv; returns the worker result fields"""
    src, out_dir = Path(src), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ext, text = src.suffix.lower(), read_text(src)
    if ext in (".csv", ".tsv"):
        md = table_to_md(text, "\t" if ext == ".tsv" else None)
    else:
        md = text.replace("\r\n", "\n")
        if not md.endswith("\n"):
            md += "\n"
    (out_dir / f"{stem}.md").write_text(md, encoding="utf-8")
    return {"pages": None, "files": [f"{stem}.md"], "via": "text"}


# ----------------------------------------------------------------- old Bangla PDFs (Bijoy / SutonnyMJ fonts)
# Bangla typeset with Bijoy fonts (SutonnyMJ, ...) maps Bangla glyphs onto Latin codes: the text layer reads "Avwg evsjvq"
# for "আমি বাংলায়". Marker would copy that; such PDFs must be read with OCR instead.
LEGACY_FONT = re.compile(r"(?i)(sutonny|bijoy|boishakhi|mj$|mj[\W_])")


def legacy_font(pdf_path, max_pages=12):
    """the name of a legacy (non-Unicode) Bangla font used for text in the first pages, else None"""
    try:
        import ctypes
        import pypdfium2 as pdfium
        import pypdfium2.raw as raw
    except Exception:
        return None
    pdf = pdfium.PdfDocument(str(pdf_path))
    try:
        for i in range(min(len(pdf), max_pages)):
            for obj in pdf[i].get_objects(filter=[raw.FPDF_PAGEOBJ_TEXT], max_depth=2):
                font = raw.FPDFTextObj_GetFont(obj.raw)
                if not font:
                    continue
                n = raw.FPDFFont_GetBaseFontName(font, None, 0)
                buf = ctypes.create_string_buffer(max(1, n))
                raw.FPDFFont_GetBaseFontName(font, buf, n)
                name = buf.value.decode("latin-1").split("+")[-1]            # ABCDEF+SutonnyMJ
                if LEGACY_FONT.search(name):
                    return name
    except Exception:
        return None
    finally:
        pdf.close()
    return None


INDIC = re.compile(r"[\u0900-\u0dff]")                 # Devanagari, Bengali, Gurmukhi, Gujarati, Oriya, Tamil, Telugu, Kannada, Malayalam, Sinhala
DOT_WORDS = re.compile(r"(?:[·•]{2,12}[ \u00a0]+){2,}[·•]{1,12}")      # "····· ··· ····": glyphs with no Unicode, shown as dots
BROKEN = re.compile(r"[\ufffd\ue000-\uf8ff]")


def _indic_skeleton(word):
    """an Indic word's consonants and independent letters, without vowel signs, virama and nukta"""
    import unicodedata
    return "".join(c for c in unicodedata.normalize("NFD", word) if unicodedata.category(c)[0] != "M" and c not in "\u200c\u200d")


def _indic_invalid(words):
    """how many words have a vowel sign where none can be (word start, or right after another vowel sign): visual-order text"""
    import unicodedata
    bad = 0
    for w in words:
        prev = None
        for i, c in enumerate(w):
            sign = "VOWEL SIGN" in unicodedata.name(c, "")
            if sign and (i == 0 or prev):
                bad += 1
                break
            prev = sign
    return bad


def _align_indic(text, ref, word):
    """words garbled beyond a reordering (ইউনস্কেণ্ট for ইউনেস্কো): the output's Indic words are aligned with the PDF's; a short
    stretch (up to 3 words) that differs between two stretches that match is replaced by the PDF's words"""
    from difflib import SequenceMatcher
    spans = [(m.start(), m.end(), m.group(0)) for m in word.finditer(text)]
    ops = SequenceMatcher(None, [w for _, _, w in spans], ref, autojunk=False).get_opcodes()
    edits = []
    for k, (tag, i1, i2, j1, j2) in enumerate(ops):
        before = k == 0 or ops[k - 1][0] == "equal"
        after = k == len(ops) - 1 or ops[k + 1][0] == "equal"
        if tag == "replace" and i2 - i1 <= 3 and j2 - j1 <= 3 and before and after and len(ops) > 1:
            edits.append((spans[i1][0], spans[i2 - 1][1], " ".join(ref[j1:j2])))
    for a, b, w in reversed(edits):
        text = text[:a] + w + text[b:]
    return text, len(edits)


def _unescaped(line):
    r"""a Markdown line without its backslash escapes (\$ -> $), and where each character came from in the line"""
    out, at, i = [], [], 0
    while i < len(line):
        if line[i] == "\\" and i + 1 < len(line) and line[i + 1] in "$_*#`|!\\.-+":
            out.append(line[i + 1]); at.append(i); i += 2
        else:
            out.append(line[i]); at.append(i); i += 1
    at.append(len(line))
    return "".join(out), at


def repair_lines(text, layer_lines, need=0.9, most=8):
    r"""one page's Markdown against its text layer (the PDF's exact characters): a line that matches a layer line (or two
    joined) at least `need` but differs in a few characters gets those characters from the layer (Case 15: the OCR's
    \sqrt[2]{2} where the page prints \sqrt{2}). Only small differences (at most `most` characters each) are taken, never
    Markdown syntax (#, *, _, `, |, >, !, links, task boxes), and never in code, maths, tables, quotes or images.
    A layer that disagrees with most lines it matches (3 or more lines differ, under half as many are the same) is not trusted: an old
    scan's own OCR layer can be worse than Surya's reading, and Marker re-reads such pages for that reason.
    -> (text, characters spans fixed)"""
    from difflib import SequenceMatcher
    cands = [l.strip() for l in layer_lines if l.strip()]
    cands += [f"{a.strip()} {b.strip()}" for a, b in zip(layer_lines, layer_lines[1:]) if a.strip() and b.strip()]
    out, fixed, fence, math = [], 0, False, False
    pending, same = [], 0                                       # (line number, its corrected text), lines the layer agrees with
    for line in text.split("\n"):
        s = line.strip()
        if s.startswith(("```", "~~~")):
            fence = not fence
        if s == "$$" or (s.startswith("$$") and not s.endswith("$$")) or (s.endswith("$$") and not s.startswith("$$") and math):
            math = not math
        dollars = bool(re.search(r"(?<!\\)\$", s))             # maths, unless the PDF's text has those "$" too (printed LaTeX)
        skip = (fence or math or s.startswith(("|", ">", "![", "<")) or (s.startswith("$$") and not dollars) or "](" in s
                or len(s) < 20)
        if not skip:
            tail = line[len(line.rstrip()):]                   # "  " at the end is a line break: kept
            line = line.rstrip()
            lead = re.match(r"\s*(?:(?:[-*+]|\d+[.)])\s+|#{1,6}\s+|>\s*|\[[ xX]?\]\s*)*", line).end()   # list / heading markers
            u, at = _unescaped(line)
            best, score = None, 0.0
            for c in (cands if not dollars else [c for c in cands if "$" in c]):
                sm = SequenceMatcher(None, u, c, autojunk=False)
                if sm.real_quick_ratio() >= need and sm.quick_ratio() >= need:
                    r = sm.ratio()
                    if r > score:
                        best, score = c, r
            if best is not None and score >= need and score < 1.0:
                markup = set("*#_`|>!") | (set("[]") if re.search(r"\[[ xX]\]", u) else set())
                edits = []
                for tag, i1, i2, j1, j2 in SequenceMatcher(None, u, best, autojunk=False).get_opcodes():
                    a, b = u[i1:i2], best[j1:j2]
                    if tag == "equal" or not (a + b).strip() or max(len(a), len(b)) > most or set(a + b) & markup:
                        continue
                    if at[i1] < lead or "\\" in b:
                        continue                                # the line's own markers; never a backslash from the layer
                    if a.strip() and b.strip() and not re.search(r"\w", a + b):
                        continue                                # one punctuation mark for another ("-" / "·"): the layer's
                    if (not a.strip() or not b.strip()) and len((a + b).strip()) <= 1 and (a + b).strip() in ".,;:":
                        continue                                # a dot the layout added or left out: not worth it
                    edits.append((at[i1], at[i2], b if dollars else re.sub(r"([$])", r"\\\1", b)))
                if edits:
                    new = line
                    for x0, x1, b in reversed(edits):
                        new = new[:x0] + b + new[x1:]
                    pending.append((len(out), new + tail, len(edits)))
                else:
                    same += 1                                   # differs in Markdown syntax only
            elif best is not None:
                same += 1
            line += tail
        out.append(line)
    if len(pending) >= 3 and same < 0.5 * len(pending):
        return text, 0                                          # the layer disagrees with most lines: it is the one that's wrong
    for k, new, n in pending:
        out[k] = new
        fixed += n
    return "\n".join(out), fixed


def repair_from_layer(md_path, pdf_path):
    """repair_lines on every page that has a text layer (page separators needed) -> character spans fixed"""
    import pypdfium2 as pdfium
    from review import PAGE_SPLIT
    md_path = Path(md_path); md = md_path.read_text(encoding="utf-8")
    parts = PAGE_SPLIT.split(md)
    if len(parts) < 3:
        return 0
    fixed = 0
    pdf = pdfium.PdfDocument(str(pdf_path))
    try:
        for i in range(1, len(parts), 2):
            p = int(parts[i])
            if p >= len(pdf):
                continue
            tp = pdf[p].get_textpage()
            try:
                layer = tp.get_text_range().replace("\r", "")
            finally:
                tp.close()
            if len(layer.split()) < 5:
                continue
            from review import tidy                   # the final clean-up escapes printed "$" (\\$): a line still holding "$(1 + ...)$$"
            body, n = repair_lines(tidy(parts[i + 1]), layer.split("\n"))   # looks like maths and was skipped (Case 15 in a chunk)
            if n:                                      # tidy is idempotent: running it here changes nothing at the end
                parts[i + 1] = body
            fixed += n
    finally:
        pdf.close()
    if fixed:
        md_path.write_text(parts[0] + "".join(f"\n\n{{{parts[i]}}}{'-' * 48}\n\n{parts[i + 1].strip(chr(10))}"
                                              for i in range(1, len(parts), 2)) + "\n", encoding="utf-8")
    return fixed


def repair_indic(md_path, pdf_path):
    """Marker's text extraction stores Indic vowel signs in visual order (আমাদের -> আমাদরে, দাবিতে -> দাবতিে) where the PDF's
    own text (pdfium) is right. Per page: when pdfium's Indic words are well-formed, an output word that differs from the
    one pdfium word with the same consonants is replaced by it. Needs the page separators. -> words fixed"""
    import pypdfium2 as pdfium
    from review import PAGE_SPLIT
    md_path = Path(md_path); md = md_path.read_text(encoding="utf-8")
    parts = PAGE_SPLIT.split(md)
    if len(parts) < 3:
        return 0
    word = re.compile(r"[\u0900-\u0dff\u200c\u200d]+")
    fixed = 0
    pdf = pdfium.PdfDocument(str(pdf_path))
    try:
        for i in range(1, len(parts), 2):
            p = int(parts[i])
            if p >= len(pdf) or not INDIC.search(parts[i + 1]):
                continue
            tp = pdf[p].get_textpage()
            try:
                from review import INDIC_SPLIT                 # "ঢাক ার" (a line break inside the word) joined first
                ref = word.findall(INDIC_SPLIT.sub("", tp.get_text_range().replace("\r", " ").replace("\n", " ")))
            finally:
                tp.close()
            if len(ref) < 5 or _indic_invalid(ref) > 0.01 * len(ref):
                continue                                   # pdfium's text is not trustworthy here either
            by_skel = {}
            for w in ref:
                by_skel.setdefault(_indic_skeleton(w), set()).add(w)
            have = set(ref)

            def fix(m):
                nonlocal fixed
                w = m.group(0)
                cands = by_skel.get(_indic_skeleton(w), set())
                if w in have or len(cands) != 1 or len(_indic_skeleton(w)) < 2:
                    return w
                fixed += 1
                return next(iter(cands))
            parts[i + 1] = word.sub(fix, parts[i + 1])
            parts[i + 1], n = _align_indic(parts[i + 1], ref, word)
            fixed += n
    finally:
        pdf.close()
    md_path.write_text(parts[0] + "".join(f"\n\n{{{parts[i]}}}{'-' * 48}\n\n{parts[i + 1].strip(chr(10))}"
                                          for i in range(1, len(parts), 2)) + "\n", encoding="utf-8")
    return fixed


def page_list(page_range, default=40):
    """"3-7,9" -> [3..7, 9]; no range: the first `default` pages"""
    if not page_range:
        return list(range(default))
    return sorted({p for part in str(page_range).split(",") for p in range(int(part.split("-")[0]), int(part.split("-")[-1]) + 1)})


def ocr_reason(pdf_path, pages):
    """why the text layer of these pages can't be used (they must be read with OCR), or None:
    - an old Bangla (Bijoy) font: Latin codes stand for Bangla glyphs;
    - Indic scripts: PDFs store their vowel signs in visual order (আমাদের comes out as আমাদরে, ফেব্রুয়ারি as ফব্রুয়ারি);
    - glyphs without Unicode: dots ("····· ···"), U+FFFD or private-use characters instead of letters"""
    font = legacy_font(pdf_path)
    if font:
        return f"old Bangla font {font} (its text layer is not Unicode)"
    try:
        import pypdfium2 as pdfium
        pdf = pdfium.PdfDocument(str(pdf_path))
    except Exception:
        return None
    indic = broken = dots = 0
    try:
        for p in pages:
            if p >= len(pdf):
                continue
            tp = pdf[p].get_textpage()
            try:
                t = tp.get_text_range()
            finally:
                tp.close()
            indic += len(INDIC.findall(t)); broken += len(BROKEN.findall(t)); dots += len(DOT_WORDS.findall(t))
    finally:
        pdf.close()
    if dots or broken >= 5:
        return "text layer with glyphs that have no Unicode (dots or replacement characters)"
    return None


# ----------------------------------------------------------------- joining page chunks of one long PDF
# Marker keeps the original page numbers when it converts a page range, so image names (_page_37_Picture_2.jpeg) and
# meta page ids never collide between chunks: joining is concatenation in page order.
REMOTE_MARK = "<!-- mdbox:remote-pages {} -->"


def _take_chunk(chunk_dir, doc_dir, stem, meta):
    """moves a chunk's images into doc_dir, merges its meta into `meta`, returns its markdown"""
    chunk_dir, doc_dir = Path(chunk_dir), Path(doc_dir)
    text = (chunk_dir / f"{stem}.md").read_text(encoding="utf-8")
    mp = chunk_dir / f"{stem}_meta.json"
    if mp.is_file():
        m = json.loads(mp.read_text(encoding="utf-8"))
        for k, v in m.items():
            if isinstance(v, list) and k in ("table_of_contents", "page_stats"):
                meta.setdefault(k, []).extend(v)
            else:
                meta.setdefault(k, v)
    for f in chunk_dir.iterdir():
        if f.is_file() and f.name not in (f"{stem}.md", f"{stem}_meta.json"):
            os.replace(f, doc_dir / f.name)
    return text.strip("\n")


def _write_meta(doc_dir, stem, meta):
    for k in ("table_of_contents", "page_stats"):
        if k in meta:
            meta[k].sort(key=lambda e: e.get("page_id", 0) if isinstance(e, dict) else 0)
    (Path(doc_dir) / f"{stem}_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def _failed(pages, why):
    return f"<!-- mdbox: pages {pages} failed: {str(why).replace('--', '-')[:300]} -->"


def merge_chunks(doc_dir, stem, parts):
    """joins chunk outputs into <doc_dir>/<stem>.md (+ one _meta.json, images moved up).
    parts: [(pages, chunk_dir, state)] in page order; state "ok", "remote" (a placeholder the client fills in) or an error."""
    doc_dir = Path(doc_dir); doc_dir.mkdir(parents=True, exist_ok=True)
    texts, meta, failed, remote = [], {}, [], []
    for pages, chunk_dir, state in parts:
        if state == "remote":
            texts.append(REMOTE_MARK.format(pages)); remote.append(pages)
        elif state == "ok" and (Path(chunk_dir) / f"{stem}.md").is_file():
            texts.append(_take_chunk(chunk_dir, doc_dir, stem, meta))
        else:
            texts.append(_failed(pages, state)); failed.append(pages)
    (doc_dir / f"{stem}.md").write_text("\n\n".join(texts) + "\n", encoding="utf-8")
    _write_meta(doc_dir, stem, meta)
    shutil.rmtree(doc_dir / ".chunks", ignore_errors=True)
    return {"failed_pages": failed, "remote_pages": remote}


def fill_remote(doc_dir, stem, pages, chunk_dir, error=None):
    """replaces the placeholder of a chunk converted on the client with its markdown (or a failure note); True if it was found"""
    doc_dir = Path(doc_dir); md_path, mp = doc_dir / f"{stem}.md", doc_dir / f"{stem}_meta.json"
    md, mark = md_path.read_text(encoding="utf-8"), REMOTE_MARK.format(pages)
    if mark not in md:
        return False
    if error or not (Path(chunk_dir) / f"{stem}.md").is_file():
        new = _failed(pages, error or "no output")
    else:
        meta = json.loads(mp.read_text(encoding="utf-8")) if mp.is_file() else {}
        new = _take_chunk(chunk_dir, doc_dir, stem, meta)
        _write_meta(doc_dir, stem, meta)
    md_path.write_text(md.replace(mark, new), encoding="utf-8")
    return True


# ----------------------------------------------------------------- figures Marker extracted but did not link
# Marker saves a crop for every Picture/Figure/Diagram block, but when such a block sits inside a block that renders its own
# HTML (a figure group, a complex region...) its <img> tag is lost: on a scanned book, 28 of 124 figures had no link.
# The worker always asks Marker for page separators, puts each unlinked image at the end of its own page, then removes the
# separators again unless the user asked for them (PAGINATE).
PAGE_SEP = re.compile(r"\n*\{(\d+)\}-{48}\n*")
IMG_NAME = re.compile(r"^_page_(\d+)_[A-Za-z]+_(\d+)\.(?:jpe?g|png|webp|gif)$", re.I)


def _norm(t):
    return re.sub(r"[\W_]+", "", t.lower())


def _insert_before_anchor(text, link, anchor):
    """puts link just before the paragraph containing anchor (a caption); None if the anchor is not in text"""
    key = _norm(anchor)[:40]
    if len(key) < 8:
        return None
    paras = text.split("\n\n")
    for i, para in enumerate(paras):
        if key in _norm(para):
            return "\n\n".join(paras[:i] + [link.strip("\n")] + paras[i:])
    return None


def place_orphan_images(md_path, img_dir, keep_pages=False, anchors=None):
    """links every unlinked image: before its caption (anchors: {name: caption text}) if found on its page, else at the
    end of its page. Returns the number of images it linked"""
    anchors = anchors or {}
    md_path = Path(md_path); md = md_path.read_text(encoding="utf-8")
    orphans = sorted(((int(m.group(1)), int(m.group(2)), f.name) for f in Path(img_dir).iterdir()
                      if (m := IMG_NAME.match(f.name)) and f.name not in md))
    parts = PAGE_SEP.split(md)
    pre, pages = parts[0], [[int(parts[i]), parts[i + 1]] for i in range(1, len(parts), 2)]
    for page, _, name in orphans:
        link = f"\n\n![]({name})\n"
        target = next((p for p in reversed(pages) if p[0] <= page), pages[0] if pages else None)
        placed = _insert_before_anchor(target[1] if target else pre, link, anchors[name]) if name in anchors else None
        if placed is not None:
            if target is None:
                pre = placed
            else:
                target[1] = placed
        elif target is None:
            pre = pre.rstrip("\n") + link
        else:
            target[1] = target[1].rstrip("\n") + link
    if keep_pages:
        out = pre + "".join(f"\n\n{{{p}}}{'-' * 48}\n\n{t}" for p, t in pages)
    else:
        out = "\n\n".join(s.strip("\n") for s in [pre] + [t for _, t in pages] if s.strip())
    md_path.write_text(out.strip("\n") + "\n", encoding="utf-8")
    return len(orphans)

In [ ]:
%%writefile mdbox/figures.py
"""Re-crops the figures Marker extracted, from the page itself, so they are complete.

The layout model's boxes are often too tight (axis labels, legends, side text cut off) or split one figure into pieces
(each die of a dice picture, each arrow of a diagram). For every page with figures:
  1. render the page at ANALYSIS_DPI and find "ink" (dark or coloured pixels; light scan noise removed)
  2. drop "figures" that are really a caption; trim strips of body text a layout box reaches into at its edges; blank out
     body-text paragraphs and captions so a figure never grows into the text around it
  3. join ink that is closer than GAP_PT and grow each figure box over the ink connected to it (at most MAX_GROW_PT per side)
  4. merge figure boxes that overlap after growing (pieces of one figure share ink): one figure, one image
  5. crop the final box (+PAD_PT) from a CROP_DPI render and overwrite Marker's file; merged-away pieces are deleted
On PDFs it also adds the figures the layout model missed: every raster image the PDF draws (pdfium knows the exact
position) that no figure covers becomes a figure too (_page_N_Picture_9000+k), placed right before its caption in the
Markdown when it has one (convert.place_orphan_images), else at the end of its page. On one digital book the layout model
had missed photos on 107 of 282 pages that have images.
The core (refine_page) works on plain arrays and boxes in PDF points, so it is tested without Marker (dev/test_pipeline.py)."""
import re
from pathlib import Path

import numpy as np
from PIL import Image

ANALYSIS_DPI = 150
CROP_DPI = 300
GAP_PT = 7.0                 # ink closer than this belongs together (strokes, labels next to a chart)
MAX_GROW_PT = 90.0           # a box grows at most this far on each side (keeps a figure from walking across the page)
PAD_PT = 3.0
LONG_TEXT = 60               # text blocks with at least this many characters are body text: figures never grow into them
JPEG_QUALITY = 95
MIN_IMAGE_PT = 30            # smaller PDF images are bullets/icons, not figures


def ink_mask(img):
    """dark or saturated pixels (pale yellow fills count, light-grey scan noise does not)"""
    a = np.asarray(img.convert("RGB"), dtype=np.int32)
    gray = (a[..., 0] * 299 + a[..., 1] * 587 + a[..., 2] * 114) // 1000
    sat = a.max(axis=2) - a.min(axis=2)
    m = (gray < 225) | ((sat > 45) & (gray < 245))         # light-grey text and pale fills count
    from scipy import ndimage
    lab, n = ndimage.label(m, structure=np.ones((3, 3), bool))
    sizes = np.bincount(lab.ravel(), minlength=n + 1)
    return m & (sizes[lab] >= 4)                            # drop isolated scan specks; thin strokes stay


def _union(a, b):
    return (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3]))


def _near(a, b, gap):
    return a[0] - gap <= b[2] and b[0] - gap <= a[2] and a[1] - gap <= b[3] and b[1] - gap <= a[3]


def _area(b):
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def _inter(a, b):
    return _area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))


def trim_box(fb, walls, frac=0.3):
    """cuts off a strip of body text / caption that a too-generous layout box reaches into at one of its edges"""
    x0, y0, x1, y1 = fb
    for tb in walls:
        if _inter((x0, y0, x1, y1), tb) <= 0:
            continue
        w, h = x1 - x0, y1 - y0
        across = min(x1, tb[2]) - max(x0, tb[0]) > 0.3 * w
        down = min(y1, tb[3]) - max(y0, tb[1]) > 0.3 * h
        if across and tb[1] <= y0 < tb[3] < y1 and tb[3] - y0 < frac * h:
            y0 = tb[3]
        elif across and y0 < tb[1] < y1 <= tb[3] and y1 - tb[1] < frac * h:
            y1 = tb[1]
        elif down and tb[0] <= x0 < tb[2] < x1 and tb[2] - x0 < frac * w:
            x0 = tb[2]
        elif down and x0 < tb[0] < x1 <= tb[2] and x1 - tb[0] < frac * w:
            x1 = tb[0]
    return (x0, y0, x1, y1)


def refine_page(ink, page_size, figs, texts):
    """ink: bool array (H, W) of the page; page_size: (w, h) in points; figs: [(name, bbox)] in reading order;
    texts: [(bbox, n_chars[, kind])] of the other blocks, kind "caption" or "text". Boxes are (x0, y0, x1, y1) in points.
    -> ({name: bbox}, {name: kept_name, or None if the "figure" was only a caption and is dropped})"""
    from scipy import ndimage
    h, w = ink.shape
    sx, sy = w / page_size[0], h / page_size[1]
    to_px = lambda b: (max(0, int(b[0] * sx)), max(0, int(b[1] * sy)), min(w, int(np.ceil(b[2] * sx))), min(h, int(np.ceil(b[3] * sy))))
    walls = [t[0] for t in texts if t[1] >= LONG_TEXT or (len(t) > 2 and t[2] == "caption")]     # body text and captions
    captions = [t[0] for t in texts if len(t) > 2 and t[2] == "caption"]
    dropped = {n: None for n, fb in figs if any(_inter(fb, c) >= 0.8 * _area(fb) for c in captions)}
    body = [t[0] for t in texts if t[1] >= LONG_TEXT]
    inside = lambda inner, outer: _area(inner) < _area(outer) and _inter(inner, outer) >= 0.8 * _area(inner)
    for n, fb in figs:            # a box around a whole screenshot/page (other figures and body text inside it) is not a figure:
        if (any(m != n and inside(ob, fb) for m, ob in figs)          # its figures are cropped on their own, its text is text
                and any(inside(tb, fb) for tb in body)):
            dropped[n] = None
    figs = [(n, trim_box(fb, walls)) for n, fb in figs if n not in dropped]
    ink = ink.copy()
    for tb in walls:                                                       # walls stop growth, except inside a figure box
        x0, y0, x1, y1 = to_px(tb)
        keep = ink[y0:y1, x0:x1].copy()
        ink[y0:y1, x0:x1] = False
        for _, fb in figs:
            fx0, fy0, fx1, fy1 = to_px(fb)
            ix0, iy0, ix1, iy1 = max(x0, fx0), max(y0, fy0), min(x1, fx1), min(y1, fy1)
            if ix0 < ix1 and iy0 < iy1:
                ink[iy0:iy1, ix0:ix1] = keep[iy0 - y0:iy1 - y0, ix0 - x0:ix1 - x0]
    g = max(1, int(round(GAP_PT * sx)))
    labels, _ = ndimage.label(ndimage.binary_dilation(ink, structure=np.ones((g, g), bool)))
    labels = np.where(ink, labels, 0)
    objs = ndimage.find_objects(labels)
    boxes = {}
    for name, fb in figs:
        x0, y0, x1, y1 = to_px(fb)
        limit = (fb[0] - MAX_GROW_PT, fb[1] - MAX_GROW_PT, fb[2] + MAX_GROW_PT, fb[3] + MAX_GROW_PT)
        out = tuple(fb)
        for lab in np.unique(labels[max(0, y0 - g):y1 + g, max(0, x0 - g):x1 + g]):     # ink touching the box counts too
            if lab == 0 or objs[lab - 1] is None:
                continue
            ys, xs = objs[lab - 1]
            cb = (xs.start / sx, ys.start / sy, xs.stop / sx, ys.stop / sy)
            if cb[0] >= limit[0] and cb[1] >= limit[1] and cb[2] <= limit[2] and cb[3] <= limit[3]:
                out = _union(out, cb)
        boxes[name] = out
    merged, order = {}, [n for n, _ in figs]
    changed = True
    while changed:                                                         # pieces of one figure become one figure
        changed = False
        alive = [n for n in order if n not in merged]
        for i, a in enumerate(alive):
            for b in alive[i + 1:]:
                if _near(boxes[a], boxes[b], 0):          # overlap only: pieces sharing ink grow into each other, separate panels do not
                    boxes[a] = _union(boxes[a], boxes[b]); merged[b] = a; changed = True
                    break
            if changed:
                break
    for b, a in list(merged.items()):                                      # follow chains: c -> b -> a
        while a in merged:
            a = merged[a]
        merged[b] = a
    final = {n: (max(0.0, b[0] - PAD_PT), max(0.0, b[1] - PAD_PT), min(page_size[0], b[2] + PAD_PT), min(page_size[1], b[3] + PAD_PT))
             for n, b in boxes.items() if n not in merged}
    return final, {**merged, **dropped}


def pdf_image_boxes(pdf_page):
    """boxes (top-left origin, points) of the raster images a PDF page draws, without tiny icons and full-page backgrounds"""
    import pypdfium2.raw as pc
    w, h = pdf_page.get_size()
    l0, b0 = pdf_page.get_cropbox()[:2]
    out = []
    for o in pdf_page.get_objects(filter=(pc.FPDF_PAGEOBJ_IMAGE,), max_depth=3):
        l, b, r, t = (getattr(o, "get_bounds", None) or o.get_pos)()       # pypdfium2 5 / 4
        box = (max(0.0, l - l0), max(0.0, h - (t - b0)), min(w, r - l0), min(h, h - (b - b0)))
        bw, bh = box[2] - box[0], box[3] - box[1]
        if bw >= MIN_IMAGE_PT and bh >= MIN_IMAGE_PT and bw * bh < 0.85 * w * h:
            out.append(box)
    return out


def missing_images(img_boxes, fig_boxes, cover=0.5):
    """the images no figure covers at least half of"""
    return [b for b in img_boxes if not any(_inter(b, f) >= cover * _area(b) for f in fig_boxes)]


def text_share(box, text_boxes):
    """the share of a box that text blocks cover (each counted once, overlaps between them ignored)"""
    return min(1.0, sum(_inter(box, t) for t in text_boxes) / max(1e-6, _area(box)))


def caption_for(box, captions, max_gap=40.0):
    """text of the caption right below (else right above) a figure box, or None. captions: [(bbox, text)]"""
    best = None
    for cb, text in captions:
        if min(box[2], cb[2]) - max(box[0], cb[0]) < 0.3 * min(box[2] - box[0], cb[2] - cb[0]):
            continue
        gap = cb[1] - box[3] if cb[1] >= box[3] - 5 else (box[1] - cb[3]) + max_gap / 2      # below is preferred
        if -5 <= gap <= max_gap and (best is None or gap < best[0]):
            best = (gap, text)
    return best[1] if best else None


def blank_labels(crop, fb, labels, k, pad=6.0):
    """paints over the caption/figure label that a figure's box holds (it is text in the Markdown already: no second copy
    in the picture). crop: PIL image of box fb (points) at k pixels per point; labels: boxes in points -> how many.
    pad: a label's box is the text's; the coloured tab around it ("FIGURE 5.3" in a pink lozenge) reaches ~5 pt further"""
    from PIL import ImageDraw
    n = 0
    for lb in labels:
        if _area(lb) <= 0 or _inter(lb, fb) < 0.9 * _area(lb) or _area(lb) > 0.25 * _area(fb):
            continue
        x0, y0 = int((lb[0] - pad - fb[0]) * k[0]), int((lb[1] - pad - fb[1]) * k[1])
        x1, y1 = int(np.ceil((lb[2] + pad - fb[0]) * k[0])), int(np.ceil((lb[3] + pad - fb[1]) * k[1]))
        x0, y0, x1, y1 = max(0, x0), max(0, y0), min(crop.width, x1), min(crop.height, y1)
        if x1 <= x0 or y1 <= y0:
            continue
        a = np.asarray(crop.convert("RGB"))
        ring = np.concatenate([a[max(0, y0 - 2), x0:x1], a[min(crop.height - 1, y1 + 1), x0:x1],
                               a[y0:y1, max(0, x0 - 2)], a[y0:y1, min(crop.width - 1, x1 + 1)]])
        light = ring[ring.min(axis=1) > 200]                  # the background around it (not the figure's own ink)
        color = tuple(int(c) for c in np.median(light if len(light) else np.array([[255, 255, 255]]), axis=0))
        ImageDraw.Draw(crop).rectangle([x0, y0, x1 - 1, y1 - 1], fill=color)
        n += 1
    return n


def drop_refs(md, names):
    """removes the image links of merged-away pieces and dropped captions"""
    for n in names:
        md = re.sub(r"!\[[^\]]*\]\(" + re.escape(n) + r"\)\n?", "", md)
    return md


# ----------------------------------------------------------------- the Marker side (runs in worker.py)
def recrop_document(document, src, out_dir, stem):
    """returns {"pages", "figures", "merged", "dropped_captions"} counts; files that Marker did not write are left alone"""
    from marker.schema import BlockTypes
    from marker.settings import settings
    ext = settings.OUTPUT_IMAGE_FORMAT.lower()
    fig_types = tuple(getattr(BlockTypes, t) for t in ("Picture", "Figure", "Diagram") if hasattr(BlockTypes, t))
    text_types = tuple(getattr(BlockTypes, t) for t in ("Text", "SectionHeader", "ListItem", "Caption", "Footnote", "PageHeader", "PageFooter",
                                                        "Table", "TableOfContents", "Form", "Code", "Equation", "Handwriting") if hasattr(BlockTypes, t))
    content_types = tuple(getattr(BlockTypes, t) for t in ("Table", "TableOfContents", "Form", "Equation", "Code") if hasattr(BlockTypes, t))
    out_dir = Path(out_dir)
    pdf = None
    if str(src).lower().endswith(".pdf"):
        import pypdfium2 as pdfium
        pdf = pdfium.PdfDocument(str(src))
    stats = {"pages": 0, "figures": 0, "merged": 0, "dropped_captions": 0, "added_from_pdf": 0, "anchors": {}, "text_heavy": {}}
    all_merged = []
    try:
        for page in document.pages:
            figs = [(f"{b.id.to_path()}.{ext}", tuple(b.polygon.bbox)) for b in page.contained_blocks(document, fig_types)]
            figs = [f for f in figs if (out_dir / f[0]).is_file()]
            blocks = page.contained_blocks(document, text_types)
            added = []
            if pdf is not None and page.page_id < len(pdf):
                # an image Marker already turned into a table / form / equation / code is covered too (no duplicate picture)
                covers = [fb for _, fb in figs] + [tuple(b.polygon.bbox) for b in blocks if b.block_type in content_types]
                text_boxes = [tuple(b.polygon.bbox) for b in blocks if b.block_type != BlockTypes.Caption]
                added = [(f"_page_{page.page_id}_Picture_{9000 + k}.{ext}", b)
                         for k, b in enumerate(missing_images(pdf_image_boxes(pdf[page.page_id]), covers))
                         if text_share(b, text_boxes) < 0.25]       # a screenshot/scan whose content became text: not a figure
            figs += added
            if not figs:
                continue
            texts = [(tuple(b.polygon.bbox), len(b.raw_text(document) or ""), "caption" if b.block_type == BlockTypes.Caption else "text")
                     for b in blocks]
            captions = [(tuple(b.polygon.bbox), b.raw_text(document) or "") for b in blocks if b.block_type == BlockTypes.Caption]
            from review import CAPTION                    # "FIGURE 5.3" tabs and captions printed inside a figure's box
            labels = [cb for cb, _ in captions] + [tuple(b.polygon.bbox) for b in blocks if b.block_type != BlockTypes.Caption
                                                   and len((b.raw_text(document) or "").strip()) <= 30
                                                   and CAPTION.fullmatch((b.raw_text(document) or "").strip())
                                                   and re.search(r"[^\W\d_]", b.raw_text(document) or "")]     # not a bare number (an axis value)
            labels += [tuple(b.polygon.bbox) for b in blocks          # the caption's words printed right next to its label
                       if b.block_type in (BlockTypes.Text, BlockTypes.Caption) and tuple(b.polygon.bbox) not in labels
                       and len(b.raw_text(document) or "") <= 300 and any(_near(tuple(b.polygon.bbox), lb, 8) for lb in labels)]
            pw, ph = page.polygon.width, page.polygon.height
            if pdf is not None:
                pg = pdf[page.page_id]
                render = lambda dpi: pg.render(scale=dpi / 72).to_pil()
            else:
                hi = page.get_image(highres=True)
                render = lambda dpi: hi
            small = render(ANALYSIS_DPI)
            boxes, merged = refine_page(ink_mask(small), (pw, ph), figs, texts)
            big = render(CROP_DPI).convert("RGB")
            kx, ky = big.width / pw, big.height / ph
            long_text = [tb for tb, n, kind in texts if kind == "text" and n >= 20]
            for name, fb in boxes.items():
                share = text_share(fb, long_text)
                if share >= 0.4:
                    stats["text_heavy"][name] = round(share, 2)
            for name, (x0, y0, x1, y1) in boxes.items():
                crop = big.crop((int(x0 * kx), int(y0 * ky), int(np.ceil(x1 * kx)), int(np.ceil(y1 * ky))))
                stats["labels_blanked"] = stats.get("labels_blanked", 0) + blank_labels(crop, (x0, y0, x1, y1), labels, (kx, ky))
                crop.save(out_dir / name, quality=JPEG_QUALITY, subsampling=0)
            for name in merged:
                (out_dir / name).unlink(missing_ok=True)
            for name, fb in added:
                if name in boxes:
                    stats["added_from_pdf"] += 1
                    cap = caption_for(fb, captions)
                    if cap:
                        stats["anchors"][name] = cap
            all_merged += list(merged)
            stats["pages"] += 1; stats["figures"] += len(boxes)
            stats["merged"] += sum(v is not None for v in merged.values()); stats["dropped_captions"] += sum(v is None for v in merged.values())
    finally:
        if pdf is not None:
            pdf.close()
    md = out_dir / f"{stem}.md"
    if all_merged and md.is_file():
        md.write_text(drop_refs(md.read_text(encoding="utf-8"), all_merged), encoding="utf-8")
    return stats


# ----------------------------------------------------------------- text inside figures, written out by the vision LLM
DESCRIBE_PROMPT = """This image is a figure from a document.{context}
Describe it for a reader who cannot see it, so that nothing it shows is lost. The image may hold several figures or panels
(a diagram next to another, (a)/(b) parts): describe every one of them, not only the one a caption names. Return JSON with:
- "parts": first look over the whole image and list every separate figure, diagram, chart, table, photo or panel in it
  (with its label or caption if it has one), one entry each; a single figure gives one entry
- "kind": one of photo, diagram, schematic, chart, table, code, screenshot, map, text, logo, decoration
- "summary": one or two sentences saying what the figure is and what it shows or explains: its subject and purpose.
  Use the caption and nearby text to name things correctly, but describe only what is visible.
- "details": everything inside every part, in Markdown, part by part, without repeating the summary:
  - every label, number, unit and value exactly as written (read carefully; the caption and text help with names and spelling);
  - schematic or wiring diagram: each component with its label and value, and what it connects to, following the circuit;
  - other diagrams and flowcharts: each part with its label, and how the parts are connected or ordered;
  - chart or graph: the axes with their ranges and units, and the values of every series;
  - table: a Markdown table with every row and column;
  - code or a program listing: the exact code in a fenced code block with its language;
  - photo: what is in it and how it is arranged, plus any visible text; skip decorative detail;
  - logo or decoration: leave empty.
Write the summary and details in the language of the document (the caption and nearby text show it; English if there
is no text); labels and values stay exactly as written in the figure. Do not add facts that are not visible. No headings."""
FIGURE_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["parts", "kind", "summary", "details"],
                 "properties": {"parts": {"type": "array", "items": {"type": "string"}}, "kind": {"type": "string"},
                                "summary": {"type": "string"}, "details": {"type": "string"}}}
SUMMARY_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["kind", "summary"],
                  "properties": {"kind": {"type": "string"}, "summary": {"type": "string"}}}
IMG_LINK = re.compile(r"!\[[^\]]*\]\(([^)\s]+)\)")
DESC_MARK = "> **Figure content:**"
DESC_BLOCK = re.compile(r"\n+> \*\*Figure content:\*\*[^\n]*(?:\n>[^\n]*)*(?:\n\n```mermaid\n[\s\S]*?\n```)?")   # + its Mermaid flowchart
MIN_DESCRIBE_PX = 150        # crops are 300 DPI: smaller than half an inch is an icon, not a figure to write out
TEXT_HEAVY_NOTE = """
Most of the running text in this image is already written out in the document next to it: describe its figures, diagrams,
charts, tables and pictures in full, but mention the running text only in a few words."""


def _ask(llm_url, model, im, prompt, max_tokens, schema=None, timeout=900, extra=None):
    """one vision-LLM call on a PIL image -> (text, finish_reason)"""
    import base64, io, json, urllib.request
    im = im.convert("RGB")
    im.thumbnail((1600, 1600))
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=92)
    body = {"model": model, "temperature": 0, "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()}},
                {"type": "text", "text": prompt}]}]}
    if schema:
        body["response_format"] = {"type": "json_schema", "json_schema": {"name": "figure", "strict": True, "schema": schema}}
    body.update(extra or {})
    req = urllib.request.Request(llm_url.rstrip("/") + "/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", "Authorization": "Bearer local"})
    import review                                           # the worker's cap on LLM requests in flight (review.LLM_GATE)
    with review.LLM_GATE:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            c = json.load(r)["choices"][0]
    return c["message"]["content"] or "", c.get("finish_reason")


def _ask_image(llm_url, model, img_path, prompt, timeout=900, max_tokens=2000, schema=None):
    """-> {"kind", "summary", "details"}; raises on failure (the caller retries)"""
    from review import latex_json_loads                     # LaTeX in JSON loses its backslashes otherwise
    text, finish = _ask(llm_url, model, Image.open(img_path), prompt, max_tokens, schema or FIGURE_SCHEMA, timeout)
    if finish == "length":
        raise ValueError("cut off")
    out = latex_json_loads(text)
    if not isinstance(out, dict) or not str(out.get("summary", "")).strip():
        raise ValueError("no summary")
    return {k: str(out.get(k, "")).strip() for k in ("kind", "summary", "details")}


GRAPH_PROMPT = """This image is a diagram from a document.{context}
If it is NOT a graph of shapes joined by arrows or lines (a flowchart, UML activity or state diagram, process, workflow,
decision tree, block diagram), return {{"diagrams": [], "links": []}}.
Otherwise write out its graph exactly, so that it could be redrawn from your answer.
The image may hold SEVERAL separate diagrams side by side, each with its own start: list each one separately in
"diagrams", in order from the leftmost to the rightmost. Dotted or dashed lines that fan out from one box to a whole other diagram mean that box is
expanded (zoomed) into that diagram: say so in "links" as "<box label> -> <diagram number>".
For each diagram:
- "nodes": EVERY shape, none skipped, each with a short id unique in the whole answer:
  start dot = shape "start"; end bull's-eye = "end"; box = "activity" with its label exactly as written (line breaks as
  spaces); diamond = "decision" (label = the question next to it) or "merge" (no label); thick bar = "fork" when one
  arrow goes in and several go out, "join" when several go in and one comes out.
- "edges": EVERY arrow, "from" id -> "to" id, with its label ("Yes", "No", a guard) or "". Follow each line to its
  arrowhead: the edge goes to the shape the arrowhead touches. Each branch that leaves a fork runs in parallel: an edge from
  the fork to its first box and from its last box to the join.
- Feedback or return lines: one line often runs along the bottom or side and branches off with an arrowhead into SEVERAL
  boxes (a waterfall's "go back" arrows). Every arrowhead on it is its own edge, from the shape the line starts at.
Before answering, count the boxes and the arrowheads in each diagram: every box must be a node, and there must be one edge
per arrowhead."""
GRAPH_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["diagrams", "links"], "properties": {
    "diagrams": {"type": "array", "items": {"type": "object", "additionalProperties": False, "required": ["nodes", "edges"], "properties": {
        "nodes": {"type": "array", "items": {"type": "object", "additionalProperties": False, "required": ["id", "label", "shape"],
                  "properties": {"id": {"type": "string"}, "label": {"type": "string"}, "shape": {"type": "string"}}}},
        "edges": {"type": "array", "items": {"type": "object", "additionalProperties": False, "required": ["from", "to", "label"],
                  "properties": {"from": {"type": "string"}, "to": {"type": "string"}, "label": {"type": "string"}}}}}}},
    "links": {"type": "array", "items": {"type": "string"}}}}
GRAPH_KINDS = ("diagram", "flowchart")             # "kind" values whose image may be a graph of boxes and arrows (not circuits)


def _chains(nodes, edges):
    """a diagram's arrows as chains "A → B → C", the main flow first: from the start (or the first box) the first arrow out
    of each box is followed as far as it goes; then every arrow not yet told starts a chain of its own (a branch, a
    feedback arrow), which goes on through boxes whose arrows are not told yet"""
    outs = {}
    for e in edges:
        outs.setdefault(e[0], []).append(e)
    has_in = {e[1] for e in edges}
    roots = [i for i in nodes if i not in has_in] or list(nodes)[:1]
    order, todo = [], list(roots)
    while todo:                                                       # boxes in the order a reader meets them
        i = todo.pop(0)
        if i not in order:
            order.append(i); todo += [e[1] for e in outs.get(i, [])]
    order += [i for i in nodes if i not in order]
    used, chains = set(), []
    for u in order:
        for e in outs.get(u, []):
            if id(e) in used:
                continue
            chain, cur = [u], e
            while cur is not None:
                used.add(id(cur)); chain.append((cur[2], cur[1]))
                seen = chain[:1] + [t for _, t in chain[1:]]
                free = [x for x in outs.get(cur[1], []) if id(x) not in used]
                if cur[1] in seen[:-1]:
                    break                                     # the loop is closed: "A → B → C → A"
                fresh = [x for x in free if x[1] not in seen]
                cur = fresh[0] if fresh else (free[0] if free else None)
            chains.append(chain)
    return chains


def graph_markdown(g):
    """a diagram's graph (GRAPH_SCHEMA) -> (Markdown of its arrows in words, a Mermaid flowchart block), or ("", "").
    Several diagrams sit side by side (left to right), each drawn top to bottom; UML start = solid dot, end = bull's-eye,
    fork/join = solid bar. Edges to unknown ids are dropped."""
    shape_text = {"start": "start", "end": "end", "fork": "fork", "join": "join", "merge": "merge"}
    quote = lambda t: '"' + t.replace('"', "#quot;") + '"'
    diagrams = []
    for k, d in enumerate(g.get("diagrams") or [], 1):
        nodes = {str(n.get("id")): n for n in d.get("nodes") or [] if str(n.get("id", "")).strip()}
        edges = [(str(e.get("from")), str(e.get("to")), (e.get("label") or "").strip()) for e in d.get("edges") or []
                 if str(e.get("from")) in nodes and str(e.get("to")) in nodes]
        if edges:
            diagrams.append((k, nodes, edges))
    if not diagrams:
        return "", ""
    many = len(diagrams) > 1
    ids, words, used_cls = {}, [], set()
    lines = ["```mermaid", "flowchart LR" if many else "flowchart TD"]
    nums = {k: n for n, (k, _, _) in enumerate(diagrams, 1)}                  # diagrams without arrows dropped: renumber
    for k, nodes, edges in diagrams:
        for i in nodes:
            ids[k, i] = f"n{len(ids) + 1}"            # per diagram: the model may call each diagram's start "start"
        name = lambda i, nodes=nodes: (nodes[i].get("label") or "").strip() or shape_text.get(nodes[i].get("shape"), "step")
        ind = "    " if many else "  "
        if many:
            words.append(f"- **Diagram {nums[k]}:**")
            lines += [f'  subgraph d{nums[k]}["Diagram {nums[k]}"]', "    direction TB"]
        for i, n in nodes.items():
            sh, label = n.get("shape"), quote(name(i))
            bar = '["' + "&nbsp;" * 16 + '"]'                              # a wide, thin black bar
            node = {"start": '((" "))', "end": '(((" ")))', "fork": bar, "join": bar,
                    "decision": "{" + label + "}", "merge": '{" "}'}.get(sh, f"({label})")
            cls = {"start": "umlStart", "end": "umlEnd", "fork": "umlBar", "join": "umlBar"}.get(sh)    # "end" is a Mermaid keyword
            if cls:
                used_cls.add(cls)
            lines.append(f"{ind}{ids[k, i]}{node}" + (f":::{cls}" if cls else ""))
        for e in edges:
            lines.append(f"{ind}{ids[k, e[0]]} -->" + (f"|{quote(e[2])}|" if e[2] else "") + f" {ids[k, e[1]]}")
        for ch in _chains(nodes, edges):
            text = name(ch[0]) + "".join(f" →{f' ({lab})' if lab else ''} {name(t)}" for lab, t in ch[1:])
            words.append(("  - " if many else "- ") + text)
        if many:
            lines.append("  end")
    labels = {}                                           # a link names its box by label or by id ("activity1 -> diagram2")
    for k, nodes, _ in diagrams:
        for i, n in nodes.items():
            if (n.get("label") or "").strip():
                for key in (n["label"].strip().lower(), i.strip().lower()):
                    labels.setdefault(key, (k, n["label"].strip()))
    for link in g.get("links") or []:
        m = re.match(r"\s*(.+?)\s*->\s*(?:diagram\s*)?(\d+)\s*$", str(link), re.I)
        if m and m.group(1).strip().lower() in labels and int(m.group(2)) in nums and many:
            k, label = labels[m.group(1).strip().lower()]
            if k != int(m.group(2)):
                words.append(f"- *{label}* (Diagram {nums[k]}) is shown in detail in Diagram {nums[int(m.group(2))]}")
                lines.append(f"  d{nums[k]} -.->|{quote(label + ' in detail')}| d{nums[int(m.group(2))]}")    # subgraph to subgraph
    style = {"umlStart": "fill:#000,stroke:#000", "umlEnd": "fill:#000,stroke:#fff,stroke-width:3px",       # end: bull's-eye
             "umlBar": "fill:#000,stroke:#000,font-size:2px"}
    lines += [f"  classDef {c} {style[c]}" for c in sorted(used_cls)] + ["```"]
    return "\n".join(["Arrows, as drawn:"] + words), "\n".join(lines)


MISSING_PROMPT = """This image is a diagram. Someone listed its arrows as:
{arrows}
The boxes are: {boxes}.
Look at the image again, line by line, and find every arrowhead that this list does NOT have yet: especially arrows that go
back or up to an earlier box, lines along the bottom or the side that branch off into several boxes (each arrowhead is one
arrow), and arrows into or out of bars. Return only the missing arrows, with "from" and "to" being box labels exactly as in
the list above, and "label" only if a word is written on that arrow (else ""), or an empty list if none is missing."""
MISSING_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["missing"], "properties": {"missing": {"type": "array",
    "items": {"type": "object", "additionalProperties": False, "required": ["from", "to", "label"], "properties": {
        "from": {"type": "string"}, "to": {"type": "string"}, "label": {"type": "string"}}}}}}


def add_missing(g, answers):
    """adds to graph g the arrows that every one of the check answers ([[{"from", "to", "label"}]] by box label) proposes,
    when both ends are labelled boxes of one diagram and the arrow is not there yet; -> how many were added. (On a complete
    graph the check invents arrows, but never the same one between two labelled boxes in every answer.)"""
    sets = [{(str(e.get("from", "")).strip().lower(), str(e.get("to", "")).strip().lower()) for e in a} for a in answers]
    agreed = set.intersection(*sets) if sets else set()
    added = 0
    for d in g.get("diagrams") or []:
        by_label = {(n.get("label") or "").strip().lower(): str(n.get("id")) for n in d.get("nodes") or [] if (n.get("label") or "").strip()}
        have = {(str(e.get("from")), str(e.get("to"))) for e in d.get("edges") or []}
        for a, b in sorted(agreed):
            if a in by_label and b in by_label and a != b and (by_label[a], by_label[b]) not in have:
                d.setdefault("edges", []).append({"from": by_label[a], "to": by_label[b], "label": ""})
                have.add((by_label[a], by_label[b])); added += 1
    return added


def ask_graph(llm_url, model, img_path, context="", tries=3, checks=2):
    """the boxes and arrows of a flow diagram: (arrows in words, Mermaid block) as graph_markdown, or ("", "").
    The model sometimes leaves arrows out (a waterfall's feedback arrows: 2 of 3 answers had all 8), never adds any: it is
    asked several times at once and the fullest answer is kept. img_path: a file or the image's bytes"""
    import io
    from concurrent.futures import ThreadPoolExecutor
    from review import latex_json_loads
    raw = img_path if isinstance(img_path, bytes) else Path(img_path).read_bytes()

    def one(_):
        for tokens in (3000, 5000):
            try:
                text, finish = _ask(llm_url, model, Image.open(io.BytesIO(raw)), GRAPH_PROMPT.format(context=context), tokens, GRAPH_SCHEMA)
                if finish != "length":
                    return latex_json_loads(text)
            except Exception:
                continue
        return None
    with ThreadPoolExecutor(tries) as ex:
        answers = [g for g in ex.map(one, range(tries)) if isinstance(g, dict)]
    size = lambda g: (sum(len(d.get("nodes") or []) for d in g.get("diagrams") or []),
                      sum(len(d.get("edges") or []) for d in g.get("diagrams") or []), len(g.get("links") or []))
    if not answers:
        return "", ""
    g = max(answers, key=size)
    lines, boxes = [], set()                # a second look for the arrows it left out (a waterfall's feedback arrows)
    for d in g.get("diagrams") or []:
        name = {str(n.get("id")): (n.get("label") or "").strip() or n.get("shape", "") for n in d.get("nodes") or []}
        boxes |= {(n.get("label") or "").strip() for n in d.get("nodes") or [] if (n.get("label") or "").strip()}
        lines += [f"- {name.get(str(e.get('from')), '?')} -> {name.get(str(e.get('to')), '?')}" + (f" ({e['label']})" if e.get("label") else "")
                  for e in d.get("edges") or []]
    if lines and checks:
        prompt = MISSING_PROMPT.format(arrows="\n".join(lines), boxes=", ".join(sorted(boxes)))

        def check(_):
            try:
                text, finish = _ask(llm_url, model, Image.open(io.BytesIO(raw)), prompt, 1500, MISSING_SCHEMA)
                return latex_json_loads(text).get("missing") or [] if finish != "length" else None
            except Exception:
                return None
        with ThreadPoolExecutor(checks) as ex:
            found = [a for a in ex.map(check, range(checks)) if isinstance(a, list)]
        if len(found) == checks:
            add_missing(g, found)
    return graph_markdown(g)


CORE_PROMPT = """This image was cut from a document page around a figure, and the cut may have taken in things that are NOT
part of the figure itself:
- the figure's number or caption, e.g. "FIGURE 5.3" in a coloured tab and the caption words next to it or under it;
- the page's running header or page number, e.g. "30 Chapter 2 Software processes", "140 PART TWO MODELING";
- horizontal rules (long thin lines) above or below that separate the figure from the page;
- body text paragraphs.
List each of those in "not_figure" with what it is and its bounding box. Then give in "figure" the bounding box of the
figure itself only: the diagram, chart, photo or drawing with all of its own boxes, labels, arrows, legends and axes,
and none of the things in "not_figure". Boxes are [x1, y1, x2, y2] in coordinates from 0 to 1000 relative to the image
width and height."""
_BOX = {"type": "array", "items": {"type": "integer"}}
CORE_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["not_figure", "figure"], "properties": {
    "not_figure": {"type": "array", "items": {"type": "object", "additionalProperties": False, "required": ["what", "bbox_2d"],
                                              "properties": {"what": {"type": "string"}, "bbox_2d": _BOX}}},
    "figure": _BOX}}
CORE_LOST_INK = 0.005        # the core crop may leave out at most this share of the figure's ink that nobody explained


def core_box(ink, fig, not_fig):
    """ink: bool (H, W); fig and not_fig boxes in pixels (the vision LLM's). -> the figure's box snapped to whole ink shapes,
    or None when it would lose ink that is neither in a not_figure box nor a thin rule (the LLM cut real content)"""
    from scipy import ndimage
    h, w = ink.shape
    rules = ndimage.binary_opening(ink, structure=np.ones((1, max(3, int(0.3 * w))), bool))    # long horizontal lines ...
    rl, _ = ndimage.label(rules)
    for k, sl in enumerate(ndimage.find_objects(rl), 1):                                        # ... outside the figure box:
        if sl is not None and _inter((sl[1].start, sl[0].start - 3, sl[1].stop, sl[0].stop + 3), fig) > 0:
            rules[sl] &= rl[sl] != k                                                            # its own lines (an axis) stay
    ink = ink & ~ndimage.binary_dilation(rules, structure=np.ones((5, 1), bool))              # page rules (one may run into a tab)
    lab, _ = ndimage.label(ndimage.binary_dilation(ink, structure=np.ones((5, 5), bool)))
    lab = np.where(ink, lab, 0)
    keep, lost = None, 0
    for k, sl in enumerate(ndimage.find_objects(lab), 1):
        if sl is None:
            continue
        cb = (sl[1].start, sl[0].start, sl[1].stop, sl[0].stop)
        a = max(1.0, _area(cb))
        if any(_inter(cb, nb) >= 0.5 * a for nb in not_fig):
            continue                                            # the caption, label, header the LLM pointed out
        if _inter(cb, fig) >= 0.5 * a:
            keep = cb if keep is None else _union(keep, cb)
        elif not (cb[3] - cb[1] <= max(4, 0.012 * h) and cb[2] - cb[0] >= 8 * (cb[3] - cb[1])):    # a thin rule may go
            lost += int((lab[sl] == k).sum())
    if keep is None or lost > CORE_LOST_INK * max(1, int(ink.sum())):
        return None
    return keep


def tighten_figure(img_path, llm_url, model, pad_px=12):
    """crops a figure image to the figure itself (the vision LLM tells what is caption, page header or rule) -> True if cropped"""
    from review import latex_json_loads
    img_path = Path(img_path)
    im = Image.open(img_path).convert("RGB")
    w, h = im.size
    try:
        text, finish = _ask(llm_url, model, im, CORE_PROMPT, 800, CORE_SCHEMA)
        j = latex_json_loads(text)
        px = lambda b: (b[0] * w / 1000, b[1] * h / 1000, b[2] * w / 1000, b[3] * h / 1000)
        fig = px(j["figure"])
        nots = [px(x["bbox_2d"]) for x in j.get("not_figure") or [] if len(x.get("bbox_2d") or []) == 4]
    except Exception:
        return False
    if finish == "length" or fig[2] - fig[0] < 0.1 * w or fig[3] - fig[1] < 0.1 * h:
        return False
    b = core_box(ink_mask(im), fig, nots)
    if b is None:
        return False
    b = (max(0, b[0] - pad_px), max(0, b[1] - pad_px), min(w, b[2] + pad_px), min(h, b[3] + pad_px))
    if _area(b) >= 0.97 * w * h:
        return False                                            # already the figure alone
    im.crop(tuple(int(v) for v in b)).save(img_path, quality=JPEG_QUALITY, subsampling=0)
    return True


TILE_PX = 1100
TRANSCRIBE_PROMPT = """This is one part of a larger image (a page, a newspaper, a scanned document). Transcribe all the text
in this part exactly as written, in reading order, keeping its language and script, as Markdown (headings, lists, tables
where the layout shows them). Text cut at the edge of the part: write what is visible. No comments; nothing if there is no text."""


NO_LOOPS = {"dry_multiplier": 0.8, "dry_base": 1.75, "dry_allowed_length": 3}   # llama.cpp's DRY: a repeated word sequence costs
                              # more each time (a newspaper's "To Auctioneers, etc." x100). Only where nothing is copied from the prompt
TRANSCRIBE_ONE_PROMPT = ("Transcribe all the text in this image exactly as written, in reading order (column by column where the "
                         "text is in columns), keeping its language and script, as Markdown (headings, lists, tables where the "
                         "layout shows them). Write a row of dots (a dot leader) as \" … \". No comments.")


def _looped(text):
    """an answer stuck in a loop: a line 4 times in a row, or one stretch repeated 6+ times"""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    run = 1
    for a, b in zip(lines, lines[1:]):
        run = run + 1 if a == b else 1
        if run >= 4:
            return True
    return bool(re.search(r"(.{6,40}?)\1{5,}", text))


def clean_transcript(text):
    """dot leaders as " … ", and a line repeated more than twice in a row kept twice"""
    text = re.sub(r"(?:\s*\.){6,}\s*", " … ", text)
    out = []
    for l in text.split("\n"):
        if len(out) >= 2 and l.strip() and l.strip() == out[-1].strip() == out[-2].strip():
            continue
        out.append(l)
    return "\n".join(out).strip()


def transcribe_image(llm_url, model, img_path):
    """every word of a picture of text: one answer for the whole image (a letter, an invoice: 2 s, every line and price);
    tile by tile only when one answer is cut off or loops (a newspaper page) -> Markdown text"""
    try:
        text, finish = _ask(llm_url, model, Image.open(img_path), TRANSCRIBE_ONE_PROMPT, 4000, extra=NO_LOOPS)
        if finish != "length" and text.strip() and not _looped(text):
            return clean_transcript(text)
    except Exception:
        pass
    return clean_transcript(transcribe_tiles(llm_url, model, img_path))


def transcribe_tiles(llm_url, model, img_path, concurrency=4):
    """the text of a dense image (a newspaper page...) too long for one answer: read tile by tile at full resolution"""
    from concurrent.futures import ThreadPoolExecutor
    im = Image.open(img_path).convert("RGB")
    w, h = im.size
    cols, rows = max(1, round(w / TILE_PX)), max(1, round(h / TILE_PX))
    tw, th, pad = w / cols, h / rows, 40
    tiles = [im.crop((max(0, int(c * tw) - pad), max(0, int(r * th) - pad), min(w, int((c + 1) * tw) + pad), min(h, int((r + 1) * th) + pad)))
             for c in range(cols) for r in range(rows)]                       # column by column: newspapers read down first

    def one(t):
        for _ in range(2):
            try:
                text, _ = _ask(llm_url, model, t, TRANSCRIBE_PROMPT, 3000, extra=NO_LOOPS)
                return text.strip()
            except Exception:
                continue
        return ""
    with ThreadPoolExecutor(max(1, concurrency)) as ex:
        return "\n\n".join(x for x in ex.map(one, tiles) if x)


SUB_CAPTION = re.compile(r"\s*(\*\*|_)?\(\s*[a-zA-Z]\s*\)\s+\S")      # "(a) Noisy unoriented data set": one panel of a figure


def _context(md, name):
    """the figure's caption (the paragraph after it), the caption of the whole figure when it is one panel of several, and the
    text before it, for the prompt"""
    at = md.find(f"]({name})")
    after = [p for p in DESC_BLOCK.sub("", md[at:at + 4000]).split("\n\n")[1:] if p.strip()]
    from review import CAPTION                              # "Figure 3-2", "图3", "Рис. 5", "চিত্র ৪", "شكل 3"...
    is_cap = lambda p: len(p) < 400 and (CAPTION.match(p) or re.match(r"\s*(\*\*|_)?(fig(ure)?|table|chart|listing|photo)\b", p, re.I))
    caption = next((p.strip() for p in after[:2] if is_cap(p) or SUB_CAPTION.match(p) and len(p) < 200), "")
    whole = ""
    if SUB_CAPTION.match(caption):                          # one panel: the whole figure's caption follows its last panel
        whole = next((p.strip() for p in after[1:12] if is_cap(p) and not SUB_CAPTION.match(p)), "")
    before = next((p.strip() for p in reversed(md[max(0, at - 1500):at].split("\n\n")[:-1])
                   if p.strip() and not p.lstrip().startswith((">", "!", "|", "#")) and not SUB_CAPTION.match(p)), "")
    clean = lambda t, n: re.sub(r"<[^>]+>|!\[[^\]]*\]\([^)]*\)", "", t).strip()[:n]
    out = ""
    if caption:
        out += f'\nA caption next to it in the document (it may name only one of several figures): "{clean(caption, 400)}"'
    if whole:
        out += f'\nIt is one panel of a larger figure whose caption is: "{clean(whole, 400)}"'
    if before and not SUB_CAPTION.match(caption):
        out += f'\nText just before it: "{clean(before, 500)}"'
    return out


def _render(d, md):
    """the description as a quote under the image: what it is, then what is in it"""
    details = d["details"]
    fence = re.search(r"```[^\n]*\n(.*?)```", details, re.S)
    if d["kind"].lower() == "code" and fence:                 # a listing already printed in the text is not copied again
        words = re.findall(r"\w{2,}", fence.group(1))
        text = set(re.findall(r"\w{2,}", md))
        if words and sum(w in text for w in words) / len(words) >= 0.9:
            details = re.sub(r"```[^\n]*\n.*?```", "", details, flags=re.S).strip()
            d = dict(d, summary=d["summary"].rstrip() + " The same code is printed in the text.")
    lines = [DESC_MARK + " " + " ".join(d["summary"].split())]
    body = [l.rstrip() for l in details.splitlines()]
    while body and not body[0].strip():
        body.pop(0)
    if body:
        lines.append(">")
        lines += [("> " + l) if l.strip() else ">" for l in body]
    if d.get("mermaid"):                                      # outside the quote: many viewers draw no Mermaid inside one
        lines += ["", d["mermaid"]]
    return "\n".join(lines)


TEXT_KINDS = ("text", "screenshot")
TEXT_IMAGE_PX = 1000          # a text picture at least this big (long side) is transcribed tile by tile, not only described


def describe_figures(md_path, img_dir, llm_url, model, concurrency=2, text_heavy=None, tighten=True, graph_tries=3,
                     graph_checks=2, defer=False, transcribe_text=False):
    """adds under each linked figure (once) what it shows and what is in it, written by the vision LLM from the image, its
    caption and the text around it; returns how many it described. tighten: crop each figure to itself first (one call);
    graph_tries/graph_checks: answers and missing-arrow checks for a flow diagram. defer=True: the Markdown is not written
    yet -> (count, apply), apply(md_path) puts the descriptions in later (the page review runs at the same time)"""
    from concurrent.futures import ThreadPoolExecutor
    import hashlib
    md_path, img_dir = Path(md_path), Path(img_dir)
    md = md_path.read_text(encoding="utf-8")
    names = [n for n in dict.fromkeys(IMG_LINK.findall(md)) if (img_dir / n).is_file()
             and not re.search(r"(?m)^\|.*\]\(" + re.escape(n) + r"\)", md)]      # a picture in a table cell: a quote would break the table
    names = [n for n in names if not re.search(re.escape(f"]({n})") + r"\s*\n+" + re.escape(DESC_MARK), md)
             and not re.search(r"!\[[^\]]+\]\(" + re.escape(n) + r"\)", md)]        # a logo already named in its alt text
    same, todo = {}, []                     # identical images (a repeated icon or logo) are described once
    for n in names:
        try:
            with Image.open(img_dir / n) as im:
                if max(im.size) < MIN_DESCRIBE_PX:
                    continue                # an icon or bullet: nothing to write out
        except Exception:
            continue
        h = hashlib.sha1((img_dir / n).read_bytes()).hexdigest()
        if h not in same:
            todo.append(n)
        same.setdefault(h, []).append(n)

    text_heavy = text_heavy or {}

    from concurrent.futures import ThreadPoolExecutor as _Pool
    graphs = _Pool(max(1, concurrency))     # a figure's graph is read while it is cropped and described (it needs neither)

    def one(n):
        wide = (img_dir / n).read_bytes()   # the graph is read better with some room around it (Fig 5.3: right order 4/4, not 0/4)
        ctx = _context(md, n)
        early = graph_tries and "_Diagram_" in n          # Marker's layout called it a diagram: likely boxes and arrows
        graph = graphs.submit(ask_graph, llm_url, model, wide, ctx, graph_tries, graph_checks) if early else None
        if tighten:
            try:                            # the figure alone: no caption, label tab, page header or rule in the picture
                tighten_figure(img_dir / n, llm_url, model)
            except Exception:
                pass
        prompt = DESCRIBE_PROMPT.format(context=_context(md, n)) + (TEXT_HEAVY_NOTE if n in text_heavy else "")
        for tokens in (2000, 2000, 4000):   # a failed or cut-off answer is asked again (never skipped silently)
            try:
                d = _ask_image(llm_url, model, img_dir / n, prompt, max_tokens=tokens)
            except Exception:
                continue
            if d["kind"].lower() in GRAPH_KINDS:
                words, mermaid = graph.result() if graph is not None else \
                    (ask_graph(llm_url, model, wide, ctx, graph_tries, graph_checks) if graph_tries else ("", ""))
                if words:                   # boxes and arrows read on their own: the structure is exact, not retold
                    d["details"], d["mermaid"] = words, mermaid
                    try:                        # and the summary says the same (it was written before the graph was read)
                        s = _ask_image(llm_url, model, img_dir / n, DESCRIBE_PROMPT.format(
                            context=_context(md, n) + f"\nIts boxes and arrows, already read exactly:\n{words}")
                            + '\nOnly "kind" and "summary" this time: no details. The summary must agree with those arrows.',
                            max_tokens=600, schema=SUMMARY_SCHEMA)
                        d["summary"] = s["summary"]
                    except Exception:
                        pass
            elif transcribe_text and d["kind"].lower() in TEXT_KINDS and n not in text_heavy:
                try:                        # a newspaper page, a letter: every word of it, read tile by tile (one answer can't hold it)
                    with Image.open(img_dir / n) as im:
                        big = max(im.size) >= TEXT_IMAGE_PX
                    if big:
                        text = transcribe_image(llm_url, model, img_dir / n)
                        if text and not _looped(text):
                            d["details"] = text
                except Exception:
                    pass
            return n, d
        # still too long (a newspaper page in one picture): a short summary, and its text read tile by tile
        try:
            d = _ask_image(llm_url, model, img_dir / n, prompt + "\nOnly \"kind\" and \"summary\" this time: no details.",
                           max_tokens=600, schema=SUMMARY_SCHEMA)
        except Exception:
            d = {"kind": "text", "summary": "A page of text.", "details": ""}
        if n not in text_heavy:
            d["details"] = transcribe_image(llm_url, model, img_dir / n)
        return n, d
    with ThreadPoolExecutor(max(1, concurrency)) as ex:
        first = {n: d for n, d in ex.map(one, todo) if d}
    graphs.shutdown(wait=False, cancel_futures=True)          # graphs of figures that were not diagrams: not needed
    got = {m: first[n] for ns in same.values() for n in ns[:1] if n in first for m in ns}
    for ns in same.values():                # the same picture elsewhere gets the same (tightened) crop
        for m in ns[1:]:
            if ns[0] in first:
                (img_dir / m).write_bytes((img_dir / ns[0]).read_bytes())

    def apply(path):
        path = Path(path)
        text = path.read_text(encoding="utf-8")
        for n, d in got.items():
            if d["kind"].lower() in ("logo", "decoration", "icon") and not d["details"].strip():
                alt = " ".join(re.sub(r"[\[\]\n]", " ", d["summary"]).split())[:200]   # a logo or icon: named in one line, no quote
                text = text.replace(f"![]({n})", f"![{alt}]({n})", 1)
                continue
            quote = _render(d, text)
            text = re.sub(r"(!\[[^\]]*\]\(" + re.escape(n) + r"\))", lambda m: m.group(1) + "\n\n" + quote, text, count=1)
        path.write_text(text, encoding="utf-8")
    if defer:
        return len(got), apply
    apply(md_path)
    return len(got)


def redescribe(md_path, img_dir, llm_url, model, concurrency=8):
    """removes every figure description from a finished file and writes them again (better prompt or model, no reconversion)"""
    md_path = Path(md_path)
    md_path.write_text(DESC_BLOCK.sub("", md_path.read_text(encoding="utf-8")), encoding="utf-8")
    return describe_figures(md_path, img_dir, llm_url, model, concurrency)


# ----------------------------------------------------------------- tables: values in the wrong column, pictures in cells
def _runs(mask, gap):
    """[(start, stop)] of the True runs of a 1-D bool array, runs closer than `gap` merged"""
    idx = np.flatnonzero(mask)
    if not len(idx):
        return []
    out, s, p = [], idx[0], idx[0]
    for i in idx[1:]:
        if i - p > gap:
            out.append((int(s), int(p) + 1)); s = i
        p = i
    out.append((int(s), int(p) + 1))
    return out


def realign_table(rows, ink):
    """rows: a table's rows (cell strings, "" = empty) as written; ink: bool array of the table's image (no ruled lines).
    -> the rows with every value moved to the column it stands under in the image (Case 14: the lower rows of a staggered
    table had slid left), or None when the image does not match (row count, values per row, column count). Values and their
    order within a row are never changed."""
    ncols = max(len(r) for r in rows)
    bands = [b for b in _runs(ink.any(axis=1), 2) if b[1] - b[0] >= 4]
    if len(bands) != len(rows):
        return None
    blobs = []
    for (y0, y1), r in zip(bands, rows):
        bl = _runs(ink[y0:y1].any(axis=0), max(2, int(0.5 * (y1 - y0))))     # a value's characters, or "..." dots, stay one
        if len(bl) != len([c for c in r if c.strip()]):
            return None
        blobs.append(bl)
    h = float(np.median([b[1] - b[0] for b in bands]))
    best = None
    for side in (1, 0, None):                      # right-aligned numbers, left-aligned text, centred
        xs = sorted((b[side] if side is not None else (b[0] + b[1]) / 2) for bl in blobs for b in bl)
        groups = [[xs[0]]]
        for x in xs[1:]:
            if x - groups[-1][-1] > 0.6 * h:
                groups.append([x])
            else:
                groups[-1].append(x)
        if len(groups) == ncols:
            spread = max(g[-1] - g[0] for g in groups)
            if best is None or spread < best[0]:
                best = (spread, side, [float(np.mean(g)) for g in groups])
    if best is None:
        return None
    _, side, centres = best
    out = []
    for bl, r in zip(blobs, rows):
        row = [""] * ncols
        for b, v in zip(bl, [c for c in r if c.strip()]):
            x = b[side] if side is not None else (b[0] + b[1]) / 2
            k = int(np.argmin([abs(x - c) for c in centres]))
            if row[k]:
                return None                         # two values under one column: the image was not read right
            row[k] = v
        out.append(row)
    return out


def grid_cells(ink, nrows, ncols, frac=0.6):
    """[[cell box (x0, y0, x1, y1)]] of a ruled table of nrows x ncols cells (pixels), or None when its lines don't make that
    grid. Lines of a frame around the table (a coloured box with its title) are left out: the widest run of consecutive
    lines with the right count is the grid."""
    xs = [(a + b) / 2 for a, b in _runs(ink.mean(axis=0) >= frac, 2)]
    ys = [(a + b) / 2 for a, b in _runs(ink.mean(axis=1) >= frac, 2)]

    def edges(v, n, size):                     # the table's box may sit just inside its border lines: its edges are the outer lines
        if len(v) < n and not any(x < 0.03 * size for x in v):
            v = [0.0] + v
        if len(v) < n and not any(x > 0.97 * size for x in v):
            v = v + [float(size)]
        return v
    xs, ys = edges(xs, ncols + 1, ink.shape[1]), edges(ys, nrows + 1, ink.shape[0])
    if len(xs) < ncols + 1 or len(ys) < nrows + 1:
        return None
    gx = max((xs[i:i + ncols + 1] for i in range(len(xs) - ncols)), key=lambda v: v[-1] - v[0])
    gy = max((ys[i:i + nrows + 1] for i in range(len(ys) - nrows)), key=lambda v: v[-1] - v[0])
    return [[(gx[c], gy[r], gx[c + 1], gy[r + 1]) for c in range(ncols)] for r in range(nrows)]


def cell_picture(ink, pad=4):
    """the picture in a table cell (bool ink of the cell): everything above its last line of text (the label), or None"""
    ink = ink.copy()
    ink[:, ink.mean(axis=0) > 0.6] = False                     # the cell's own ruled lines
    ink[ink.mean(axis=1) > 0.6, :] = False
    bands = [(a + pad, b + pad) for a, b in _runs(ink[pad:-pad, pad:-pad].any(axis=1), 3) if b - a >= 2]
    if len(bands) < 2:
        return None
    label = bands[-1]
    y0, y1 = bands[0][0], bands[-2][1]
    if y1 - y0 < 0.5 * (label[1] - label[0]):
        return None                                            # only text above the label: no picture
    xr = _runs(ink[y0:y1, pad:-pad].any(axis=0), 10)
    return (xr[0][0] + pad, y0, xr[-1][1] + pad, y1) if xr else None


SHAPES = re.compile(r"\\(triangle|square|Box|bigtriangleup|bigtriangledown|diamond|Diamond|lozenge|bigcirc|pentagon|hexagon|"
                    r"blacksquare|blacktriangle|boxdot|boxslash|boxbslash|boxtimes|boxplus|varhexagon|octagon)\b|\\text\{\s*-{2,}\s*\}")
DISPLAY_MATH = re.compile(r"(?<!\\)\$\$([\s\S]+?)(?<!\\)\$\$")


def drawn_formula(latex):
    r"""a "formula" made of drawings (Case 15: Q = — + △ + □ + pentagons cut by diagonals): LaTeX has no way to write those
    shapes, so the OCR wrote \triangle, \square... for each one"""
    return len(SHAPES.findall(latex)) >= 3


def _drawn_paragraph(p):
    """a paragraph that is only a formula of drawn shapes: display maths, or inline maths with at most a few characters
    around it ("Q = $\\_ + \\triangle + ...$." after Marker's inline-maths rewrite, accurate quality)"""
    if not drawn_formula(p):
        return False
    rest = re.sub(r"(?<!\\)\$\$[\s\S]+?(?<!\\)\$\$|(?<!\\)\$[^$]+(?<!\\)\$", "", p)
    return len(re.sub(r"\s", "", rest)) <= 12


DRAWN_SPAN = re.compile(r"(?:\b[A-Za-z]{1,3}\s*=\s*)?(?<!\\)\$\$?[^$]+?(?<!\\)\$\$?[.,;]?")   # "Q = $\\_ + \\triangle ...$."


def picture_formulas(md_path, document, src, out_dir, skipped=None):
    """a display formula made of drawings becomes a picture of it, cropped from the page (Marker's Equation block), and is
    described like a figure; its LaTeX was a guess. Found as a maths span (display, or inline after Marker's inline-maths
    rewrite, even inside a sentence: only the formula becomes the picture), paired with the Marker block sharing most of it.
    skipped (a dict) gets why a drawn formula was left as it was. -> how many"""
    from marker.schema import BlockTypes
    from review import PAGE_SPLIT
    md_path, out_dir = Path(md_path), Path(out_dir)
    if not str(src).lower().endswith(".pdf") or not hasattr(BlockTypes, "Equation"):
        return 0
    import pypdfium2 as pdfium
    md = md_path.read_text(encoding="utf-8")
    parts = PAGE_SPLIT.split(md)
    pages = {p.page_id: p for p in document.pages}
    done = 0
    pdf = pdfium.PdfDocument(str(src))
    try:
        for i in range(1, len(parts), 2):
            pid, body = int(parts[i]), parts[i + 1]
            found = [m for m in DRAWN_SPAN.finditer(body) if drawn_formula(m.group(0))]
            page = pages.get(pid)
            if not found or page is None or pid >= len(pdf):
                continue
            kinds = tuple(getattr(BlockTypes, k) for k in ("Equation", "TextInlineMath", "Text") if hasattr(BlockTypes, k))
            blocks = [b for b in page.contained_blocks(document, kinds) if drawn_formula(getattr(b, "html", "") or "")]
            toks = lambda t: set(re.findall(r"\\[a-zA-Z]+|\w+", re.sub(r"<[^>]+>", " ", t)))
            pairs, used = [], set()
            for m in found:                                    # each formula with the block sharing most of it (after Marker's
                ft = toks(m.group(0))                         # inline-maths rewrite, the same formula can be in two blocks)
                score = [(len(ft & toks(getattr(b, "html", "") or "")) / max(1, len(ft)), k) for k, b in enumerate(blocks) if k not in used]
                best = max(score, default=(0, None))
                if best[1] is not None and best[0] >= 0.5:
                    used.add(best[1]); pairs.append((m, blocks[best[1]]))
            if skipped is not None and len(pairs) < len(found):
                why = "no Marker block with that formula" if blocks else "no Marker block with drawn shapes"
                skipped[why] = skipped.get(why, 0) + len(found) - len(pairs)
            found = [m for m, _ in pairs]; blocks = [b for _, b in pairs]
            if not found:
                continue
            img = pdf[pid].render(scale=CROP_DPI / 72).to_pil().convert("RGB")
            sx, sy = img.width / page.polygon.width, img.height / page.polygon.height
            for k, (m, block) in reversed(list(enumerate(zip(found, blocks)))):
                x0, y0, x1, y1 = block.polygon.bbox
                name = f"_page_{pid}_Formula_{k}.jpeg"
                img.crop((max(0, int((x0 - 4) * sx)), max(0, int((y0 - 4) * sy)), min(img.width, int((x1 + 4) * sx)),
                          min(img.height, int((y1 + 4) * sy)))).save(out_dir / name, quality=JPEG_QUALITY, subsampling=0)
                body = body[:m.start()].rstrip(" ") + f"\n\n![]({name})\n\n" + body[m.end():].lstrip(" ")
                done += 1
            parts[i + 1] = body
    finally:
        pdf.close()
    if done:
        md_path.write_text(parts[0] + "".join(f"\n\n{{{parts[i]}}}{'-' * 48}\n\n{parts[i + 1].strip(chr(10))}"
                                              for i in range(1, len(parts), 2)) + "\n", encoding="utf-8")
    return done


TABLE_MD = re.compile(r"(?m)^(?:\|.*\|[ \t]*\n?){2,}")
SEP_ROW = re.compile(r"^\|[\s:|-]+\|$")
NO_SRC_IMG_CELL = re.compile(r'<img(?![^>]*\bsrc=)[^>]*?\balt="([^"]*)"[^>]*/?>')


def _table_rows(t):
    lines = [l.strip() for l in t.strip().splitlines()]
    return [[c.strip() for c in l.strip("|").split("|")] for l in lines if not SEP_ROW.match(l)]


def _table_md(rows):
    n = max(len(r) for r in rows)
    rows = [r + [""] * (n - len(r)) for r in rows]
    line = lambda r: "| " + " | ".join(r) + " |"
    return "\n".join([line(rows[0]), "|" + "---|" * n] + [line(r) for r in rows[1:]]) + "\n"


def fix_tables(md_path, document, src, out_dir, stem=""):
    """after the page review: each Markdown table of a page is paired with Marker's Table block (same order on the page)
    and checked against the page image. Values that slid into the wrong column are moved back (realign_table), and a
    picture in a cell that Marker left as <img alt="surfel"> (no file) is cropped from the ruled cell and linked.
    -> {"tables_realigned", "table_pictures"}"""
    from marker.schema import BlockTypes
    from review import PAGE_SPLIT
    md_path, out_dir = Path(md_path), Path(out_dir)
    stats = {"tables_realigned": 0, "table_pictures": 0, "tables_skipped": {}}
    skip = lambda why: stats["tables_skipped"].__setitem__(why, stats["tables_skipped"].get(why, 0) + 1)
    if not str(src).lower().endswith(".pdf"):
        return stats
    import pypdfium2 as pdfium
    md = md_path.read_text(encoding="utf-8")
    parts = PAGE_SPLIT.split(md)
    pages = {p.page_id: p for p in document.pages}
    pdf = pdfium.PdfDocument(str(src))
    try:
        for i in range(1, len(parts), 2):
            pid, body = int(parts[i]), parts[i + 1]
            tables = list(TABLE_MD.finditer(body))
            page = pages.get(pid)
            if not tables or page is None or pid >= len(pdf):
                continue
            blocks = page.contained_blocks(document, (BlockTypes.Table,))
            words = lambda s: set(re.findall(r"[^\W_]{2,}", re.sub(r"<[^>]+>", " ", s).lower()))
            pairs = []                                         # each Markdown table with the Table block sharing most words
            for m in tables:
                mw = words(m.group(0))
                scored = [(len(mw & words(b.raw_text(document) or "")) / max(1, len(mw)), b) for b in blocks]
                best = max(scored, key=lambda x: x[0], default=(0, None))
                if best[1] is None or best[0] < 0.3:
                    skip("no matching table block"); continue
                pairs.append((m, best[1]))
            k = CROP_DPI / 72
            img = None
            for t, (m, block) in reversed(list(enumerate(pairs))):
                if img is None:
                    img = pdf[pid].render(scale=k).to_pil().convert("RGB")
                sx, sy = img.width / page.polygon.width, img.height / page.polygon.height
                x0, y0, x1, y1 = block.polygon.bbox
                box = (max(0, int((x0 - 3) * sx)), max(0, int((y0 - 3) * sy)), min(img.width, int((x1 + 3) * sx)), min(img.height, int((y1 + 3) * sy)))
                crop = img.crop(box)
                ink = ink_mask(crop)
                rows = _table_rows(m.group(0))
                new = None
                if any(NO_SRC_IMG_CELL.search(c) for r in rows for c in r):
                    cells = grid_cells(ink, len(rows), max(len(r) for r in rows))
                    if not cells:
                        skip("no ruled grid for its pictures")
                    if cells:
                        new = [list(r) for r in rows]
                        for r, row in enumerate(rows):
                            for c, cell in enumerate(row):
                                im_m = NO_SRC_IMG_CELL.search(cell)
                                if not im_m or c >= len(cells[r]):
                                    continue
                                cx0, cy0, cx1, cy1 = (int(v) for v in cells[r][c])
                                pic = cell_picture(ink[cy0:cy1, cx0:cx1])
                                if not pic:
                                    skip("no picture found in a cell"); continue
                                name = f"_page_{pid}_Table_{t}_r{r}c{c}.jpeg"
                                crop.crop((cx0 + pic[0] - 6, cy0 + pic[1] - 6, cx0 + pic[2] + 6, cy0 + pic[3] + 6)).save(
                                    out_dir / name, quality=JPEG_QUALITY, subsampling=0)
                                alt = re.sub(r"[\[\]]", "", im_m.group(1))
                                new[r][c] = cell[:im_m.start()] + f"![{alt}]({name})" + cell[im_m.end():]
                                stats["table_pictures"] += 1
                elif not any(re.search(r"<|!\[", c) for r in rows for c in r):
                    clean = ink.copy()                         # ruled lines would join the rows: left out
                    clean[:, clean.mean(axis=0) > 0.6] = False
                    clean[clean.mean(axis=1) > 0.6, :] = False
                    out = realign_table(rows, clean)
                    if out and out != [r + [""] * (len(out[0]) - len(r)) for r in rows]:
                        new = out
                        stats["tables_realigned"] += 1
                if new:
                    body = body[:m.start()] + _table_md(new) + body[m.end():]
            parts[i + 1] = body
    finally:
        pdf.close()
    if not stats["tables_skipped"]:
        del stats["tables_skipped"]
    if stats["tables_realigned"] or stats["table_pictures"]:
        md_path.write_text(parts[0] + "".join(f"\n\n{{{parts[i]}}}{'-' * 48}\n\n{parts[i + 1].strip(chr(10))}"
                                              for i in range(1, len(parts), 2)) + "\n", encoding="utf-8")
    return stats


# ----------------------------------------------------------------- did any text get lost?
WORD = re.compile(r"[^\W_]{3,}")
LOW_COVERAGE = 0.9


def _words(text):
    """words of any script (review.script_tokens), character pairs for Chinese/Japanese/Thai; short Latin words left out"""
    import unicodedata
    from review import script_tokens
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[-­￾\x02]\s*\n\s*|[­￾\x02]", "", text)   # words hyphenated across lines (pdfium marks them U+FFFE)
    return [t for t in script_tokens(text, ngram=2) if len(t) >= 3 or not t.isascii()]


def text_coverage(pdf_path, md_text, pages):
    """for each page of a PDF that has a text layer: the share of its words that appear in the Markdown.
    -> [{"page", "words", "found", "low"}]; pages with fewer than 20 words are not judged"""
    import pypdfium2 as pdfium
    raw = md_text
    md_text = re.sub(r"\\[a-zA-Z]+|[_^]\{([^{}]*)\}", r"\1", md_text)              # V_{CAP} in LaTeX is the page's "VCAP"
    have = set(_words(md_text)) | set(_words(md_text.replace("-", "")))      # "double-throw" = "double-\nthrow" on the page
    have |= set(_words(raw.replace("\\_", "_")))                                # LaTeX printed as text ("\\mathbf{e}_1") counts too
    out, texts = [], {}
    pdf = pdfium.PdfDocument(str(pdf_path))
    try:
        for p in pages:
            if p >= len(pdf):
                continue
            tp = pdf[p].get_textpage()
            try:
                texts[p] = tp.get_text_range()
            finally:
                tp.close()
    finally:
        pdf.close()
    # running headers/footers ("Chapter 3: Getting Somewhat More Serious 131") repeat on many pages and are left out of the
    # Markdown on purpose: they don't count (page 144 of Make Electronics was flagged for its header alone)
    key = lambda line: re.sub(r"[\d\W_]+", "", line.lower())
    seen = {}
    for t in texts.values():
        for k in {key(l) for l in t.splitlines() if key(l)}:
            seen[k] = seen.get(k, 0) + 1
    running = {k for k, n in seen.items() if n >= 3 and n >= 0.2 * len(texts)}
    for p, t in texts.items():
        words = _words("\n".join(l for l in t.splitlines() if key(l) not in running))
        if not words:
            continue
        found = sum(w in have for w in words) / len(words)
        out.append({"page": p, "words": len(words), "found": round(found, 3), "low": len(words) >= 20 and found < LOW_COVERAGE})
    return out

In [ ]:
%%writefile mdbox/review.py
"""Page review: the vision LLM checks every page's Markdown against the page itself and writes the final version.

For each page it gets the page image, Marker's draft Markdown of that page and (PDFs with text) the page's text layer,
and returns the corrected page: missing or misread words fixed, headings, lists, GitHub tables, LaTeX math ($...$,
$$...$$), fenced code, paragraphs joined for reading. Figures (image link + "Figure content" quote) and link anchors are
swapped for <<KEEP n>> tokens first, so the LLM cannot drop, move or rewrite them.
A rewrite is only used when it loses nothing (accept()): every KEEP token exactly once, at least WORDS_KEPT of the draft's
words, no fewer words overall, and (with a text layer) at least the draft's share of the page's real words. Otherwise
the page keeps Marker's draft. tidy() then fixes Markdown syntax over the whole document, deterministically.
shrink_images() caps image size so the .md stays light to open (crops are made at 300 DPI)."""
import base64
import io
import json
import re
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from PIL import Image

PAGE_SPLIT = re.compile(r"\n*\{(\d+)\}-{48}\n*")
KEEP = re.compile(r"!\[[^\]]*\]\([^)\s]+\)(?:\n\n> \*\*Figure content:\*\*[^\n]*(?:\n>[^\n]*)*(?:\n\n```mermaid\n[\s\S]*?\n```)?)?"   # figure + its description (+ Mermaid)
                  r"|<span id=\"[^\"]*\">\s*</span>")                                                # anchor of an internal link
TOKEN = "<<KEEP {}>>"
WORDS_KEPT = 0.9             # share of the draft's distinct words a rewrite must keep (room for fixed OCR errors)
CHARS_PER_TOKEN = 3.0        # conservative, to size requests to the LLM's context
IMAGE_TOKENS = 1536          # the server's --image-max-tokens
PAGE_PX = 1600               # page render, long side (the server scales it to IMAGE_TOKENS)
IMAGE_MAX_PX = 2400          # images in the output are at most this big (long side): a full page at 300 DPI, so crops keep their detail
WORD = re.compile(r"[^\W_]{3,}")

PROMPT = """You check and correct ONE page of a document that was converted to Markdown.
You get: the page image, the DRAFT Markdown of this page{layer_note}.
Return the corrected Markdown of this page and nothing else (no explanations, no ``` around the whole answer).

Rules:
- Keep ALL content of the page: every word, number, label, list item, table cell, footnote, sidebar and caption, in reading
  order, in the page's own language and script (digits too: ১০ stays ১০). Never summarize, shorten, explain, translate or
  add anything that is not on the page.
- Fix words the draft got wrong or left out, by reading the image{layer_rule}.
- Tokens like <<KEEP 3>> stand for figures and link anchors: keep every one exactly once, at the same place.
- Headings: '#' marks that follow the page's visual hierarchy; keep the draft's heading levels unless clearly wrong.
- Paragraphs: join lines broken only by the page layout; join words hyphenated across a line break.
- Lists: '-' or '1.' items. Tables: GitHub pipe tables (HTML <table> only when cells are merged); if the table on the page
  has no header row, its first row is the header: never invent column names.
- Math: inline $...$ and display $$...$$ (on its own lines), valid LaTeX, for formulas, symbols and variables only.
  Check every symbol against the image: ceiling \\lceil \\rceil vs floor \\lfloor \\rfloor, \\leqslant vs \\le, hats, bars,
  primes, sub- and superscripts. A symbol LaTeX has no command for (a drawn box, a domino) stays that Unicode character in
  \\text{{}} (\\text{{▯}}); an equation's number goes inside it as \\tag{{2.13}}.
- Program code and command listings: fenced ``` blocks (with the language if known), exact characters and indentation.
- A dollar sign for money is written \\$ (\\$34.99), so it is not read as maths.\n- Bold and italic as on the page.{header_rule}

DRAFT:
{draft}{layer}"""


# scripts written without spaces: every character is a token (a whole clause would otherwise be one "word")
NO_SPACE = ((0x3040, 0x30FF), (0x3400, 0x4DBF), (0x4E00, 0x9FFF), (0xF900, 0xFAFF), (0x20000, 0x2FA1F),   # kana, Han
            (0x0E00, 0x0EFF), (0x1000, 0x109F), (0x1780, 0x17FF))                                          # Thai, Lao, Myanmar, Khmer


def _no_space(ch):
    o = ord(ch)
    return any(a <= o <= b for a, b in NO_SPACE)


import unicodedata


def script_tokens(text, ngram=1):
    """words of any script: letters and digits apart (9V -> 9, v), vowel signs and other combining marks kept with their
    letter (Bangla বাংলা is one word, not ব ল), Chinese/Japanese/Thai one token per character (ngram=2: character pairs,
    for coverage, where single characters would match anywhere)"""
    import unicodedata
    out, cur, run = [], "", ""
    def flush_word():
        nonlocal cur
        if cur:
            out.append(cur); cur = ""
    def flush_run():
        nonlocal run
        if run:
            out.extend(run if ngram == 1 or len(run) == 1 else [run[i:i + ngram] for i in range(len(run) - ngram + 1)]); run = ""
    for ch in text:
        cat = unicodedata.category(ch)
        if _no_space(ch) and cat[0] in "LM":
            flush_word()
            if cat[0] == "M" and run:
                run = run[:-1] + run[-1] + ch if ngram > 1 else run + ch
            else:
                run += ch
            continue
        flush_run()
        if cat[0] == "M" and cur:
            cur += ch
        elif cat[0] in "LN":
            if cur and cur[-1].isdigit() != ch.isdigit():
                flush_word()
            cur += ch
        else:
            flush_word()
    flush_word(); flush_run()
    res = []
    for t in out:                     # a 20+ letter run in a script not listed (Tibetan, Javanese...): no spaces, per character
        letters = [c for c in t if unicodedata.category(c)[0] != "M"]
        if len(letters) >= 20 and not t.isascii() and not t.isdigit():
            chars = re.findall(r".[\u0300-\u036f\u0900-\u0dff\u0f00-\u0fff]*", t)
            res += chars if ngram == 1 else [chars[i] + chars[i + 1] for i in range(len(chars) - 1)]
        else:
            res.append(t)
    return res


GREEK_NAMES = dict(zip("αβγδεζηθικλμνξπρστυφχψω", "alpha beta gamma delta e zeta eta theta iota kappa lambda mu nu xi pi rho "
                                                     "sigma tau upsilon phi chi psi omega".split()))
LATEX_GREEK = re.compile(r"\\(var)?(" + "|".join(sorted(set(GREEK_NAMES.values()) | {"epsilon"}, key=len, reverse=True)) + r")(?![a-z])")
LONE_GREEK = re.compile(r"(?<![\u0370-\u03ff\u1f00-\u1fff])[\u03b1-\u03c9](?![\u0370-\u03ff\u1f00-\u1fff])")


def LATEX_GREEK_NAMES(text):
    """the Greek letters of a formula by name ("lambda"), however written (λ, \\lambda, 𝜆); ε / \\epsilon count as "e"."""
    import unicodedata
    t = unicodedata.normalize("NFKC", text).lower()
    names = ["e" if m.group(2) == "epsilon" else m.group(2) for m in LATEX_GREEK.finditer(t)]
    return names + [GREEK_NAMES[c] for c in t if c in GREEK_NAMES]


def _clean(text):
    """for comparing a draft and its rewrite: a maths letter is the same word as λ, \\lambda or 𝜆 ("lambda"); OCR's ε for an
    ê and \\epsilon / \\varepsilon are all "e" (Greek words, letters next to other Greek letters, stay as they are)"""
    import unicodedata
    text = unicodedata.normalize("NFKC", text).lower()
    text = LATEX_GREEK.sub(lambda m: " " + ("e" if m.group(2) == "epsilon" else m.group(2)) + " ", text)
    text = LONE_GREEK.sub(lambda m: " " + GREEK_NAMES[m.group(0)] + " ", text)
    return re.sub(r"[-\xad\ufffe\x02]\s*\n\s*|[\xad\ufffe\x02]", "", text)     # words hyphenated across lines


def _words(text):
    """words for the distinct-word and text-layer checks (short Latin words left out)"""
    return [t for t in script_tokens(_clean(TOKEN_RE.sub(" ", text))) if len(t) >= 3 or not t.isascii()]


TOKEN_RE = re.compile(r"<<KEEP \d+>>", re.I)


def protect(text):
    """-> (text with figures/anchors replaced by <<KEEP n>>, {token: original})"""
    kept = {}

    def sub(m):
        tok = TOKEN.format(len(kept) + 1); kept[tok] = m.group(0); return tok
    return KEEP.sub(sub, text), kept


def restore(text, kept):
    for tok, orig in kept.items():
        text = text.replace(tok, orig)
    return text


def repair_tokens(new, body, kept):
    """the LLM often drops or repeats a <<KEEP n>>: repeats are removed, a dropped one goes back next to the text that
    surrounded it in the draft (anchors at the start of the paragraph they belong to, figures as their own paragraph)"""
    new = TOKEN_RE.sub(lambda m: m.group(0) if m.group(0) in kept else "", new)   # tokens the LLM made up
    for tok in kept:
        at = new.find(tok)
        if at >= 0:
            new = new[:at + len(tok)] + new[at + len(tok):].replace(tok, "")
            continue
        i = body.find(tok)
        after, before = _words(body[i + len(tok):])[:5], _words(body[:i])[-5:]
        paras = new.split("\n\n")
        norm = [" ".join(_words(p)) for p in paras]
        anchor = kept[tok].startswith("<span")
        j = next((k for k, t in enumerate(norm) if after and " ".join(after) in t), None)
        if j is not None:
            if anchor:
                paras[j] = tok + paras[j]
            else:
                paras.insert(j, tok)
        else:
            j = next((k for k in range(len(norm) - 1, -1, -1) if before and " ".join(before) in norm[k]), None)
            if j is None:                                      # same relative place in the page
                j = min(len(paras), round(len(_words(body[:i])) / max(1, len(_words(body))) * len(paras))) - 1
            paras.insert(j + 1, tok)
        new = "\n\n".join(paras)
    return new


def _tokens(text):
    """every word and number (also 1-2 characters: 9V, mA, IC), LaTeX commands left out (\\text, \\frac...)"""
    text = re.sub(r"</?[a-zA-Z][^>]*>", " ", TOKEN_RE.sub(" ", text))             # HTML tags (<math>, <br>) are markup
    text = re.sub(r"(?m)^([ \t>]*)(```|~~~)[\w+#.-]*", r"\1 ", text)
    text = STALE_NUMBER.sub(r"\1", text)                                          # Marker's "1. 2. item" = the page's "2. item"                                # a code block's language is markup too
    return script_tokens(re.sub(r"\\[a-z]+", " ", _clean(text)))           # 1.2K and 1.2\\text{K} give the same tokens


CONFUSABLE = set(zip("εοννρκιτχαμ", "eovyrkitxau"))     # Greek letters an OCR gives for Latin ones that look the same


def _changes(draft_t, new_t, layer_t):
    """-> (lost, added) word Counters. lost: draft words gone from the rewrite (only words that are on the page, when the
    PDF has a text layer: OCR junk may go). added: rewrite words on neither the draft nor the page, except near-spellings
    of a lost word (an OCR fix: "rcsistor" -> "resistor")"""
    from collections import Counter
    from difflib import SequenceMatcher
    D, N, L = Counter(draft_t), Counter(new_t), Counter(layer_t)
    lost = D - N
    added = Counter({w: n for w, n in (N - (D + L)).items() if not w.isdigit()})     # a layer word the draft lacked once more (a footer)
    added.update({w: n for w, n in (N - (D | L)).items() if w.isdigit()})          # numbers: exact
    for w in list(added):                                   # OCR read a Latin letter as Greek (ê -> ε): the fix is no new word
        for m in list(lost):
            if (w, m) in CONFUSABLE or (m, w) in CONFUSABLE:
                k = min(added[w], lost[m]); added[w] -= k; lost[m] -= k
    added, lost = +added, +lost
    if L:
        lost = Counter({w: n for w, n in lost.items() if w in L})
    for w in list(added):
        for m in list(lost):
            if not w.isdigit() and not m.isdigit() and SequenceMatcher(None, w, m).ratio() >= 0.75:
                k = min(added[w], lost[m]); added[w] -= k; lost[m] -= k
    return +lost, +added


def brackets(text):
    """ceiling / floor brackets of a text, by kind: a rewrite may not turn one into the other (a model "normalises" the
    page's ⌈x⌉ to the far more common ⌊x⌋: Case 15 ex. 49b). Only the audit can, after a second look at the image."""
    return (len(re.findall(r"\\[lr]ceil\b|[⌈⌉]", text)), len(re.findall(r"\\[lr]floor\b|[⌊⌋]", text)))


def accept(draft, new, kept, layer="", detail=None):
    """(True, "") if the rewrite loses and invents nothing, else (False, why); detail (a list) gets what differed"""
    if not new.strip():
        return False, "empty"
    for tok in kept:
        if new.count(tok) != 1:
            return False, "figure/anchor lost or repeated"
    if len(TOKEN_RE.findall(new)) != len(kept):
        return False, "invented token"
    if brackets(new) != brackets(draft):
        return False, "ceiling/floor changed"
    boxes = lambda t: (len(re.findall(r"\[[xX✓✔]\]|[☑☒✅]", t)), len(re.findall(r"\[ \]|☐", t)))
    if boxes(new) != boxes(draft):
        return False, "checkbox changed"                    # Case 25: "[X] Step 1" came back unticked
    heads = lambda t: len(re.findall(r"(?m)^#{1,6} \S", t))
    if heads(new) < heads(draft):
        return False, "heading lost"                        # "## Case 08: ..." flattened into a plain line
    if new.count("\\$") < draft.count("\\$"):
        return False, "escaped $ unescaped"                 # "(\$\$ ... \$\$)" in text would become maths
    dt, nt, lt = _tokens(draft), _tokens(new), _tokens(layer)
    lost, added = _changes(dt, nt, lt)
    shapes = sum(0x25A0 <= ord(c) <= 0x25FF for c in new) - sum(0x25A0 <= ord(c) <= 0x25FF for c in draft)
    for w in ("0", "o"):                                    # OCR's 0 / O for a drawn box (▯ ▭ □) the rewrite restored
        k = min(max(0, shapes), lost.get(w, 0))
        if k:
            lost[w] -= k; shapes -= k
    lost = +lost
    if detail is not None and (lost or added):
        detail.append(f"lost {sorted(lost.elements())[:8]} added {sorted(added.elements())[:8]}")
    if any(w.isdigit() for w in lost) or any(w.isdigit() for w in added):
        return False, "number changed or lost"              # 9V -> 6V, 20 mA -> 2 mA
    if added:
        return False, "words added"                         # on neither the draft nor the page: made up
    if sum(lost.values()) > max(2, 0.01 * len(dt)):         # a symbol the LaTeX spells as a command (Ω -> \\Omega)
        return False, "draft words lost"
    dw, nw = _words(draft), _words(new)
    if len(set(dw)) >= 15 and len(set(dw) & set(nw)) < WORDS_KEPT * len(set(dw)):
        return False, "draft words lost"
    lw = _words(layer)
    if len(lw) >= 20:
        cov = lambda ws: sum(w in ws for w in lw) / len(lw)
        if cov(set(nw)) < cov(set(dw)) - 0.01:
            return False, "page text lost"
    return True, ""


def _unwrap(text):
    t = text.strip()
    m = re.fullmatch(r"```(?:markdown|md)?\s*\n(.*)\n```", t, re.S)
    return (m.group(1) if m else t).strip()


import threading as _threading
LLM_GATE = _threading.BoundedSemaphore(10 ** 6)       # LLM requests this worker may have in flight (set_llm_inflight)


def set_llm_inflight(n):
    """at most n LLM requests in flight from this worker (figures, review, audit together; Marker's own steps aside). The
    nested pools (describe x graph answers, review x audit) could otherwise put hundreds in the servers' queues, and a request
    waiting longer than its 900 s timeout makes a page keep its unreviewed draft. 0 = no cap."""
    global LLM_GATE
    LLM_GATE = _threading.BoundedSemaphore(int(n) if n and int(n) > 0 else 10 ** 6)


def ask_page(llm_url, model, jpeg, prompt, max_tokens, timeout=900, schema=None):
    """-> (text, finish_reason); schema: a JSON schema the answer must follow"""
    body = {"model": model, "temperature": 0, "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + base64.b64encode(jpeg).decode()}},
                {"type": "text", "text": prompt}]}]}
    if schema:
        body["response_format"] = {"type": "json_schema", "json_schema": {"name": "answer", "strict": True, "schema": schema}}
    req = urllib.request.Request(llm_url.rstrip("/") + "/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", "Authorization": "Bearer local"})
    with LLM_GATE:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            c = json.load(r)["choices"][0]
    return c["message"]["content"] or "", c.get("finish_reason")


def page_inputs(src, page_ids):
    """{page id: (jpeg bytes, text layer)} rendered one after another (pdfium is not thread-safe)"""
    out, src = {}, Path(src)
    if src.suffix.lower() != ".pdf":
        try:
            im = Image.open(src).convert("RGB")
        except Exception:
            return out                                     # not a page image (e.g. EPUB/HTML): nothing to compare with
        im.thumbnail((PAGE_PX, PAGE_PX)); buf = io.BytesIO(); im.save(buf, "JPEG", quality=90)
        return {p: (buf.getvalue(), "") for p in page_ids}
    import pypdfium2 as pdfium
    pdf = pdfium.PdfDocument(str(src))
    try:
        for p in page_ids:
            if p >= len(pdf):
                continue
            page = pdf[p]
            w, h = page.get_size()
            im = page.render(scale=PAGE_PX / max(w, h)).to_pil().convert("RGB")
            buf = io.BytesIO(); im.save(buf, "JPEG", quality=90)
            tp = page.get_textpage()
            try:
                layer = tp.get_text_range()
            finally:
                tp.close()
            out[p] = (buf.getvalue(), layer.replace("￾", "-").replace("\r", ""))
    finally:
        pdf.close()
    return out


def _need(s):
    """tokens a text takes: ~3 Latin characters per token, but about one per character in Bangla, Chinese, Japanese..."""
    ascii_n = sum(1 for c in s if c < "\x80")
    return int(ascii_n / CHARS_PER_TOKEN + 1.3 * (len(s) - ascii_n))


def _review_part(body, kept, jpeg, layer, llm_url, model, room, keep_headers, part=""):
    """one LLM call for (a part of) a page: -> (rewritten text with tokens, None) or (None, why)"""
    out_tokens = int(2.0 * _need(body)) + 800              # tables and LaTeX take more tokens than the draft (1.6x + 600 cut 3 suite pages off)
    if _need(body) + _need(layer) + out_tokens > room:
        layer = ""                                          # dense page: the image alone must do
    prompt = PROMPT.format(
        layer_note=", and the TEXT LAYER of the PDF page (exact characters, but its reading order and layout may be wrong)" if layer else "",
        layer_rule=" and the text layer" if layer else "",
        header_rule="",                                    # removing headers/page numbers would fail the number check
        draft=body, layer=f"\n\nTEXT LAYER:\n{layer}" if layer else "")
    if part:
        prompt = prompt.replace("Return the corrected Markdown of this page", f"The DRAFT is {part} of this page only: return the corrected Markdown of that part")
    try:
        text, finish = ask_page(llm_url, model, jpeg, prompt, min(out_tokens, room - _need(body) - _need(layer)))
    except Exception as e:
        return None, f"LLM error: {type(e).__name__}"
    if finish == "length":
        return None, "answer cut off"
    if not _unwrap(text).strip():
        return None, "empty"
    new = repair_tokens(_unwrap(text), body, kept)
    detail = []
    ok, why = accept(body, new, kept, layer, detail)
    if ok:
        return new, None
    merged, taken = merge_paragraphs(body, new, kept, layer)    # the good paragraphs of a rewrite with one bad one
    if taken:
        return merged, None
    return None, f"{why}|{detail[0]}" if detail else why


def layer_prefers(draft, new, layer, need=0.9):
    """the PDF's own text settles a paragraph the strict check refused: every word and number of the rewrite is on the page,
    and the stretch of the page's text it matches best agrees with the rewrite (>= 90%) more than with the draft
    (Case 15: the OCR's \\sqrt[2]{2} where the page prints \\sqrt{2}; a rewrite that drops a real number matches worse)"""
    from collections import Counter
    from difflib import SequenceMatcher
    lt, nt, dt = _tokens(layer), _tokens(new), _tokens(draft)
    if not nt or Counter(nt) - Counter(lt):
        return False

    def best(t):
        n, top = len(t), 0.0
        for i in range(0, max(1, len(lt) - n + 1)):
            top = max(top, SequenceMatcher(None, t, lt[i:i + n], autojunk=False).ratio())     # a stretch as long as it
        return top
    rn = best(nt)
    return rn >= need and rn > best(dt)


def _take(da, nb, kept, layer):
    """-> the rewrite nb of draft paragraph(s) da if it may be used, else None"""
    sub = {t: v for t, v in kept.items() if t in da}
    if not nb.strip() or any(nb.count(t) != 1 for t in sub) or len(TOKEN_RE.findall(nb)) != len(sub):
        return None
    verdict = accept(da, nb, sub, layer)
    if not verdict[0] and verdict[1] == "escaped $ unescaped":
        loose = re.findall(r"(?<!\\)\$", nb)                   # printed "$...$" text the rewrite turned into maths:
        if len(loose) == da.count("\\$"):                     # escaped again, its other fixes (\sqrt[2] -> \sqrt) kept
            nb = re.sub(r"(?<!\\)\$", lambda m: "\\$", nb)
            verdict = accept(da, nb, sub, layer)
    if not verdict[0] and layer and verdict[1] in ("number changed or lost", "words added", "draft words lost"):
        verdict = (layer_prefers(da, nb, layer), "")
    return nb if verdict[0] else None


def merge_paragraphs(body, new, kept, layer=""):
    """paragraph by paragraph: each changed group of the rewrite is used if it passes on its own (accept(), or the PDF's
    text agrees with it), else the draft's stays: one made-up line no longer throws away a whole page of fixes. A group
    with as many paragraphs on both sides is judged pair by pair. -> (text, groups taken)"""
    from difflib import SequenceMatcher
    a, b = body.split("\n\n"), new.split("\n\n")
    norm = lambda p: " ".join(p.split())
    out, taken = [], 0
    for tag, i1, i2, j1, j2 in SequenceMatcher(None, [norm(x) for x in a], [norm(x) for x in b], autojunk=False).get_opcodes():
        if tag == "equal":
            out += b[j1:j2]                                    # same words, the rewrite's spacing
            continue
        pairs = list(zip(a[i1:i2], b[j1:j2])) if i2 - i1 == j2 - j1 else [("\n\n".join(a[i1:i2]), "\n\n".join(b[j1:j2]))]
        for da, nb in pairs:
            got = _take(da, nb, kept, layer)
            if got is not None:
                out.append(got); taken += 1
            elif da.strip():
                out.append(da)                                 # nothing of the draft is ever dropped
    return "\n\n".join(x for x in out if x.strip()), taken


def _split(body, limit):
    """paragraph groups of at most `limit` tokens each (a dense page in parts)"""
    parts, cur = [], []
    for para in body.split("\n\n"):
        if cur and _need("\n\n".join(cur + [para])) > limit:
            parts.append("\n\n".join(cur)); cur = []
        cur.append(para)
    return parts + (["\n\n".join(cur)] if cur else [])


def layer_for(part, layer, share=0.5):
    """the lines of a page's text layer that belong to one part of it (half their words or more are in the part): a part
    reviewed on its own must not be handed the rest of the page, or words of other parts could be copied in"""
    have = set(_words(part))
    keep = [l for l in layer.splitlines() if _words(l) and sum(w in have for w in _words(l)) >= share * len(_words(l))]
    return "\n".join(keep)


def review_page(draft, jpeg, layer, llm_url, model, ctx, keep_headers=False, part_tokens=0):
    """-> (final page text, None if rewritten else why the draft was kept). part_tokens: review the page in sections of at
    most this many tokens (each against the page image and its own lines of the text layer): more careful, more calls"""
    if not re.search(r"\w", draft) and not layer.strip():
        return draft, "blank page"
    body, kept = protect(draft)
    room = ctx - IMAGE_TOKENS - 600                         # prompt, chat template
    hard = int((room - 600) / 2.6)                          # draft + 1.6x draft + 600 must fit
    limit = min(hard, part_tokens) if part_tokens else hard
    parts = [body] if _need(body) <= limit else _split(body, limit)
    if any(_need(x) > hard for x in parts):                 # (a paragraph longer than a section is a section of its own)
        return draft, "page too long for the LLM context"   # one paragraph alone is too long
    out, whys = [], []
    for k, part in enumerate(parts):
        mine = {t: v for t, v in kept.items() if t in part}
        new, why = _review_part(part, mine, jpeg, layer if len(parts) == 1 else layer_for(part, layer), llm_url, model, room, keep_headers,
                                f"part {k + 1} of {len(parts)}" if len(parts) > 1 else "")
        out.append(new if why is None else part); whys.append(why)
    if all(whys):
        return draft, whys[0]
    return restore("\n\n".join(out), kept), next((w for w in whys if w), None)


def page_settled(draft, layer):
    """a page the review can't improve in content: every word and number of it is in the PDF's own text (no OCR), nothing of
    that text is left out (a running header aside), and it has no maths, code or table to convert. Digital books are mostly
    such pages. (Its formatting, e.g. heading levels, is Marker's.)"""
    from collections import Counter
    body = TOKEN_RE.sub(" ", protect(draft)[0])
    if not layer.strip() or re.search(r"(?<!\\)\$|```|^\s*\|", body, re.M):
        return False
    d, l = Counter(_tokens(body)), Counter(_tokens(layer))
    if sum(d.values()) < 20 or d - l:
        return False
    return sum((l - d).values()) <= max(8, 0.03 * sum(l.values()))


def review_pages(md_path, src, llm_url, model, concurrency=2, ctx=8192, keep_headers=False, use_layer=True, part_tokens=0,
                 after=None, skip_settled=False):
    """reviews every page of a Markdown file written with page separators (in place).
    after(text, jpeg, layer) -> text: a next step for each page as soon as its review is done (the audit), so no page waits
    for the slowest page's review. -> {"pages", "rewritten", "kept_draft": {why: count}}"""
    md_path = Path(md_path); md = md_path.read_text(encoding="utf-8")
    parts = PAGE_SPLIT.split(md)
    pre, pages = parts[0], [[int(parts[i]), parts[i + 1]] for i in range(1, len(parts), 2)]
    inputs = page_inputs(src, [p for p, _ in pages])
    todo = [k for k, (p, _) in enumerate(pages) if p in inputs]

    def one(k):
        p, text = pages[k]
        jpeg, layer = inputs[p]
        layer = layer if use_layer else ""
        if skip_settled and page_settled(text, layer):        # balanced: the PDF's own text already is this page
            new, why = text.strip("\n"), "settled by the text layer"
        else:
            new, why = review_page(text.strip("\n"), jpeg, layer, llm_url, model, ctx, keep_headers, part_tokens)
        return k, ((after(new, jpeg, layer) if after else new), why)
    stats = {"pages": len(todo), "rewritten": 0, "kept_draft": {}, "examples": []}
    with ThreadPoolExecutor(max(1, concurrency)) as ex:
        for k, (text, why) in ex.map(one, todo):
            pages[k][1] = text
            if why:
                reason, _, what = why.partition("|")
                stats["kept_draft"][reason] = stats["kept_draft"].get(reason, 0) + 1
                if what and len(stats["examples"]) < 10:
                    stats["examples"].append(f"page {pages[k][0]}: {reason}: {what}")
            else:
                stats["rewritten"] += 1
    md_path.write_text(pre + "".join(f"\n\n{{{p}}}{'-' * 48}\n\n{t.strip(chr(10))}" for p, t in pages) + "\n", encoding="utf-8")
    return stats


# ----------------------------------------------------------------- line-by-line audit (accurate quality)
# A whole-page rewrite keeps missing small things: an OCR's 李生 for 孪生, \sqrt[2]{2} for the printed \sqrt{2}, y^n for y^m.
# Each section of a page is proofread against the image; the model lists exact corrections, and each one is applied on its own
# only when it is a small local change that the PDF's own text confirms, or that a second, independent look at the image
# (which of the two versions is printed?) picks.
AUDIT_PROMPT = """You proofread one SECTION of a document page that was converted to Markdown, against the page image (and the
page's text layer when there is one: its characters are exact, its order may not be).
Go through the section line by line and compare every word, number, symbol and formula with the image. Look closely at:
characters an OCR confuses (Chinese/Japanese look-alike characters, l/1, O/0, rn/m), maths symbols (ceiling = \\lceil \\rceil
vs floor = \\lfloor \\rfloor, \\sqrt{2} vs \\sqrt[2]{2}, primes, hats, sub- and superscripts), and drawn symbols LaTeX has no
command for (write them as Unicode, e.g. \\text{▯}).
For every error, give "find": the exact wrong text copied from the SECTION, only the wrong word(s) with a few words around them
so it occurs once (under 60 characters, never a whole paragraph), "replace": the same text corrected, and "why" (a few words). Change only what is wrong; keep the Markdown formatting and escapes (\\$, \\_) as they are. Tokens
like <<KEEP 2>> are figures: never touch them. Do not list text that is already right. No error: an empty list.

SECTION:
<<SECTION>>

TEXT LAYER:
<<LAYER>>"""
AUDIT_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["fixes"], "properties": {"fixes": {"type": "array",
    "items": {"type": "object", "additionalProperties": False, "required": ["find", "replace", "why"],
              "properties": {"find": {"type": "string"}, "replace": {"type": "string"}, "why": {"type": "string"}}}}}}
VERIFY_PROMPT = """Look carefully at the page image. A passage of it reads: «<<CONTEXT>>»
Which of these two is what the page actually shows at that place?
A: «<<A>>»
B: «<<B>>»
Answer "A", "B", or "neither" if the image shows neither or you cannot tell."""
VERIFY_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["answer"],
                 "properties": {"answer": {"type": "string", "enum": ["A", "B", "neither"]}}}
AUDIT_SECTION_PARAS = 6


def _find_once(text, find):
    """where an audit fix applies: its "find" exactly once in the text (also with doubled backslashes undone, or with any
    spacing) -> the matched text, or None"""
    for c in dict.fromkeys((find, find.replace("\\\\", "\\"))):
        if c.strip() and text.count(c) == 1:
            return c
        words = c.split()
        if words:
            ms = list(re.finditer(r"\s+".join(map(re.escape, words)), text))
            if len(ms) == 1:
                return ms[0].group(0)
    return None


def _local_change(a, b):
    """a correction, not a rewrite: similar and about as long"""
    from difflib import SequenceMatcher
    return 0 < len(a) <= 600 and len(b) <= 1.5 * len(a) + 20 and SequenceMatcher(None, a, b, autojunk=False).ratio() >= 0.5


def _verify(jpeg, llm_url, model, context, old, new):
    """a second, independent look: is the proposed text what the page shows? Asked twice, the two versions in both orders:
    both answers must pick the new text (a model that leans to "A", or to the more common symbol, can't confirm a fix:
    Cases 1/5/15, accurate: ceiling -> floor was "confirmed" by one question) -> True / False"""
    def ask(new_first):
        a, b = (new, old) if new_first else (old, new)
        prompt = VERIFY_PROMPT.replace("<<CONTEXT>>", context[:600]).replace("<<A>>", a).replace("<<B>>", b)
        try:
            text, finish = ask_page(llm_url, model, jpeg, prompt, 20, schema=VERIFY_SCHEMA)
            return json.loads(text).get("answer") == ("A" if new_first else "B")
        except Exception:
            return False
    return ask(True) and ask(False)                    # in turn: the second reuses the image; a first "no" ends it


def audit_section(section, jpeg, layer, llm_url, model, stats):
    """-> the section with the audit's confirmed corrections applied (stats: counts by outcome, examples)"""
    body, kept = protect(section)
    prompt = AUDIT_PROMPT.replace("<<SECTION>>", body).replace("<<LAYER>>", layer_for(body, layer) or "(none)")
    try:
        text, finish = ask_page(llm_url, model, jpeg, prompt, 800, schema=AUDIT_SCHEMA)     # corrections are short
        fixes = json.loads(text).get("fixes") or [] if finish != "length" else []
    except Exception:
        stats["errors"] = stats.get("errors", 0) + 1
        return section
    import threading
    lock = stats.setdefault("_lock", threading.Lock())

    def count(k):
        with lock:
            stats[k] = stats.get(k, 0) + 1
    for fx in fixes:
        find, new = str(fx.get("find", "")), str(fx.get("replace", ""))
        if not find.strip() or find == new or find.replace("\\\\", "\\") == new.replace("\\\\", "\\"):
            continue                                            # "no error found" listed anyway
        old = _find_once(body, find)
        if old is None:
            count("not found"); continue
        if old != find and "\\\\" in find and old == find.replace("\\\\", "\\"):
            new = new.replace("\\\\", "\\")                     # the answer doubled its backslashes: undo in both
        if not _local_change(old, new) or TOKEN_RE.findall(old) != TOKEN_RE.findall(new):
            count("too big"); continue
        marks = lambda t: (len(re.findall(r"(?<!\\)\$", t)), len(re.findall(r"(?<!\\)[*_`]", t)))
        if marks(new) < marks(old):
            count("would drop maths or formatting"); continue   # "*stick*" -> "stick", "$\\lambda_i$" -> "lambda_i"
        candidate = body.replace(old, new, 1)
        para = next((p for p in body.split("\n\n") if old in p), old)
        plain = lambda x: re.sub(r"\\([$_*#])", r"\1", x).strip()        # Markdown escapes: the text layer has none
        if layer and len(plain(old)) >= 8 and plain(old) in layer:      # the draft has the page's exact characters there
            count("the page's text has it as written"); continue  # run 1-3 suite: "Figure 2.1 T" (cut off on the page) -> "Figure 2.1"
        if layer and ((plain(old) not in layer and plain(new) in layer) or layer_prefers(para, para.replace(old, new, 1), layer)):
            how = "confirmed by the text layer"
        elif _verify(jpeg, llm_url, model, para, old, new):
            how = "confirmed by a second look"
        else:
            count("refused"); continue
        body = candidate
        count(how)
        with lock:
            if len(stats.setdefault("examples", [])) < 12:
                stats["examples"].append(f"{old[:60]} -> {new[:60]} ({how})")
    return restore(body, kept)


def settled(section, layer):
    """a section the audit can't improve: every word and number of it is in the PDF's own text (no OCR in it) and it has
    no maths, code or table, where a converted symbol could still be wrong. Real digital PDFs are mostly such pages."""
    from collections import Counter
    if not layer or re.search(r"(?<!\\)\$|```|^\s*\|", section, re.M):
        return False
    words = _tokens(TOKEN_RE.sub(" ", section))
    return len(words) >= 5 and not (Counter(words) - Counter(_tokens(layer)))


def audit_page(text, jpeg, layer, llm_url, model, stats, sizes=(AUDIT_SECTION_PARAS,), workers=1):
    """one page, pass after pass (sections of `sizes[i]` paragraphs; a second pass in smaller sections finds what the first
    missed). Sections go one after the other: the server keeps the page image from the previous call (2.8 s of GPU time
    saved per call), and many pages are audited at once anyway, so the GPUs stay busy. -> the page with the corrections"""
    import threading
    lock = stats.setdefault("_lock", threading.Lock())
    for size in sizes:
        paras = text.strip("\n").split("\n\n")
        secs = ["\n\n".join(paras[j:j + int(size)]) for j in range(0, len(paras), int(size))]

        def one(sec):
            if not re.search(r"\w", sec):
                return sec
            if settled(sec, layer):
                with lock:
                    stats["settled by the text layer"] = stats.get("settled by the text layer", 0) + 1
                return sec
            with lock:
                stats["sections"] = stats.get("sections", 0) + 1
            return audit_section(sec, jpeg, layer, llm_url, model, stats)
        with ThreadPoolExecutor(max(1, min(workers, len(secs)))) as ex:
            text = "\n\n".join(ex.map(one, secs))
    return text


def _public(stats):
    return {k: v for k, v in stats.items() if not k.startswith("_")}


def audit_pages(md_path, src, llm_url, model, concurrency=2, use_layer=True, sizes=(AUDIT_SECTION_PARAS,)):
    """proofreads every page of a Markdown file with page separators (in place) -> stats. (With the page review on, the
    worker audits each page right after its review instead: review_pages(after=...))"""
    md_path = Path(md_path); md = md_path.read_text(encoding="utf-8")
    parts = PAGE_SPLIT.split(md)
    pre, pages = parts[0], [[int(parts[i]), parts[i + 1]] for i in range(1, len(parts), 2)]
    inputs = page_inputs(src, [p for p, _ in pages])
    stats = {}

    def one(k):
        jpeg, layer = inputs[pages[k][0]]
        return k, audit_page(pages[k][1], jpeg, layer if use_layer else "", llm_url, model, stats, sizes)
    with ThreadPoolExecutor(max(1, concurrency)) as ex:
        for k, text in ex.map(one, [k for k, (p, _) in enumerate(pages) if p in inputs]):
            pages[k][1] = text
    md_path.write_text(pre + "".join(f"\n\n{{{p}}}{'-' * 48}\n\n{t.strip(chr(10))}" for p, t in pages) + "\n", encoding="utf-8")
    return _public(stats)


# ----------------------------------------------------------------- quality: fast / balanced / accurate
QUALITY_PROFILES = {
    # Marker's own LLM steps (+ its inline-maths rewrite), figure descriptions (crop to the figure, flow-diagram graph
    # answers/checks), page review (sections of at most N tokens; 0 = the whole page at once), line-by-line audit (paragraphs
    # per section, one pass per number). Everything without an LLM (tables, drawn formulas, text-layer repair, Indic repair,
    # tidy-up) runs at every quality.
    # fast: no Marker LLM steps and no page rewrite (the two big costs); the cheap checks that fix the most stay: figures
    #       cropped to themselves, one graph read, and a light audit (short answers: only the corrections)
    "fast": dict(MARKER_LLM=False, REDO_INLINE_MATH=False, DESCRIBE_FIGURES=True, CORE_CROP=True, GRAPH_TRIES=1, GRAPH_CHECKS=0,
                 PAGE_REVIEW=False, REVIEW_PART_TOKENS=0, REVIEW_SKIP_SETTLED=True, AUDIT=True, AUDIT_SECTIONS=(12,),
                 TRANSCRIBE_TEXT_IMAGES=False),
    # balanced: a page its PDF text settles completely is not rewritten (only its formatting could change); a picture that is
    #       mostly text (a newspaper page, a letter) is transcribed tile by tile, not only described
    "balanced": dict(MARKER_LLM=True, REDO_INLINE_MATH=False, DESCRIBE_FIGURES=True, CORE_CROP=True, GRAPH_TRIES=2, GRAPH_CHECKS=2,
                     PAGE_REVIEW=True, REVIEW_PART_TOKENS=0, REVIEW_SKIP_SETTLED=True, AUDIT=True, AUDIT_SECTIONS=(8,),
                     TRANSCRIBE_TEXT_IMAGES=True),
    # accurate: nothing left out; the review section by section and two audit passes (the second in smaller sections finds
    #       what the first missed: run 1 of the suite caught 数字李生 but not 音服热线)
    "accurate": dict(MARKER_LLM=True, REDO_INLINE_MATH=True, DESCRIBE_FIGURES=True, CORE_CROP=True, GRAPH_TRIES=3, GRAPH_CHECKS=2,
                     PAGE_REVIEW=True, REVIEW_PART_TOKENS=900, REVIEW_SKIP_SETTLED=False, AUDIT=True, AUDIT_SECTIONS=(6, 3),
                     TRANSCRIBE_TEXT_IMAGES=True),
}


def quality_option(opts, key):
    """an option as the job's quality sets it, unless the job sets it itself"""
    if opts.get(key) is not None:
        return opts[key]
    return QUALITY_PROFILES.get(str(opts.get("QUALITY") or "balanced").lower(), QUALITY_PROFILES["balanced"])[key]


# ----------------------------------------------------------------- LLM answers in JSON that carry LaTeX
# The model writes "\boldsymbol{\rho}" inside a JSON string with one backslash: JSON then reads \b as a backspace and \r
# as a carriage return, and the equation becomes "oldsymbol{<newline>ho}". Before parsing, a lone backslash in front of a
# LaTeX command is doubled. Only command names are touched, so real JSON escapes (\n before a word, \u00e9) stay as they are.
LATEX_CMDS = ("alpha beta gamma delta epsilon varepsilon zeta eta theta vartheta iota kappa lambda mu nu xi pi varpi rho varrho "
              "sigma varsigma tau upsilon phi varphi chi psi omega Gamma Delta Theta Lambda Xi Pi Sigma Upsilon Phi Psi Omega "
              "frac dfrac tfrac sqrt sum prod int iint oint lim log ln exp sin cos tan sec csc cot arcsin arccos arctan sinh cosh "
              "tanh min max sup inf det dim ker deg gcd Pr mod pmod bmod nabla partial infty times div cdot cdots ldots dots vdots "
              "ddots pm mp neq ne leq geq le ge ll gg approx equiv sim simeq cong propto in notin ni subset subseteq supset "
              "supseteq cup cap setminus emptyset varnothing forall exists nexists neg not land lor to rightarrow leftarrow "
              "Rightarrow Leftarrow leftrightarrow Leftrightarrow mapsto implies iff uparrow downarrow left right big Big bigg Bigg "
              "bigl bigr langle rangle lfloor rfloor lceil rceil lvert rvert lVert rVert vert Vert mathbf mathbb mathcal mathrm "
              "mathit mathsf mathtt mathfrak boldsymbol bm text textbf textit textrm texttt textsf emph hat bar tilde vec dot ddot "
              "breve check acute grave widehat widetilde overline underline overbrace underbrace overrightarrow binom tbinom "
              "dbinom choose begin end quad qquad hspace vspace mbox operatorname displaystyle textstyle scriptstyle prime circ "
              "bullet star ast dagger perp parallel mid nmid angle triangle square top bot hbar ell Re Im aleph wp neq nleq ngeq "
              "nsubseteq newline tag label limits nolimits substack stackrel overset underset cases matrix pmatrix bmatrix vmatrix "
              "array hline frown smile because therefore").split()
_LATEX_ESC = re.compile(r"(?<!\\)\\(?=(?:" + "|".join(sorted(set(LATEX_CMDS), key=len, reverse=True)) + r")(?![A-Za-z]))")


def latex_json_loads(raw):
    """json.loads for LLM answers: a lone backslash before a LaTeX command is doubled first"""
    import json
    return json.loads(_LATEX_ESC.sub(r"\\\\", raw))


CHATTER = re.compile(r"(?i)\b(the (image|picture|figure|equation|text|table|block)( provided)? (shows|appears|contains|seems|is (likely|probably|a))|"
                     r"appears to (be|show)|is likely a typo|(likely|probably) (a|an) (typo|error|mistake)|i (cannot|can't|am unable|could not)|"
                     r"as an ai|here is the (corrected|latex)|"
                     r"(is|are) (already )?(correct(ly)?|properly|well)[ -]?(formatted|written|aligned)|the (current )?latex( code)? is|"
                     r"should be (written|placed|enclosed|formatted|aligned|separated)|needs? to be (separated|placed|aligned|enclosed|written)|"
                     r"to ensure (katex|mathjax|proper|correct)|\bmath tags?\b)\b")       # ("No corrections needed" is Marker's own "keep it")
MATH_MARK = re.compile(r"<math|\\[a-zA-Z]+|\$|[=^_]")
MARKER_INPUT = re.compile(r"\*\*Input:\*\*\s*```html\n(.*)\n```\s*$", re.S)


def check_marker_llm(prompt, response):
    """Marker's own LLM steps (inline maths, equations, tables...) rewrite blocks without any check. -> (keep, why).
    A rewrite is dropped (Marker keeps its own text) when it talks about the image instead of transcribing it, and, for a
    text block rewritten for its maths, when it loses or changes words or numbers of the original (accept())."""
    if not isinstance(response, dict):
        return True, ""
    m = MARKER_INPUT.search(prompt or "")
    original = m.group(1) if m else ""
    for key, value in response.items():
        if key == "analysis" or not isinstance(value, str):
            continue
        if CHATTER.search(value) and not CHATTER.search(original):
            return False, "commentary instead of content"
        if (original and re.search(r"<math|\\[a-zA-Z]+", original) and len(value.split()) >= 8
                and not MATH_MARK.search(value) and "no corrections needed" not in value.lower()):
            return False, "maths replaced by prose"           # an equation came back as sentences about it
    for key in ("corrected_equation", "corrected_html", "html"):
        v = response.get(key)
        if original and isinstance(v, str) and "no corrections needed" not in v.lower() and brackets(v) != brackets(original):
            return False, "ceiling/floor changed"
    eq = response.get("corrected_equation")
    if original and isinstance(eq, str) and "no corrections needed" not in eq.lower():
        greek = lambda t: set(LATEX_GREEK_NAMES(t))
        before, after = greek(original), greek(eq)
        if before and after and not (before & after):
            return False, "symbols swapped"                   # Case 03: λ and ê came back as \\boldsymbol{\\nu}
    if original and isinstance(response.get("corrected_html"), str) and "no corrections needed" not in response["corrected_html"].lower():
        ok, why = accept(original, response["corrected_html"], {})
        if not ok:
            return False, why
    return True, ""


# ----------------------------------------------------------------- heading levels from the PDF's own outline
def _key(t):
    import unicodedata
    return "".join(script_tokens(unicodedata.normalize("NFKC", re.sub(r"<[^>]+>", "", t)).lower()))   # keeps vowel signs


def outline(src):
    """the PDF's bookmarks: [(level 0.., title key, page index)]; [] if it has (almost) none"""
    if not str(src).lower().endswith(".pdf"):
        return []
    import pypdfium2 as pdfium
    pdf = pdfium.PdfDocument(str(src))
    try:
        marks = []
        for b in pdf.get_toc(max_depth=8):
            d, title = b.get_dest(), (b.get_title() or "").strip()
            if d is not None and d.get_index() is not None and len(_key(title)) >= 3:
                marks.append((b.level, _key(title), d.get_index()))
    finally:
        pdf.close()
    return marks if len(marks) >= 3 else []


def apply_outline(md_path, marks):
    """page-by-page review cannot see the book's structure (Make Electronics came out with chapters, experiments and their
    sections all as ##). The PDF's outline has it: a heading matching a bookmark on its page (+-1) gets that bookmark's
    level (chapter #, experiment ##, section ###); a heading the outline doesn't list goes one level below the outline
    entry it follows; a short paragraph that is exactly a bookmark title becomes that heading. -> {"matched", "below", "promoted"}"""
    md_path = Path(md_path); md = md_path.read_text(encoding="utf-8")
    parts = PAGE_SPLIT.split(md)
    pre, pages = parts[0], [[int(parts[i]), parts[i + 1]] for i in range(1, len(parts), 2)]
    stats = {"matched": 0, "below": 0, "promoted": 0, "repeats_dropped": 0}
    if not marks or not pages:
        return stats
    prior = [m for m in marks if m[2] < pages[0][0]]
    cur = prior[-1][0] + 1 if prior else None              # heading level of the outline entry we are under
    chain = {}                                             # outline level -> title key of the entries we are inside
    for m in prior:
        chain = {l: k for l, k in chain.items() if l < m[0]}; chain[m[0]] = m[1]
    for pg in pages:
        near = [m for m in marks if abs(m[2] - pg[0]) <= 1]
        segs = CODE_BLOCK.split(pg[1])
        for k in range(0, len(segs), 2):
            paras = segs[k].split("\n\n")
            for i, para in enumerate(paras):
                line = para.strip()
                if not line or "\n" in line:
                    continue
                h = re.match(r"(#{1,6})\s+(.*)$", line)
                title = h.group(2) if h else line
                key = _key(title.replace("*", ""))
                if len(key) < 3:
                    continue
                if key in chain.values() and (h or len(line) <= 120):   # the title of a section we're in, again: a running header
                    paras[i] = None; stats["repeats_dropped"] += 1
                    continue
                m = next((m for m in near if m[1] == key), None) or (
                    next((m for m in near if min(len(key), len(m[1])) >= 12 and (m[1].startswith(key) or key.startswith(m[1]))), None) if h else None)
                if m:
                    level = min(6, m[0] + 1); cur = level
                    chain = {l: k for l, k in chain.items() if l < m[0]}; chain[m[0]] = m[1]
                    if not h:
                        if len(line) > 120 or re.match(r"(\||!\[|>|[-*+] |\d+[.)] |\$)", line):
                            continue
                        stats["promoted"] += 1
                    else:
                        stats["matched"] += 1
                elif h and cur:
                    level = min(6, cur + 1); stats["below"] += 1
                else:
                    continue
                paras[i] = para.replace(line, "#" * level + " " + title.strip(), 1)
            segs[k] = "\n\n".join(x for x in paras if x is not None)
        pg[1] = "".join(segs)
    md_path.write_text(pre + "".join(f"\n\n{{{p}}}{'-' * 48}\n\n{t.strip(chr(10))}" for p, t in pages) + "\n", encoding="utf-8")
    return stats


SENTENCE_END = re.compile(r"[.!?:;。！？｡」』）)\]\"”’»।॥؟۔።։;]\s*$")      # Latin, CJK, Indic, Arabic/Urdu, Ethiopic, Armenian, Greek
CAPTION = re.compile(r"\s*(\*\*|_)?\S{1,14}\.?\s*\d+([-.:]\d+)*\b")          # "Figure 3-2", "图3", "Рис. 5", "Abb. 2", "চিত্র ৪", "شكل 3"


def _plain(para):
    p = para.strip()
    return p and not re.match(r"(#|\||!\[|>|\$\$|```|~~~|<(?!span)|[-*+] |\d+[.)] )", p) and not CAPTION.match(p)


def join_pages(texts):
    """joins pages, mending a paragraph cut by the page break in any script: the page ends without closing punctuation
    and the next starts with text (English, which has the lower-case clue, is also handled anywhere by join_broken)"""
    out = ""
    for t in (x.strip("\n") for x in texts if x.strip()):
        if out:
            last, first = out.rsplit("\n\n", 1)[-1], t.split("\n\n", 1)[0]
            body = ANCHOR_START.sub("", first.lstrip())
            if (_plain(last) and _plain(first) and not SENTENCE_END.search(last) and body[:1].isalpha()
                    and not body[:1].isascii()):
                out += ("" if _no_space(last.rstrip()[-1]) and _no_space(body[0]) else " ") + first.lstrip() + t[len(first):]
                continue
            out += "\n\n"
        out += t
    return out


def finish_markdown(md_path, keep_pages=False):
    """removes the page separators (unless asked to keep them) and tidies the Markdown, in place"""
    md_path = Path(md_path); md = md_path.read_text(encoding="utf-8")
    if not keep_pages:
        parts = PAGE_SPLIT.split(md)
        md = join_pages([parts[0]] + parts[2::2])
    md_path.write_text(tidy(md), encoding="utf-8")


# ----------------------------------------------------------------- Markdown syntax, deterministic
FENCE = re.compile(r"^\s*(```|~~~)")


PLAIN_END = re.compile(r"[^\W\d_],?$|,$")                    # a paragraph that stops mid-sentence (letter or comma)
NOT_PLAIN = re.compile(r"(#|\||!\[|>|\$\$|```|~~~|<(?!span)|\[|[-*+] |\d+[.)] )")
ANCHOR_START = re.compile(r"^(?:<span id=\"[^\"]*\">\s*</span>)*")


def join_broken(md):
    """joins a paragraph cut by a page or column break ("because I consider it" + "the most essential...") when the first
    stops without punctuation and the next starts in lower case; code blocks and display maths are left alone"""
    out = []
    for k, seg in enumerate(re.split(r"(^(?:```|~~~)[^\n]*\n.*?^(?:```|~~~)[ \t]*$)", md, flags=re.M | re.S)):
        if k % 2:
            out.append(seg); continue
        paras = seg.split("\n\n")
        res = []
        for para in paras:
            prev = res[-1].rstrip() if res else ""
            body = ANCHOR_START.sub("", para.lstrip())
            if (prev and "\n" not in prev and PLAIN_END.search(prev) and not re.match(r"\s*(#|\||!\[|>|\$\$|<(?!span))", prev) and not CAPTION.match(prev)
                    and "\n|" not in prev and body[:1].islower() and not NOT_PLAIN.match(para.lstrip())):
                res[-1] = prev + " " + para.lstrip()
            else:
                res.append(para)
        out.append("\n\n".join(res))
    return "".join(out)


CODE_BLOCK = re.compile(r"(^(?:```|~~~)[^\n]*\n.*?^(?:```|~~~)[ \t]*$)", re.M | re.S)
AMOUNT = re.compile(r"(?<![\\$\w])\$(?=\d[\d,]*(?:\.\d+)?)")


def escape_currency(text):
    """US $34.99 CAN $40.99 -> US \\$34.99 CAN \\$40.99: with math on, the two $ would make a formula of the text between.
    A $ is money when it starts an amount and no formula closes after it (next $ right after a non-space, operators between)."""
    def fix(m):
        rest = text[m.end():]
        nxt = re.search(r"(?<!\\)\$", rest)
        inner = rest[:nxt.start()] if nxt else ""
        if nxt and inner and not inner[-1].isspace() and (re.search(r"[=+*/^_\\{}<>]|\s-\s", inner) or not re.search(r"\s", inner)):
            return "$"                                     # $5 + 3$ and $5$ are maths
        return "\\$"
    return AMOUNT.sub(fix, text)


def _hard_breaks(para):
    """a block of short lines (an address, a receipt, a poem) stays line by line: Markdown would run them together"""
    lines = para.strip("\n").split("\n")
    if (len(lines) >= 2 and all(0 < len(l.strip()) <= 80 for l in lines) and not any(l.endswith("  ") for l in lines[:-1])
            and not any(re.match(r"\s*(#|\||>|[-*+] |\d+[.)] |\$\$|```|~~~|<|!\[)", l) for l in lines)):
        lead = para[:len(para) - len(para.lstrip("\n"))]
        return lead + "  \n".join(l.rstrip() for l in lines) + para[len(para.rstrip("\n")):]
    return para


EQ_NUM = r"(?:\\q?quad|\\hfill|\\[,;!]|~|\s)*(?:\\text\{\s*)?\(\s*(\d{1,3}(?:\.\d{1,3}){0,2}[a-z]?)\s*\)[ \t]*\}?"
DISPLAY_EQ = re.compile(r"(?<!\\)\$\$([\s\S]+?)(?<!\\)\$\$(\n{1,2}[ \t]*" + EQ_NUM + r"[ \t]*(?=\n|$))?")
NUM_AT_END = re.compile(EQ_NUM + r"\s*$")


NO_SRC_IMG = re.compile(r'<img(?![^>]*\bsrc=)[^>]*?\balt="([^"]*)"[^>]*>(?:\s*<br\s*/?>)?\s*')


def _html_bits(seg):
    """Marker's HTML left in tables: an <img> with no picture (an icon it did not save) is its alt text unless the same words
    follow it ("<img alt="surfel"><br/>surfel</img>" -> "surfel"); <b>/<i> are Markdown bold/italic"""
    def img(m):
        rest = seg[m.end():m.end() + len(m.group(1)) + 2]
        return "" if m.group(1) and rest.lower().startswith(m.group(1).lower()) else (m.group(1) + " " if m.group(1) else "")
    seg = NO_SRC_IMG.sub(img, seg)
    seg = re.sub(r"</img>", "", seg)
    seg = re.sub(r"<b>([^<\n]+)</b>", r"**\1**", seg)
    return re.sub(r"<i>([^<\n]+)</i>", r"*\1*", seg)


def _tag_numbers(seg):
    r"""an equation's number goes into it as \tag{7.3} (shown at the right): "(7.3)" on the line after it, or "\quad (2.11)" /
    "\quad \text{(7.7)}" at its end. A bracket that closes the formula itself ("= (x+1)") is not a number."""
    def fix(m):
        body, num = m.group(1), m.group(3)
        if num is None:
            e = NUM_AT_END.search(body)
            if not e or not re.search(r"\\q?quad|\\text\{|\\hfill|\n", e.group(0)):
                return m.group(0)
            body, num = body[:e.start()], e.group(1)
        if "\\tag" in body:
            return m.group(0) if not re.search(r"\\tag\{" + re.escape(num) + r"\}", body) else f"$${body}$$"
        end = "\n" if "\n" in body.strip() or body.startswith("\n") else ""
        return f"$${body.rstrip()} \\tag{{{num}}}{end}$$"
    return DISPLAY_EQ.sub(fix, seg)


STALE_NUMBER = re.compile(r"(?m)^([ \t>]*)\d{1,3}[.)][ \t]+(?=\d{1,3}[.)][ \t])")    # "1. 2. Incremental": the list's own number, then the page's


def _table_pipes(seg):
    r"""a "|" inside a table row's maths ($|x|$, a norm \|w\|) or code (`a || b`) splits the row into extra cells: in maths it
    becomes \vert / \Vert, in code \| (GitHub tables read that as a "|" in the cell)"""
    def math(m):
        t = m.group(1)
        t = re.sub(r"\\(left|right|big|Big|bigg|Bigg)\|", r"\\\1\\vert ", t)
        t = t.replace("\\|", "\\Vert ")
        return "$" + re.sub(r"(?<!\\)\|", r"\\vert ", t) + "$"

    def row(line):
        if not line.lstrip().startswith("|"):
            return line
        line = re.sub(r"(?<![\\$])\$(?!\$)([^$\n]+?)(?<!\\)\$", math, line)
        return re.sub(r"`[^`\n]+`", lambda m: re.sub(r"(?<!\\)\|", r"\\|", m.group(0)), line)
    return "\n".join(row(l) for l in seg.split("\n"))


def _prose(seg):
    """the clean-ups that must never touch code"""
    seg = "\n".join(re.sub(r'<math[^>]*>(.*?)</math>', lambda m: f"${m.group(1).strip()}$", l) if l.lstrip().startswith("|") else l
                    for l in seg.split("\n"))            # in a table row maths stays inline (a newline would break the table)
    seg = re.sub(r'<math[^>]*display="block"[^>]*>(.*?)</math>', lambda m: f"\n\n$$\n{m.group(1).strip()}\n$$\n\n", seg, flags=re.S)
    seg = re.sub(r"<math[^>]*>(.*?)</math>", lambda m: f"${m.group(1).strip()}$", seg, flags=re.S)
    seg = re.sub(r"^(\s*)(?:[-*+]\s+)?[•●▪◦·]\s+", r"\1- ", seg, flags=re.M)    # "- • item" / "• item" -> "- item"
    seg = re.sub(r"(?<!!)\[([^\]\n]+)\]\(#\)", r"\1", seg)                     # links with no target: plain text
    seg = STALE_NUMBER.sub(r"\1", seg)                                          # "1. 2. item" -> "2. item" (the page's number)
    seg = _table_pipes(seg)
    seg = re.sub(r"(?m)^([ \t]*)\$\$([^$\n]+?)(?<![\\$])\$[ \t]*$", r"\1$$\2$$", seg)   # "$$ x = y $": closed as display maths
    seg = _tag_numbers(_html_bits(seg))
    seg = re.sub(r"^(\s*)(?:[-*+]\s+)?(?:\[([ xX✓✔])\]|([☐☑☒✅]))\s+",          # "[X] Step 1" / "☐ item" -> GFM task list
                 lambda m: f"{m.group(1)}- [{'x' if (m.group(2) or '').strip() or (m.group(3) or '') in '☑☒✅' and m.group(3) else ' '}] ", seg, flags=re.M)
    seg = "\n\n".join(_hard_breaks(x) for x in seg.split("\n\n"))
    out, quote, in_qcode = [], [], False
    for l in seg.split("\n") + [""]:
        if l.startswith(">"):
            quote.append(l)
            continue
        if quote:                                         # a quote block: an unmatched fence (the LLM's stray ```) goes
            fences = [k for k, q in enumerate(quote) if re.match(r">[ \t]*(```|~~~)", q)]
            if len(fences) % 2:
                del quote[fences[-1]]
            for q in quote:
                if re.match(r">[ \t]*(```|~~~)", q):
                    in_qcode = not in_qcode; out.append(q)
                else:
                    out.append(q if in_qcode else escape_currency(q))   # code quoted in a figure description stays as is
            quote, in_qcode = [], False
        out.append(escape_currency(l))
    return "\n".join(out[:-1])


INDIC_SIGNS = "".join(chr(c) for c in range(0x0900, 0x0E00)
                      if "VOWEL SIGN" in unicodedata.name(chr(c), "") or "SIGN VIRAMA" in unicodedata.name(chr(c), ""))
INDIC_SPLIT = re.compile(r"(?<=[\u0900-\u0dff]) +(?=[" + INDIC_SIGNS + "])")


def _no_loops(seg):
    """the same paragraph three or more times in a row is a model loop (OCR repeated one line 14 times): kept once"""
    paras, out = seg.split("\n\n"), []
    for para in paras:
        key = para.strip()
        if len(key) >= 20 and len(out) >= 2 and out[-1].strip() == key and out[-2].strip() == key:
            continue
        out.append(para)
    res = []
    for k, para in enumerate(out):                          # the third copy was dropped above: drop the second too
        if len(para.strip()) >= 20 and res and res[-1].strip() == para.strip() and k + 1 < len(out) and out[k + 1].strip() != para.strip() \
                and seg.count(para.strip()) >= 3:
            continue
        res.append(para)
    return "\n\n".join(res)


def tidy(md):
    """valid, readable Markdown: <math> tags as $/$$, money not read as maths, blank lines around headings, tables,
    display math and code, every code fence closed, no runs of blank lines, broken paragraphs joined. Code is left as is."""
    md = re.sub(r"<\|\d{1,6}\|>", "", md)                      # the OCR model's own tokens (<|713|>), even inside code
    md = INDIC_SPLIT.sub("", md)                                # "ঢাক ার": a vowel sign never starts a word
    md = "".join(seg if k % 2 else _prose(seg) for k, seg in enumerate(CODE_BLOCK.split(md)))
    md = join_broken(md)
    md = "".join(seg if k % 2 else _no_loops(seg) for k, seg in enumerate(CODE_BLOCK.split(md)))
    out, in_code, in_math = [], False, False
    for line in md.split("\n"):
        s = line.strip()
        prev = out[-1].strip() if out else ""
        if FENCE.match(line):
            if not in_code and prev:
                out.append("")
            out.append(line); in_code = not in_code
            if not in_code:
                out.append("")
            continue
        if in_code:
            out.append(line); continue
        if s == "$$":
            if not in_math and prev:
                out.append("")
            out.append("$$"); in_math = not in_math
            if not in_math:
                out.append("")
            continue
        if in_math:
            out.append(line); continue
        if not s and (not out or not out[-1].strip()):
            continue                                        # one blank line at most (outside code)
        if re.match(r"#{1,6}\s", s) and prev:
            out.append("")
        elif s.startswith("|") and prev and not prev.startswith("|"):
            out.append("")
        elif prev.startswith("|") and s and not s.startswith("|"):
            out.append("")
        elif re.match(r"#{1,6}\s", prev) and s:
            out.append("")
        out.append(line.rstrip() + ("  " if line.endswith("  ") and line.strip() else ""))   # a hard line break stays
    if in_code:
        out.append("```")
    if in_math:
        out.append("$$")
    return "\n".join(out).strip("\n") + "\n"


def drop_unused_anchors(md):
    """<span id="page-15-0"></span> marks that no link points to (Marker writes one per heading and page): only clutter.
    For a whole document only: a link in one chunk may point into another"""
    linked = set(re.findall(r"\]\(#([^)\s]+)\)", md))
    return re.sub(r'<span id="([^"]+)">\s*</span>', lambda m: m.group(0) if m.group(1) in linked else "", md)


def shrink_images(img_dir, max_px=IMAGE_MAX_PX):
    """scales down images whose long side is above max_px (same file, same format); returns how many"""
    n = 0
    for f in Path(img_dir).iterdir():
        if f.suffix.lower() not in (".jpg", ".jpeg", ".png", ".webp"):
            continue
        try:
            with Image.open(f) as im:
                if max(im.size) <= max_px:
                    continue
                im.load(); small = im.copy()
            small.thumbnail((max_px, max_px), Image.LANCZOS)
            if f.suffix.lower() in (".jpg", ".jpeg"):
                small.convert("RGB").save(f, "JPEG", quality=92)
            else:
                small.save(f)
            n += 1
        except Exception:
            continue
    return n

In [ ]:
%%writefile mdbox/lb.py
"""Tiny least-busy load balancer in front of the llama-servers (Surya VLM or the vision LLM), one per GPU.
Marker and the pipeline talk to ONE OpenAI-compatible URL; this fans their concurrent requests out over both GPUs.

Image affinity: a request carrying an image the same server saw before goes back to that server (unless it is clearly
busier), because llama-server reuses an image it already processed: measured on a T4, a page image is 1484 tokens and
2.8 s of prompt processing the first time, 0.06 s when reused. With plain least-busy routing, the several calls a page
gets (review, audit sections, second looks, a diagram's graph answers) were spread over both GPUs: 2 of 48 reused.
Usage: python lb.py PORT BACKEND_PORT [BACKEND_PORT ...]"""
import hashlib, os, sys, time
from collections import OrderedDict

import httpx
import uvicorn
from fastapi import FastAPI, Request, Response

PORT, BACKENDS = int(sys.argv[1]), [f"http://127.0.0.1:{p}" for p in sys.argv[2:]]
HOP = {"host", "content-length", "connection", "keep-alive", "transfer-encoding", "te", "trailer", "upgrade", "proxy-authorization", "proxy-authenticate"}
SPILL = int(os.environ.get("LB_SPILL", "3"))          # a preferred backend this many requests busier than the least busy is passed over (6 left one GPU 15-20 points idler)
AFFINITY_MAX = 4096                                    # images remembered (most recent)
inflight = {b: 0 for b in BACKENDS}
down_until = {b: 0.0 for b in BACKENDS}
affinity = OrderedDict()                               # image digest -> backend that has it
stats = {"with_image": 0, "affinity_kept": 0, "spilled": 0}
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
client = httpx.AsyncClient(timeout=httpx.Timeout(900.0, connect=10.0), limits=httpx.Limits(max_connections=256, max_keepalive_connections=64))


def image_key(body):
    """digest of the first base64 image in a request body, or None"""
    i = body.find(b";base64,")
    if i < 0:
        return None
    j = body.find(b'"', i)
    return hashlib.blake2b(body[i + 8:j if j > 0 else len(body)], digest_size=12).digest()


def order(key=None):
    now = time.time()
    up = [b for b in BACKENDS if down_until[b] <= now] or list(BACKENDS)       # if everything looks down, still try
    spread = key[0] if key is not None else int(time.time() * 1000)     # equal load: a new image's digest picks, not list order
    ranked = sorted(up, key=lambda b: (inflight[b], (BACKENDS.index(b) + spread) % len(BACKENDS)))
    pref = affinity.get(key) if key is not None else None
    if pref in up:
        if inflight[pref] - inflight[ranked[0]] < SPILL:
            ranked.remove(pref); ranked.insert(0, pref)
            stats["affinity_kept"] += 1
        else:
            stats["spilled"] += 1
    return ranked


def remember(key, backend):
    if key is None:
        return
    affinity[key] = backend
    affinity.move_to_end(key)
    while len(affinity) > AFFINITY_MAX:
        affinity.popitem(last=False)


@app.get("/lb-status")
async def status():
    return {"inflight": inflight, "down": {b: max(0, round(t - time.time())) for b, t in down_until.items()}, "affinity": stats}


@app.api_route("/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS", "HEAD"])
async def proxy(path: str, request: Request):
    body = await request.body()
    headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP}
    key = image_key(body) if request.method == "POST" and body else None
    if key is not None:
        stats["with_image"] += 1
    last = None
    for b in order(key):
        remember(key, b)                               # at once: the page's next calls follow, even while this one runs
        inflight[b] += 1
        try:
            r = await client.request(request.method, f"{b}/{path}", params=request.query_params, content=body, headers=headers)
        except (httpx.ConnectError, httpx.ReadError, httpx.RemoteProtocolError, httpx.ConnectTimeout) as e:
            down_until[b] = time.time() + 10; last = e
            continue
        finally:
            inflight[b] -= 1
        if r.status_code == 503 and path.rstrip("/") == "health":     # a backend that is still loading: try the other one
            last = r; continue
        out = {k: v for k, v in r.headers.items() if k.lower() not in HOP and k.lower() != "content-encoding"}
        return Response(content=r.content, status_code=r.status_code, headers=out)
    if isinstance(last, httpx.Response):
        return Response(content=last.content, status_code=last.status_code)
    return Response(content=f"no backend reachable: {last!r}".encode(), status_code=502)


if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

In [ ]:
%%writefile mdbox/worker.py
"""Conversion worker. One process per slot, kept alive between jobs so Marker's models stay loaded.
It claims document tasks from pool/tasks/ (atomic claim files, so any number of workers can share the queue) and
runs Marker's PdfConverter on them (after LibreOffice for office formats, see convert.py). The Surya VLM lives in the llama-server processes behind the load balancer;
the worker only holds the small CPU models (Indra ONNX), so a crash here never takes the GPUs down.
Usage: python worker.py pool_dir worker_id vlm_url vlm_parallel cpu_threads"""
import ctypes, gc, json, os, shutil, sys, tempfile, time, traceback
from pathlib import Path

import convert as C
import figures as FIG
import review as R

POOL, WID, VLM_URL, PARALLEL = Path(sys.argv[1]), sys.argv[2], sys.argv[3], sys.argv[4]
CPU_THREADS = int(sys.argv[5]) if len(sys.argv) > 5 else 2
PARENT = os.getppid()
try:                                    # if RAM ever fills anyway, the kernel kills a worker (whose piece is requeued),
    with open("/proc/self/oom_score_adj", "w") as f:      # not the notebook kernel, a GPU server or the API
        f.write("1000")
except OSError:
    pass
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["SURYA_INFERENCE_BACKEND"] = "llamacpp"
if VLM_URL:
    os.environ["SURYA_INFERENCE_URL"] = VLM_URL            # attach to the running servers; never spawn one from a worker
    os.environ["SURYA_INFERENCE_PARALLEL"] = PARALLEL      # concurrent VLM requests from this worker


def log(*a):
    print(f"[{WID} {time.strftime('%H:%M:%S')}]", *a, flush=True)


def jwrite(path, obj):
    tmp = Path(str(path) + f".{WID}.tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False)); os.replace(tmp, path)


def claim(path):
    try:
        fd = os.open(str(path), os.O_CREAT | os.O_EXCL | os.O_WRONLY); os.write(fd, WID.encode()); os.close(fd); return True
    except FileExistsError:
        return False


def shutting_down():
    return (POOL / "shutdown").exists() or os.getppid() != PARENT


RECYCLE_BYTES = float(os.environ.get("MD_WORKER_RECYCLE_GB") or 3) * 2 ** 30
RECYCLED = 75                                   # exit code pipeline.Pool relaunches without counting it as a crash


def own_rss():
    with open("/proc/self/status") as f:
        return next((int(l.split()[1]) * 1024 for l in f if l.startswith("VmRSS:")), 0)


def give_memory_back():
    """Python and onnxruntime free memory but glibc keeps it; malloc_trim returns it to the system"""
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


from marker.config.parser import ConfigParser
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import save_output

import marker.services.openai as _marker_openai


class _LatexJson:
    """Marker parses the LLM's JSON with json.loads: LaTeX in it loses its backslashes (\\b, \\r, \\n, \\t, \\f are escapes)"""
    loads = staticmethod(R.latex_json_loads)

    def __getattr__(self, name):
        return getattr(json, name)


_marker_openai.json = _LatexJson()
LLM_REJECTED = {}                     # per piece: Marker LLM rewrites dropped, by reason
_marker_call = _marker_openai.OpenAIService.__call__


def _checked_call(self, prompt, image, block, response_schema, *a, **k):
    """every rewrite Marker's LLM steps make is checked like the page review's: a lossy one is dropped (Marker keeps its text)"""
    out = _marker_call(self, prompt, image, block, response_schema, *a, **k)
    ok, why = R.check_marker_llm(prompt, out)
    if not ok:
        LLM_REJECTED[why] = LLM_REJECTED.get(why, 0) + 1
        return {}
    return out


_marker_openai.OpenAIService.__call__ = _checked_call

t0 = time.time()
MODELS = create_model_dict()          # Indra ONNX (CPU) + a lazy client for the VLM server
MODELS["indra_model"].threads = CPU_THREADS      # onnxruntime threads: the pool's workers share the machine's cores
log(f"models ready in {time.time() - t0:.1f}s | VLM: {VLM_URL or 'none (fast / no_ocr only)'} | LibreOffice: {C.SOFFICE or 'none'}")


def cli_options(o, out_dir):
    """job options -> the same keys Marker's CLI hands to ConfigParser"""
    k = {"output_format": o.get("OUTPUT_FORMAT", "markdown"), "output_dir": str(out_dir), "mode": o["MODE"],
         "disable_multiprocessing": True,                                  # pdftext single-process: the pool already runs workers in parallel
         "disable_image_extraction": not o.get("IMAGES", True)}
    if k["output_format"] == "markdown":
        k["paginate_output"] = True           # page separators let convert.place_orphan_images put unlinked figures on their page
    for opt, key in (("FORCE_OCR", "force_ocr"), ("STRIP_EXISTING_OCR", "strip_existing_ocr"),
                     ("PAGINATE", "paginate_output"), ("KEEP_HEADERS_FOOTERS", "keep_headers_footers")):
        if o.get(opt):
            k[key] = True
    if o.get("USE_LLM") and R.quality_option(o, "REDO_INLINE_MATH"):      # accurate (or asked for): inline maths rewritten too
        k["redo_inline_math"] = True
    if o.get("PAGE_RANGE"):
        k["page_range"] = o["PAGE_RANGE"]
    if o.get("USE_LLM") and R.quality_option(o, "MARKER_LLM"):      # fast quality: Marker's own LLM steps are left out
        k["use_llm"] = True
        if o.get("LLM_URL"):                      # the box's own vision LLM (OpenAI-compatible llama-server), nothing leaves Kaggle
            k.update(llm_service="marker.services.openai.OpenAIService", openai_base_url=o["LLM_URL"], openai_model=o.get("LLM_MODEL") or "local",
                     openai_api_key="local", timeout=900, max_retries=1,
                     max_concurrency=int(o.get("MARKER_LLM_CONCURRENCY") or 2 * int(o.get("LLM_CONCURRENCY") or 2)))
        else:
            k["gemini_api_key"] = os.environ.get("GOOGLE_API_KEY", "")
    return k


def short_document(src, max_pages=2):
    """a picture or a PDF of at most 2 pages (the whole file: chunks and batches are page ranges of it)"""
    s = str(src).lower()
    if s.endswith(C.MARKER_EXT) and not s.endswith((".pdf", ".html", ".htm", ".epub")):
        return True                                       # an image: one page
    if not s.endswith(".pdf"):
        return False
    try:
        import pypdfium2 as pdfium
        pdf = pdfium.PdfDocument(str(src))
        try:
            return len(pdf) <= max_pages
        finally:
            pdf.close()
    except Exception:
        return False


def convert(t):
    out = Path(t["out"]); out.mkdir(parents=True, exist_ok=True)
    action, fmt = C.route(t["src"], t["opts"].get("OFFICE", "pdf"))
    if action == "text":
        return C.text_to_md(t["src"], out, t["stem"])
    src, scratch, via = t["src"], None, "marker"
    try:
        if action in ("soffice", "ddjvu"):
            scratch = tempfile.mkdtemp(prefix=f"conv_{WID}_", dir=str(POOL))
            src = str(C.soffice(src, fmt, scratch, POOL / f"lo_profile_{WID}") if action == "soffice" else C.djvu_to_pdf(src, scratch))
            via = f"{'libreoffice' if action == 'soffice' else 'ddjvu'}->{fmt}+marker"
        clock, last = {}, [time.time()]

        def lap(stage):                                   # seconds per stage, for the report (where does the time go?)
            now = time.time(); clock[stage] = round(clock.get(stage, 0) + now - last[0], 1); last[0] = now
        if scratch:
            lap("render_to_pdf")
        LLM_REJECTED.clear()
        legacy = C.ocr_reason(src, C.page_list(t["opts"].get("PAGE_RANGE"))) \
            if str(src).lower().endswith(".pdf") and not t["opts"].get("FORCE_OCR") else None
        if legacy:                                    # its text layer is not real text: OCR the pages, and don't trust that layer
            t = dict(t, opts=dict(t["opts"], FORCE_OCR=True))
            log(f"{legacy}: pages read with OCR")
        if not t["opts"].get("KEEP_HEADERS_FOOTERS") and short_document(src):
            t = dict(t, opts=dict(t["opts"], KEEP_HEADERS_FOOTERS=True))    # nothing repeats page to page: a header is content
        cp = ConfigParser(cli_options(t["opts"], out))
        conv = PdfConverter(config=cp.generate_config_dict(), artifact_dict=MODELS, processor_list=cp.get_processors(),
                            renderer=cp.get_renderer(), llm_service=cp.get_llm_service())
        document = conv.build_document(src)                  # = conv(src), keeping the document for the figure re-crop
        conv.page_count = len(document.pages)
        save_output(conv.resolve_dependencies(conv.renderer)(document), str(out), t["stem"])
        lap("marker")
        extra, md = {"stage_seconds": clock}, out / f"{t['stem']}.md"
        if md.is_file() and t["opts"].get("OUTPUT_FORMAT", "markdown") == "markdown":
            if t["opts"].get("IMAGES", True) and t["opts"].get("FIGURE_FIX", True):
                try:
                    extra["figures"] = FIG.recrop_document(document, src, out, t["stem"])
                except Exception as e:                        # keep Marker's own crops rather than fail the document
                    extra["figures"] = {"error": f"{type(e).__name__}: {e}"[:300]}
                    log("figure re-crop failed, kept Marker's crops:", extra["figures"]["error"])
            anchors = extra.get("figures", {}).pop("anchors", {}) if isinstance(extra.get("figures"), dict) else {}
            text_heavy = extra.get("figures", {}).pop("text_heavy", {}) if isinstance(extra.get("figures"), dict) else {}
            extra["linked_figures"] = C.place_orphan_images(md, out, keep_pages=True, anchors=anchors)   # separators: per-page review
            if t["opts"].get("IMAGES", True):
                try:                                  # a formula drawn with shapes (polygons) LaTeX can't write: a picture of it
                    why = {}
                    n = FIG.picture_formulas(md, document, src, out, skipped=why)
                    if n:
                        extra["drawn_formulas"] = n
                    if why:
                        extra["drawn_formulas_skipped"] = why
                except Exception as e:
                    log("drawn formulas skipped:", repr(e)[:200])
                extra["shrunk_images"] = R.shrink_images(out, int(t["opts"].get("IMAGE_MAX_PX") or R.IMAGE_MAX_PX))
            lap("figures")
            o, llm = t["opts"], t["opts"].get("USE_LLM") and t["opts"].get("LLM_URL")
            q = lambda key: R.quality_option(o, key)
            conc, model = int(o.get("LLM_CONCURRENCY") or 2), o.get("LLM_MODEL") or "local"
            R.set_llm_inflight(int(o.get("LLM_INFLIGHT") or 0))  # one worker may fill every LLM slot, never the queue
            extra["quality"] = str(o.get("QUALITY") or "balanced")

            def describe():
                try:                                  # the Markdown is written after the page review (they run at the same time)
                    return FIG.describe_figures(md, out, o["LLM_URL"], model, conc, text_heavy=text_heavy, tighten=bool(q("CORE_CROP")),
                                                graph_tries=int(q("GRAPH_TRIES")), graph_checks=int(q("GRAPH_CHECKS")), defer=True,
                                                transcribe_text=bool(q("TRANSCRIBE_TEXT_IMAGES")))
                except Exception as e:
                    log("figure descriptions failed:", repr(e)[:300])
                    return 0, None

            audit_stats, sizes = {}, tuple(q("AUDIT_SECTIONS") or (R.AUDIT_SECTION_PARAS,))
            audit = (lambda text, jpeg, layer: R.audit_page(text, jpeg, layer, o["LLM_URL"], model, audit_stats, sizes)) \
                if llm and q("AUDIT") else None

            def review():
                if not q("PAGE_REVIEW"):              # fast: no rewrite, only the audit (still side by side with the figures)
                    try:
                        audit_stats.update(R.audit_pages(md, src, o["LLM_URL"], model, conc, use_layer=not legacy, sizes=sizes))
                    except Exception as e:
                        log("audit skipped:", repr(e)[:300])
                    return None
                try:                                  # every page checked against its image and rewritten (only if nothing is lost),
                    return R.review_pages(md, src, o["LLM_URL"], model, conc, int(o.get("LLM_CTX") or 8192),   # then audited
                                          keep_headers=bool(o.get("KEEP_HEADERS_FOOTERS")), use_layer=not legacy,
                                          part_tokens=int(q("REVIEW_PART_TOKENS")), after=audit,
                                          skip_settled=bool(q("REVIEW_SKIP_SETTLED")))
                except Exception as e:
                    log("page review failed, kept Marker's pages:", repr(e)[:300])
            from concurrent.futures import ThreadPoolExecutor
            with ThreadPoolExecutor(2) as ex:         # both only wait for the LLM: side by side, the LLM's slots stay full
                fd = ex.submit(describe) if llm and q("DESCRIBE_FIGURES") else None
                fr = ex.submit(review) if llm and (q("PAGE_REVIEW") or audit) else None
                if fr is not None and fr.result():
                    extra["page_review"] = fr.result()
                if fd is not None:
                    n_desc, put = fd.result()
                    if put:
                        put(md)
                    extra["described_figures"] = n_desc
            lap("describe_review_audit")
            if audit:
                extra["audit"] = R._public(audit_stats)
            if str(src).lower().endswith(".pdf") and not legacy:
                try:                                  # a few characters an OCR got wrong where the PDF's own text has the line
                    n = C.repair_from_layer(md, src)
                    if n:
                        extra["layer_fixes"] = n
                except Exception as e:
                    log("text-layer repair skipped:", repr(e)[:200])
            if str(src).lower().endswith(".pdf"):
                try:                                  # Indic vowel signs in visual order: fixed from the PDF's own text
                    n_fixed = C.repair_indic(md, src)
                    if n_fixed:
                        extra["indic_words_fixed"] = n_fixed
                except Exception as e:
                    log("Indic repair skipped:", repr(e)[:200])
            try:                                      # tables against the page image: columns, pictures in cells
                for key, n in FIG.fix_tables(md, document, src, out, t["stem"]).items():
                    if n:
                        extra[key] = n
            except Exception as e:
                extra["tables_error"] = repr(e)[:200]
                log("table check skipped:", repr(e)[:200])
            try:                                      # heading levels from the PDF's outline (bookmarks), when it has one
                extra["outline_headings"] = R.apply_outline(md, R.outline(src))
            except Exception as e:
                log("outline headings skipped:", repr(e)[:200])
            R.finish_markdown(md, keep_pages=bool(t["opts"].get("PAGINATE")))
        if legacy:
            extra["forced_ocr"] = f"{legacy}: read with OCR"
        if LLM_REJECTED:
            extra["marker_llm_rejected"] = dict(LLM_REJECTED)
        if md.is_file() and str(src).lower().endswith(".pdf") and not legacy:
            try:
                extra["coverage"] = FIG.text_coverage(src, md.read_text(encoding="utf-8"), [p.page_id for p in document.pages])
            except Exception as e:
                log("coverage check failed:", repr(e)[:300])
        return {"pages": conv.page_count, "files": sorted(p.name for p in out.iterdir()), "via": via, **extra}
    finally:
        if scratch:
            shutil.rmtree(scratch, ignore_errors=True)


def run_task(t):
    tid = t["tid"]
    t1 = time.time()
    jwrite(POOL / "running" / f"{tid}.json", {"wid": WID, "t0": t1, "name": t["rel"]})
    try:
        if (POOL / "cancel" / t["job"]).exists():
            res = {"ok": False, "error": "cancelled"}
        else:
            res = {"ok": True, **convert(t)}
    except BaseException as e:
        if isinstance(e, (KeyboardInterrupt, SystemExit)):
            raise
        res = {"ok": False, "error": f"{type(e).__name__}: {e}"[:600], "trace": traceback.format_exc()[-1500:]}
        log("task", tid, t["rel"], "FAILED:", res["error"])
    res.update(tid=tid, job=t["job"], doc=t["doc"], rel=t["rel"], worker=WID, sec=round(time.time() - t1, 2))
    jwrite(POOL / "results" / f"{tid}.json", res)
    give_memory_back()
    log("task", tid, t["rel"], "ok" if res["ok"] else "failed", f"{res['sec']}s", f"{res.get('pages')} pages", f"{own_rss() / 2 ** 30:.1f} GB")
    if own_rss() > RECYCLE_BYTES:
        log(f"memory {own_rss() / 2 ** 30:.1f} GB after this piece: restarting to give it back")
        sys.exit(RECYCLED)


def anything_running():
    """a piece some worker is converting right now (running file without a result)"""
    return any(not (POOL / "results" / rp.name).exists() for rp in (POOL / "running").glob("*.json"))


def take_solo():
    """RAM is low but nothing runs: one worker (the one that gets this token) may still convert, so the box never deadlocks"""
    try:
        fd = os.open(str(POOL / "solo"), os.O_CREAT | os.O_EXCL | os.O_WRONLY); os.write(fd, str(os.getpid()).encode()); os.close(fd)
        return True
    except FileExistsError:
        return False


def free_task_waiting():
    """a piece nobody has claimed or finished (task files stay after their result is written)"""
    for tp in (POOL / "tasks").glob("*.json"):
        if not (POOL / "results" / tp.name).exists() and not (POOL / "tasks" / f"{tp.stem}.claim").exists():
            return True
    return False


SLOT = None                                      # the start slot this worker holds (handed back if no piece was taken)


def may_start():
    """a free start slot (pipeline.start_budget: free RAM above the pause level / what a starting piece grows by). Slots are
    files made exclusively, so two workers can't take the same one; the pool frees each after START_WINDOW_S. True when no
    budget is published (old pool, tests)."""
    try:
        slots = int(json.loads((POOL / "admit.json").read_text())["slots"])
    except Exception:
        return True
    (POOL / "starts").mkdir(exist_ok=True)
    for k in range(slots):
        try:
            fd = os.open(str(POOL / "starts" / f"slot_{k}"), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, str(os.getpid()).encode()); os.close(fd)
            global SLOT
            SLOT = POOL / "starts" / f"slot_{k}"
            return True
        except FileExistsError:
            continue
    return False


def next_task():
    """claims the oldest free piece; None if there is none"""
    for tp in sorted((POOL / "tasks").glob("*.json")):
        if shutting_down():
            return None
        tid = tp.stem
        if (POOL / "results" / f"{tid}.json").exists() or not claim(POOL / "tasks" / f"{tid}.claim"):
            continue
        return json.loads(tp.read_text())
    return None


BASE_RSS = own_rss()                             # this worker with its models loaded and nothing else
jwrite(POOL / f"alive_{WID}.json", {"pid": os.getpid(), "vlm": bool(VLM_URL)})
idle_since = time.time()
while not shutting_down():
    solo = False
    if (POOL / "pause").exists():                # RAM is low (pipeline.guard_memory)
        if anything_running() or not take_solo():                  # someone is converting: wait for it to finish and free memory
            if time.time() - idle_since > 20 and own_rss() > max(1.5 * BASE_RSS, BASE_RSS + 0.7 * 2 ** 30):
                give_memory_back()               # an idle worker that grew gives it back, or restarts to (models reload on start)
                if own_rss() > max(1.5 * BASE_RSS, BASE_RSS + 0.7 * 2 ** 30):
                    log(f"RAM low and idle at {own_rss() / 2 ** 30:.1f} GB: restarting to give it back")
                    sys.exit(RECYCLED)
            time.sleep(1)
            continue
        solo = True                              # nothing runs: this worker may convert one piece
    elif not free_task_waiting():
        time.sleep(0.4)                          # nothing to take: no start slot used up
        continue
    elif anything_running() and not may_start():
        time.sleep(1)                            # RAM for one more piece is not there yet: wait (never when nothing runs)
        continue
    try:
        t = next_task()
        if not t and SLOT is not None:           # another worker took it: the slot goes back at once
            SLOT.unlink(missing_ok=True)
        SLOT = None
        if t:
            run_task(t)
            idle_since = time.time()
    finally:
        if solo:
            (POOL / "solo").unlink(missing_ok=True)
    if not t:
        time.sleep(0.4)
log("shutdown")

In [ ]:
%%writefile mdbox/pipeline.py
"""Document -> Markdown pipeline (called by server.py, one job per runner thread).
A job is one uploaded document, or a .zip of documents. Every document becomes one task in a shared queue - a long PDF
becomes several page-range tasks - and the warm worker processes (worker.py) pull tasks from it, so documents and pages
of the same job and of different jobs run in parallel while the VLM requests of all of them are spread over both GPUs.
Page-range tasks can also be "stolen" by the client's own local workers (steal()), which then fill them in on the PC.
The Pool class is also used by md_client.py to run local workers on the PC."""
import json, math, os, re, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path

import convert
import review as R

HERE = Path(__file__).resolve().parent
WORKER_PY = Path(os.environ.get("MD_WORKER_PY") or HERE / "worker.py")
TMP_ROOT = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else Path("/tmp")

DOC_EXT = convert.ALL_EXT
# WORKERS 0 = from the vCPUs and free RAM (WORKERS_PER_CPU, MEM_PER_WORKER_GB); VLM_PARALLEL 0 = from VLM_SLOTS
DEFAULTS = dict(WORKERS=0, CPU_THREADS=0, VLM_URL="", VLM_PARALLEL=0, VLM_SLOTS=0, MODE="auto", OFFICE="pdf", DOC_TIMEOUT_MIN=90,
                LLM_URL="", LLM_MODEL="", LLM_SLOTS=0, USE_LLM=False, REDO_INLINE_MATH=None, DESCRIBE_FIGURES=None, PAGE_REVIEW=None, LLM_CTX=8192,
                QUALITY="balanced",                           # fast / balanced / accurate (review.QUALITY_PROFILES); None above = as the quality sets it
                TIME_BUDGET_H=7.0, MAX_ZIP_FILES=2000, MAX_UNZIPPED_GB=6.0,
                SPLIT_MIN_PAGES=8, CHUNK_MIN_PAGES=4, CHUNK_MAX_PAGES=24,
                MEM_PER_WORKER_GB=3.0, MEM_RESERVE_GB=3.0,     # RAM a Marker worker may need (measured: ~2 GB at its peak in the suite runs) / keep free for the API, tunnel, page cache
                MEM_PAUSE_FREE_GB=4.0, MEM_RESUME_FREE_GB=6.0, MEM_KILL_FREE_GB=2.0,   # free RAM: stop new pieces / start again / stop a worker
                MEM_GPU_SERVER_MAX_GB=5.0,                    # a llama-server bigger than this (healthy: ~2.3 + its prompt cache) is restarted when RAM runs out
                WORKER_RECYCLE_GB=3.0,                        # a worker bigger than this after a piece restarts itself (frees leaks)
                WORKERS_PER_CPU=1.5,                          # workers per vCPU at most (RAM permitting): a worker waiting on the GPUs leaves its CPU free
                MEM_START_GB=2.5, START_WINDOW_S=60)          # RAM a piece grows by in its first minute; how long a start counts against the free RAM
DEFAULTS.update(json.loads(os.environ.get("MD_DEFAULTS") or "{}"))      # set from the notebook's JOB_DEFAULTS
MODES = ("auto", "no_ocr", "fast", "balanced", "vllm")


class Cancelled(Exception):
    pass


def jwrite(path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False)); os.replace(tmp, path)


def safe_part(s, limit=80):
    """keeps non-ASCII names (Bangla, Hindi, ...) but drops what Windows/zip cannot hold"""
    s = re.sub(r'[\x00-\x1f<>:"/\\|?*]+', "_", s).strip(" .")
    return s[:limit].strip(" .") or "doc"


# ----------------------------------------------------------------- memory (the Kaggle session dies when its RAM is full)
def _read(path):
    try:
        with open(path) as f:
            return f.read().strip()
    except OSError:
        return None


def memory_detail():
    """bytes: limit, used = what the kernel cannot free (process memory + tmpfs/shared memory), cache = file cache it drops first.
    File cache (e.g. model files the GPU servers just read) is NOT counted as used: counting it paused work for nothing."""
    for lim_f, cur_f, stat_f, file_k, shm_k in (
            ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.stat", "file", "shmem"),
            ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes",
             "/sys/fs/cgroup/memory/memory.stat", "total_cache", "total_shmem")):
        lim, cur = _read(lim_f), _read(cur_f)
        if lim and cur and lim != "max" and int(lim) < 1 << 50:
            stat = dict(l.split() for l in (_read(stat_f) or "").splitlines() if len(l.split()) == 2)
            file, shm = int(stat.get(file_k, 0)), int(stat.get(shm_k, 0))
            return {"limit": int(lim), "used": max(0, int(cur) - file + shm), "cache": max(0, file - shm)}
    info = dict((l.split(":")[0], int(l.split()[1]) * 1024) for l in (_read("/proc/meminfo") or "").splitlines() if l.split()[1:2])
    total = info.get("MemTotal", 1)
    return {"limit": total, "used": total - info.get("MemAvailable", total), "cache": 0}


def memory():
    """(limit, used) in bytes, see memory_detail"""
    d = memory_detail()
    return d["limit"], d["used"]


def _scan():
    """[(bytes, pid, name)] of every process"""
    out = []
    for pid in os.listdir("/proc"):
        if not pid.isdigit():
            continue
        st = _read(f"/proc/{pid}/status") or ""
        kb = next((int(l.split()[1]) for l in st.splitlines() if l.startswith("VmRSS:")), 0)
        if kb:
            cmd = (_read(f"/proc/{pid}/cmdline") or "").replace("\x00", " ").split()
            name = " ".join(Path(c).name for c in cmd[:3])[:60] or next((l.split()[1] for l in st.splitlines() if l.startswith("Name:")), pid)
            out.append((kb * 1024, int(pid), name))
    return out


def processes(n=8):
    """the n biggest processes by resident memory: [(MB, name)], for /health (where does the RAM go?)"""
    return [(b // 2 ** 20, name) for b, _, name in sorted(_scan(), reverse=True)[:n]]


def gpu_servers():
    """[(bytes, pid, name)] of the llama-server processes (the notebook's watchdog restarts one that dies)"""
    return [p for p in _scan() if p[2].startswith("llama-server")]


def rss(pid):
    """resident memory of a process and its children (e.g. LibreOffice under a worker), bytes"""
    total, todo = 0, [str(pid)]
    while todo:
        p = todo.pop()
        st = _read(f"/proc/{p}/status") or ""
        total += next((int(l.split()[1]) * 1024 for l in st.splitlines() if l.startswith("VmRSS:")), 0)
        todo += (_read(f"/proc/{p}/task/{p}/children") or "").split()
    return total


# ----------------------------------------------------------------- the warm worker pool
class Pool:
    def __init__(self, root=None):
        self.dir = Path(root or os.environ.get("MD_POOL") or TMP_ROOT / "md_pool")
        self.procs, self.seq, self.restarts, self.broken = {}, 0, 0, False
        self.requeued, self.mem_events = {}, []           # tid -> times put back after a memory kill; notes for the job report
        self.vlm_url, self.parallel, self.threads = "", 8, 2
        self.mtx, self.seq_lock = threading.Lock(), threading.Lock()
        self.planned, self.last_kill = set(), 0.0           # workers stopped or restarting on purpose

    @property
    def vlm_ok(self):
        return bool(self.vlm_url)

    def start(self, n, vlm_url, parallel, threads):
        shutil.rmtree(self.dir, ignore_errors=True)
        for d in ("tasks", "results", "running", "logs", "cancel"):
            (self.dir / d).mkdir(parents=True, exist_ok=True)
        self.vlm_url, self.parallel, self.threads, self.restarts = vlm_url or "", parallel, threads, 3 * n + 3
        for k in range(n):
            self.launch(f"w{k}")

    def launch(self, wid):
        cmd = [sys.executable, str(WORKER_PY), str(self.dir), wid, self.vlm_url, str(self.parallel), str(self.threads)]
        self.procs[wid] = subprocess.Popen(cmd, stdout=open(self.dir / "logs" / f"{wid}.log", "ab"), stderr=subprocess.STDOUT,
                                           env=dict(os.environ, TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS=str(self.threads),
                                                    MD_WORKER_RECYCLE_GB=str(DEFAULTS["WORKER_RECYCLE_GB"])))

    def stop(self):
        (self.dir / "shutdown").write_text("1")
        for p in self.procs.values():
            if p.poll() is None:
                p.terminate()

    def enqueue(self, task):
        with self.seq_lock:
            self.seq += 1
            tid = f"{self.seq:08d}"
        task["tid"] = tid
        jwrite(self.dir / "tasks" / f"{tid}.json", task)
        return tid

    def result(self, tid):
        p = self.dir / "results" / f"{tid}.json"
        try:
            return json.loads(p.read_text())
        except (FileNotFoundError, json.JSONDecodeError):
            return None

    def waiting(self):
        """tasks no worker (or client) has claimed yet"""
        claimed = {p.stem for p in (self.dir / "tasks").glob("*.claim")}
        return sum(1 for p in (self.dir / "tasks").glob("*.json") if p.stem not in claimed and not (self.dir / "results" / p.name).exists())

    def cancel_job(self, job_id):
        (self.dir / "cancel" / job_id).write_text("1")

    def _fail(self, tid, task, why, wid=None):
        """record a failure for a task unless the worker managed to write its own result first"""
        res = {"ok": False, "error": why, "tid": tid, "job": task["job"], "doc": task["doc"], "rel": task["rel"], "worker": wid, "sec": None}
        tmp = self.dir / "results" / f"{tid}.fail.tmp"
        tmp.write_text(json.dumps(res))
        try:
            os.link(tmp, self.dir / "results" / f"{tid}.json")
        except FileExistsError:
            pass
        tmp.unlink(missing_ok=True)

    def running(self):
        out = {}
        for rp in (self.dir / "running").glob("*.json"):
            if (self.dir / "results" / rp.name).exists():
                continue
            try:
                out[rp.stem] = json.loads(rp.read_text())
            except Exception:
                pass
        return out

    RECYCLED = 75          # exit code of a worker that restarts itself on purpose (worker.py)

    def requeue(self, tid, wid, why, count=True):
        """puts a piece that lost its worker to memory pressure back in the queue (twice at most when its own size was the
        problem, i.e. count=True), else fails it"""
        task = json.loads((self.dir / "tasks" / f"{tid}.json").read_text())
        n = self.requeued[tid] = self.requeued.get(tid, 0) + (1 if count else 0)
        if n > 2:
            self._fail(tid, task, f"{why} (3 times): this piece needs more memory than the box has; try fewer pages per job", wid)
            return
        (self.dir / "running" / f"{tid}.json").unlink(missing_ok=True)
        (self.dir / "tasks" / f"{tid}.claim").unlink(missing_ok=True)
        self.mem_events.append(f"{task['rel']}: {why}; converted again")

    def guard_memory(self, d=None):
        """little free RAM: no new pieces (but see worker.py: with nothing running, one worker may still start, so it can never
        deadlock). Very little: stop the biggest busy worker and requeue its piece. Returns free GB."""
        d = d or DEFAULTS
        if not (self.dir / "tasks").is_dir():                # pool not started (yet)
            return None
        lim, used = memory()
        free_gb, pause = (lim - used) / 2 ** 30, self.dir / "pause"
        try:                                                 # the start budget the workers read (worker.py: may_start)
            starts = self.dir / "starts"
            starts.mkdir(exist_ok=True)
            for f in starts.glob("slot_*"):                  # a start counts for START_WINDOW_S, then the free RAM shows it
                if time.time() - f.stat().st_mtime > float(d["START_WINDOW_S"]):
                    f.unlink(missing_ok=True)
            jwrite(self.dir / "admit.json", {"slots": start_budget(free_gb, float(d["MEM_PAUSE_FREE_GB"]), float(d["MEM_START_GB"])),
                                             "free_gb": round(free_gb, 2)})
        except OSError:
            pass
        if free_gb < float(d["MEM_PAUSE_FREE_GB"]) and not pause.exists():
            pause.write_text(f"{free_gb:.1f}")
        elif free_gb > float(d["MEM_RESUME_FREE_GB"]) and pause.exists():
            pause.unlink(missing_ok=True)
        solo = self.dir / "solo"                             # the one worker allowed while paused; drop it if that worker died
        if solo.exists():
            pid = (_read(str(solo)) or "0").strip()
            if not pid.isdigit() or not Path(f"/proc/{pid}").exists():
                solo.unlink(missing_ok=True)
        if free_gb >= float(d["MEM_KILL_FREE_GB"]) or time.time() - self.last_kill < 20:   # give the kernel time to free the last one
            return free_gb
        why = f"memory nearly full ({free_gb:.1f} GB free of {lim / 2 ** 30:.0f} GB)"
        busy = [(rss(self.procs[i["wid"]].pid), tid, i["wid"]) for tid, i in self.running().items()
                if i["wid"] in self.procs and self.procs[i["wid"]].poll() is None]
        size, tid, wid = max(busy) if busy else (0, None, None)
        grown = [s for s in gpu_servers() if s[0] > float(d.get("MEM_GPU_SERVER_MAX_GB", 4)) * 2 ** 30]
        if grown and max(grown)[0] > size:                   # a GPU server holds the RAM: restart it rather than lose pieces
            b, pid, name = max(grown)
            try:
                os.kill(pid, 9); self.last_kill = time.time()
                self.mem_events.append(f"{why}: GPU server {name} held {b / 2 ** 30:.1f} GB; restarted")
            except ProcessLookupError:
                pass
        elif busy:
            p = self.procs[wid]
            p.kill(); p.wait(10); self.last_kill = time.time()
            # a small worker was not the cause: its piece is redone without counting toward "needs more memory than the box has"
            self.requeue(tid, wid, f"{why}, worker {wid} ({size / 2 ** 30:.1f} GB) stopped",
                         count=size > 0.6 * float(d["MEM_PER_WORKER_GB"]) * 2 ** 30)
            self.planned.add(wid)
        return free_gb

    def maintain(self, doc_timeout_s):
        """kill workers stuck on a cancelled/over-long document, relaunch crashed workers, release their claims"""
        with self.mtx:
            self.guard_memory()
            now = time.time()
            for tid, info in self.running().items():
                try:
                    task = json.loads((self.dir / "tasks" / f"{tid}.json").read_text())
                except Exception:
                    continue
                why = "cancelled" if (self.dir / "cancel" / task["job"]).exists() else (
                    f"timed out after {doc_timeout_s / 60:.0f} min" if now - info["t0"] > doc_timeout_s else None)
                if why:
                    p = self.procs.get(info["wid"])
                    if p and p.poll() is None:
                        p.kill(); p.wait(10)
                    self._fail(tid, task, why, info["wid"])
            for wid, p in list(self.procs.items()):
                if p.poll() is None:
                    continue
                planned = wid in self.planned or p.returncode == self.RECYCLED
                self.planned.discard(wid)
                for tid, info in self.running().items():
                    if info["wid"] == wid:
                        if p.returncode in (-9, 137):          # killed by the kernel: almost always memory
                            self.requeue(tid, wid, "worker killed by the system (out of memory)")
                        else:
                            task = json.loads((self.dir / "tasks" / f"{tid}.json").read_text())
                            self._fail(tid, task, f"the worker process died (exit code {p.returncode}); most likely a damaged file", wid)
                for cl in (self.dir / "tasks").glob("*.claim"):       # claimed but never started: give it to someone else
                    try:
                        if cl.read_text() == wid and not (self.dir / "running" / f"{cl.stem}.json").exists() and not (self.dir / "results" / f"{cl.stem}.json").exists():
                            cl.unlink()
                    except OSError:
                        pass
                if planned:
                    self.launch(wid)                              # restarts on purpose do not use up the crash budget
                elif self.restarts > 0:
                    self.restarts -= 1; self.launch(wid)
                else:
                    self.broken = True

    TQDM = re.compile(r"([A-Za-z][A-Za-z ./()-]{2,40}?):\s+(\d+)%\|[^|]*\|\s*(\d+/\d+)")

    def progress_line(self, wid):
        """the last progress bar Marker printed in this worker's log, e.g. 'Recognizing Layout 12/40'"""
        try:
            p = self.dir / "logs" / f"{wid}.log"
            with open(p, "rb") as f:
                f.seek(max(0, p.stat().st_size - 2000)); tail = f.read().decode("utf-8", "replace")
            m = list(self.TQDM.finditer(tail.replace("\r", "\n")))
            return f"{m[-1].group(1).strip()} {m[-1].group(3)}" if m else ""
        except Exception:
            return ""

    def log(self, wid, n=25):
        p = self.dir / "logs" / f"{wid}.log"
        return "\n".join(p.read_text(errors="replace").replace("\r", "\n").splitlines()[-n:]) if p.exists() else "(no log)"

    def status(self):
        out = {}
        for wid, p in list(self.procs.items()):
            out[wid] = "dead" if p.poll() is not None else ("ready" if (self.dir / f"alive_{wid}.json").exists() else "loading")
        return out


POOL = Pool()


def pool_shape(cpus, workers=0, threads=0, vlm_slots=0, vlm_parallel=0):
    """-> (workers, threads per worker, VLM requests in flight per worker).
    Indra's ONNX layout/OCR runs on the CPU and scales with processes, not threads, so the default is one single-threaded
    worker per CPU; together the workers keep ~1.5x the GPU servers' slots busy (the extra hides CPU gaps between requests)."""
    workers = int(workers) or max(1, cpus)
    threads = int(threads) or max(1, cpus // workers)
    parallel = int(vlm_parallel) or (max(2, math.ceil(1.5 * int(vlm_slots) / workers)) if int(vlm_slots) else 4)
    return workers, threads, parallel


def workers_for_memory(cpus, limit, used, per_worker_gb, reserve_gb, per_cpu=1.0):
    """up to per_cpu workers per CPU (much of a worker's time is waiting for the GPU servers, its CPU idle), but only as many
    as the free RAM can hold (measured after the GPU servers have started)"""
    free_gb = (limit - used) / 2 ** 30 - reserve_gb
    return max(1, min(math.ceil(cpus * per_cpu), int(free_gb // per_worker_gb)))


def start_budget(free_gb, pause_gb, start_gb):
    """how many pieces may start now: each grows by about start_gb in its first minute, and free RAM must stay above the
    pause level (several workers starting at once would otherwise all see the same free RAM)"""
    return max(0, int((free_gb - pause_gb) // start_gb))


def start_pool():
    d = DEFAULTS
    cpus = os.cpu_count() or 2
    if not int(d["WORKERS"]):
        lim, used = memory()
        n = workers_for_memory(cpus, lim, used, float(d["MEM_PER_WORKER_GB"]), float(d["MEM_RESERVE_GB"]), float(d["WORKERS_PER_CPU"]))
        print(f"RAM {used / 2 ** 30:.1f} of {lim / 2 ** 30:.1f} GB in use -> {n} workers", flush=True)
        d = dict(d, WORKERS=n)
    workers, threads, parallel = pool_shape(cpus, d["WORKERS"], d["CPU_THREADS"], d["VLM_SLOTS"], d["VLM_PARALLEL"])
    POOL.start(workers, d["VLM_URL"], parallel, threads)


# ----------------------------------------------------------------- page chunks of long PDFs
def parse_pages(spec, total):
    """'0,5-10' -> sorted page numbers below total (None/'' = every page)"""
    if not spec:
        return list(range(total))
    pages = set()
    for part in str(spec).replace(" ", "").split(","):
        a, _, b = part.partition("-")
        pages.update(range(int(a), int(b or a) + 1))
    return sorted(p for p in pages if 0 <= p < total)


def format_pages(pages):
    """[0,1,2,5,7,8] -> '0-2,5,7-8'"""
    runs, start = [], None
    for k, p in enumerate(pages):
        if start is None:
            start = p
        if k + 1 == len(pages) or pages[k + 1] != p + 1:
            runs.append(str(start) if start == p else f"{start}-{p}"); start = None
    return ",".join(runs)


def plan_chunks(pages, workers, min_pages=10, max_pages=40):
    """contiguous page chunks, min..max pages long, enough for every worker to get about two, in whole rounds: the count is a
    multiple of the workers (6 chunks on 5 workers left one worker a second round while four waited: the 27-page suite took
    ~1/3 longer than its work), rounded up while every chunk keeps min_pages, and the sizes as even as possible"""
    n = len(pages)
    if n < 2 * min_pages:
        return [pages]
    most = n // min_pages
    k = max(math.ceil(n / max_pages), min(2 * workers, most))
    if workers > 1 and k > workers and k % workers:
        up = math.ceil(k / workers) * workers
        k = up if up <= most else max(workers, k // workers * workers)
    k = max(1, min(k, n))
    base, extra = divmod(n, k)
    out, at = [], 0
    for i in range(k):
        size = base + (1 if i < extra else 0)
        out.append(pages[at:at + size]); at += size
    return out


def pdf_pages(path):
    try:
        import pypdfium2 as pdfium                       # installed with Marker
        d = pdfium.PdfDocument(str(path))
        try:
            return len(d)
        finally:
            d.close()
    except Exception:
        return None


def chunk_plan(path, opts, workers, o):
    """page lists to convert as separate tasks, or None to convert the document in one piece"""
    # shortcut: only native PDFs in markdown output are split; revisit when long DjVu/office files or html/json output need it
    if Path(path).suffix.lower() != ".pdf" or opts.get("OUTPUT_FORMAT", "markdown") != "markdown" or not int(o["SPLIT_MIN_PAGES"]):
        return None
    total = pdf_pages(path)
    pages = parse_pages(opts.get("PAGE_RANGE"), total or 0)
    if len(pages) < int(o["SPLIT_MIN_PAGES"]):
        return None
    parts = plan_chunks(pages, workers, int(o["CHUNK_MIN_PAGES"]), int(o["CHUNK_MAX_PAGES"]))
    return parts if len(parts) > 1 else None


RENDER_LOCK = threading.Lock()      # one LibreOffice / ddjvu at a time on the box: each can take GBs of RAM


def pre_render(path, i, opts, warnings, note=lambda name: None):
    """office documents and DjVu -> PDF here, before planning, so a long one is split into page chunks like any PDF and no
    worker ever holds LibreOffice and Marker in memory together. Anything else (or a failure) is left to the worker as before."""
    try:
        action, fmt = convert.route(str(path), opts.get("OFFICE", "pdf"))
    except convert.Unsupported:
        return path                                          # the worker reports it
    if action not in ("soffice", "ddjvu") or fmt != "pdf":
        return path
    out = Path(path).parent / ".rendered" / str(i)           # one folder per document: a.doc and a.ppt never collide
    with RENDER_LOCK:
        note(Path(path).name)
        try:
            return (convert.soffice(path, "pdf", out, TMP_ROOT / "lo_profile_api") if action == "soffice"
                    else convert.djvu_to_pdf(path, out))
        except Exception as e:
            warnings.append(f"{Path(path).name}: rendering to PDF first failed ({str(e)[:150]}); converted in one piece")
            return path


STEALABLE = {}          # job id -> chunk task ids, for steal()


def steal(job_id, n):
    """hands up to n not-yet-started page chunks of a running job to the client's local workers (latest pages first, since
    the box's workers take the earliest). The box writes a placeholder for them; the client fills it in."""
    got = []
    for tid in reversed(STEALABLE.get(job_id, [])):
        if len(got) >= n:
            break
        if (POOL.dir / "results" / f"{tid}.json").exists():
            continue
        try:
            fd = os.open(str(POOL.dir / "tasks" / f"{tid}.claim"), os.O_CREAT | os.O_EXCL | os.O_WRONLY)    # same claim as a worker
            os.write(fd, b"remote"); os.close(fd)
        except FileExistsError:
            continue
        t = json.loads((POOL.dir / "tasks" / f"{tid}.json").read_text())
        jwrite(POOL.dir / "results" / f"{tid}.json", {"ok": True, "remote": True, "tid": tid, "job": t["job"], "doc": t["doc"], "rel": t["rel"],
                                                      "worker": "client", "sec": 0, "pages": None})
        got.append({"file": t["rel"], "pages": t["opts"]["PAGE_RANGE"], "folder": t["out_rel"], "stem": t["stem"]})
    return got


# ----------------------------------------------------------------- inputs
def zip_name(info):
    """the real file name: zips made on Windows store non-English names in the local code page (GBK, Shift-JIS...), which
    Python reads as CP437 gibberish; names flagged UTF-8 are kept"""
    if info.flag_bits & 0x800:
        return info.filename
    raw = info.filename.encode("cp437", errors="replace")
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return convert.guess_decode(raw) or info.filename


def safe_extract(zpath, dest, max_files, max_bytes):
    """extract only the supported documents of a zip, refusing path tricks and zip bombs; returns [(abs_path, rel_path)]"""
    dest = Path(dest).resolve(); docs = []
    with zipfile.ZipFile(zpath) as z:
        infos = [i for i in z.infolist() if not i.is_dir() and i.filename.lower().endswith(DOC_EXT)
                 and not i.filename.startswith("__MACOSX") and not Path(i.filename).name.startswith(".")]
        if len(infos) > max_files:
            raise RuntimeError(f"the zip holds {len(infos)} documents; the limit is {max_files}")
        if sum(i.file_size for i in infos) > max_bytes:
            raise RuntimeError("the documents in the zip are too large when unpacked")
        for i in infos:
            parts = [safe_part(p, 100) for p in Path(zip_name(i).replace("\\", "/")).parts if p not in ("", ".", "..")]
            target = dest.joinpath(*parts).resolve()
            if dest not in target.parents:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(i) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, 8 * 1024 * 1024)
            docs.append((target, "/".join(parts)))
    return sorted(docs, key=lambda d: d[1].lower())


def llm_plan(opts, workers, warnings):
    """which LLM passes a job gets: the box's own vision LLM (LLM_URL) first, else Gemini, else none"""
    d = DEFAULTS
    have = bool(d["LLM_URL"]) or bool(os.environ.get("GOOGLE_API_KEY"))
    want = opts.get("USE_LLM", bool(d["USE_LLM"]))
    if want and not have:
        if opts.get("USE_LLM"):
            warnings.append("--llm asked for, but this box has no LLM (no local model running, no GOOGLE_API_KEY): converted without it")
        return {"USE_LLM": False}
    if not want:
        return {"USE_LLM": False}
    plan = {"USE_LLM": True, "LLM_CTX": int(d["LLM_CTX"])}
    for key in ("REDO_INLINE_MATH", "DESCRIBE_FIGURES", "PAGE_REVIEW"):   # set by the job or the box, else by the quality
        if opts.get(key, d[key]) is not None:
            plan[key] = bool(opts.get(key, d[key]))
    if d["LLM_URL"]:
        slots = max(1, int(d["LLM_SLOTS"]))
        plan.update(LLM_URL=d["LLM_URL"], LLM_MODEL=d["LLM_MODEL"],
                    LLM_CONCURRENCY=max(2, slots),           # pages / figures a worker works on at once: alone (the last chunk) it fills every slot
                    LLM_INFLIGHT=max(4, slots),              # its requests in flight at most: all workers together can't flood the queues
                    MARKER_LLM_CONCURRENCY=2 * max(1, math.ceil(2.5 * slots / workers)))   # Marker's own LLM steps (not gated): as before
        # 2.5x the slots over all workers: while some workers are in Marker's GPU stage, the others still fill every LLM slot
        # (extra requests wait in llama-server's queue, well within the 900 s timeout)
    return plan


def add_review(row, stats):
    """page-review totals for a report row: pages_rewritten, and why the others kept Marker's draft"""
    stats = [s for s in stats if s]
    if stats:
        row["pages_rewritten"] = sum(s.get("pages_rewritten", s.get("rewritten", 0)) for s in stats)
        kept = {}
        for s in stats:
            for why, n in (s.get("pages_kept_draft") or s.get("kept_draft") or {}).items():
                kept[why] = kept.get(why, 0) + n
        row["pages_kept_draft"] = kept
        row["review_examples"] = [e for s in stats for e in (s.get("examples") or [])][:12]


def add_stages(row, stages):
    """worker seconds per stage (marker, figures, describe_figures, page_review), summed over the pieces"""
    tot = {}
    for st in stages:
        for k, v in (st or {}).items():
            tot[k] = round(tot.get(k, 0) + v, 1)
    if tot:
        row["stage_seconds"] = tot


def resolve_mode(requested, warnings):
    m = requested or DEFAULTS["MODE"]
    if m == "auto":
        return "balanced" if POOL.vlm_ok else "fast"
    if m in ("balanced", "vllm") and not POOL.vlm_ok:
        warnings.append(f"mode '{m}' needs the VLM server, which is not running on this box; used 'fast' instead")
        return "fast"
    return m


# ----------------------------------------------------------------- the job
class ResourceSampler(threading.Thread):
    """GPU busy time and memory, CPU and free RAM every few seconds while a job runs, for its report: is the box used fully?"""
    def __init__(self, every=2.0):
        super().__init__(daemon=True)
        self.every, self.samples, self.done = every, [], threading.Event()

    @staticmethod
    def cpu_seconds():
        for path, scale in (("/sys/fs/cgroup/cpu.stat", 1e-6), ("/sys/fs/cgroup/cpuacct/cpuacct.usage", 1e-9)):
            try:
                text = Path(path).read_text()
                return float(text.split("usage_usec")[1].split()[0] if "usage_usec" in text else text.strip()) * scale
            except (OSError, ValueError, IndexError):
                continue
        try:                                                 # no cgroup: the whole machine
            f = Path("/proc/stat").read_text().split("\n")[0].split()[1:]
            return (sum(map(float, f)) - float(f[3]) - float(f[4])) / os.sysconf("SC_CLK_TCK")
        except (OSError, ValueError, IndexError):
            return None

    def run(self):
        cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
        last_t, last_c = time.time(), self.cpu_seconds()
        while not self.done.wait(self.every):
            try:
                out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader,nounits"],
                                     capture_output=True, text=True, timeout=5).stdout
                gpus = [tuple(int(x) for x in l.split(",")) for l in out.strip().splitlines() if l.strip()]
            except Exception:
                gpus = []
            lim, used = memory()
            now, c = time.time(), self.cpu_seconds()
            cpu = 100 * (c - last_c) / max(1e-6, (now - last_t) * cpus) if c is not None and last_c is not None else None
            last_t, last_c = now, c
            self.samples.append((gpus, (lim - used) / 2 ** 30, cpu))

    def summary(self):
        self.done.set()
        s = self.samples
        if not s:
            return {}
        n = max(len(g) for g, _, _ in s)
        util = [[g[i][0] for g, _, _ in s if len(g) > i] for i in range(n)]
        out = {"seconds_sampled": round(len(s) * self.every),
               "gpu_busy_pct": [round(sum(u) / len(u)) for u in util if u],
               "gpu_mem_max_mb": [max(g[i][1] for g, _, _ in s if len(g) > i) for i in range(n)],
               "ram_free_min_gb": round(min(f for _, f, _ in s), 1)}
        cpu = [c for _, _, c in s if c is not None]
        if cpu:
            out["cpu_busy_pct"] = round(sum(cpu) / len(cpu))
        return out


def run_job(job, opts, session_deadline, upd, cancelled):
    """job: {id, name, src, dir}. upd(stage, pct, msg) reports progress; cancelled() -> bool. Returns a result summary dict."""
    o = {**DEFAULTS, **(opts or {})}
    jdir = Path(job["dir"]); inp, out = jdir / "in", jdir / "out"
    for d in (inp, out):
        shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    t_start = time.time()
    deadline = min(session_deadline - 120, t_start + float(o["TIME_BUDGET_H"]) * 3600)
    warnings, tids = [], {}
    upd("preparing", 1, "unpacking")
    name = job["name"]
    if name.lower().endswith(".zip"):
        docs = safe_extract(job["src"], inp, int(o["MAX_ZIP_FILES"]), int(float(o["MAX_UNZIPPED_GB"]) * 2 ** 30))
        if not docs:
            raise RuntimeError("no supported documents in the zip (supported: " + ", ".join(DOC_EXT) + ")")
        tag = safe_part(Path(name).stem)
    else:
        dst = inp / safe_part(name, 120)
        shutil.move(str(job["src"]), str(dst)); docs = [(dst, dst.name)]
        tag = safe_part(dst.stem)
    mode = resolve_mode((opts or {}).get("MODE"), warnings)
    task_opts = {k: v for k, v in (opts or {}).items() if k != "MODE"}
    task_opts["MODE"] = mode
    task_opts["OFFICE"] = (opts or {}).get("OFFICE") or o["OFFICE"]
    task_opts["QUALITY"] = str((opts or {}).get("QUALITY") or o["QUALITY"]).lower()
    if task_opts["QUALITY"] not in R.QUALITY_PROFILES:
        warnings.append(f"unknown quality {task_opts['QUALITY']!r}: balanced used"); task_opts["QUALITY"] = "balanced"
    llm = llm_plan(opts or {}, max(1, len(POOL.procs)), warnings)
    task_opts.update(llm)
    used, out_rel, doc_dirs, chunks = set(), {}, {}, {}      # chunks: doc index -> [(tid, pages)]
    workers = max(1, len(POOL.procs))
    for i, (path, rel) in enumerate(docs):
        rel_dir, stem = str(Path(rel).parent) if Path(rel).parent != Path(".") else "", Path(rel).stem
        key = (rel_dir.lower(), stem.lower())
        folder = stem if key not in used else f"{stem}_{Path(rel).suffix.lstrip('.').lower()}"
        used.add(key); used.add((rel_dir.lower(), folder.lower()))
        out_dir = out / rel_dir / folder if rel_dir else out / folder
        out_rel[i] = str(out_dir.relative_to(out)).replace(os.sep, "/")
        doc_dirs[i] = (out_dir, folder)
        path = pre_render(path, i, task_opts, warnings, lambda name: upd("preparing", 2, f"rendering {name} to PDF"))
        t = {"job": job["id"], "doc": i, "src": str(path), "rel": rel, "stem": folder, "out": str(out_dir), "out_rel": out_rel[i], "opts": task_opts}
        plan = chunk_plan(path, task_opts, workers, o)
        if not plan:
            tids[POOL.enqueue(t)] = i
            continue
        for k, part in enumerate(plan):
            pages = format_pages(part)
            tid = POOL.enqueue(dict(t, out=str(out_dir / ".chunks" / f"{k:04d}"), opts=dict(task_opts, PAGE_RANGE=pages)))
            tids[tid] = i; chunks.setdefault(i, []).append((tid, pages))
    STEALABLE[job["id"]] = [tid for c in chunks.values() for tid, _ in c]
    n = len(docs)
    n_tasks = len(tids)
    results, last_msg, expired = {}, 0, False
    doc_timeout = float(o["DOC_TIMEOUT_MIN"]) * 60
    sampler = ResourceSampler(); sampler.start()           # GPU / CPU / RAM use while this job runs (its report)
    try:
        while len(results) < n_tasks:
            if cancelled():
                POOL.cancel_job(job["id"]); POOL.maintain(doc_timeout)      # stops the documents of this job that are running right now
                raise Cancelled()
            if not expired and time.time() > deadline:
                expired = True; POOL.cancel_job(job["id"])
                warnings.append("time budget reached: documents that were not finished are listed as failed")
            POOL.maintain(doc_timeout)
            if POOL.broken:
                raise RuntimeError("conversion workers keep crashing:\n" + "\n".join(f"--- {w}\n{POOL.log(w, 15)}" for w in POOL.procs))
            for tid in tids:
                if tid not in results:
                    r = POOL.result(tid)
                    if r:
                        results[tid] = r
            if time.time() - last_msg > 2:
                last_msg = time.time()
                run = [(tid, i) for tid, i in POOL.running().items() if tid in tids and tid not in results]
                now_txt = "; ".join(f"{docs[tids[tid]][1]} ({i['wid']}, {time.time() - i['t0']:.0f}s" + (f", {POOL.progress_line(i['wid'])}" if POOL.progress_line(i['wid']) else "") + ")" for tid, i in run[:3])
                done = f"{len(results)}/{n_tasks} parts of {n} documents" if chunks else f"{len(results)}/{n} documents"
                if (POOL.dir / "pause").exists():
                    done += f" · RAM low ({(_read(str(POOL.dir / 'pause')) or '?')} GB free): one piece at a time"
                upd("converting", 3 + 92 * len(results) / n_tasks, f"[{mode}] {done} done" + (f" · working on {now_txt}" if now_txt else " · waiting for a free worker"))
            time.sleep(0.4)
        # ---- collect + bundle
        upd("bundling", 96, "writing the report and the zip")
        STEALABLE.pop(job["id"], None)
        per_doc = []
        for i in range(n):
            rs = [results[tid] for tid, d in tids.items() if d == i]
            r = next((x for x in rs if x.get("ok") and not x.get("remote")), rs[0])
            row = {"file": docs[i][1], "ok": any(x.get("ok") for x in rs), "pages": sum(x.get("pages") or 0 for x in rs if x.get("ok")) or None,
                   "seconds": round(sum(x.get("sec") or 0 for x in rs), 2), "worker": ",".join(sorted({str(x.get("worker")) for x in rs})), "via": r.get("via")}
            if i in chunks:
                parts = [(pages, Path(doc_dirs[i][0]) / ".chunks" / f"{k:04d}",
                          "remote" if results[tid].get("remote") else ("ok" if results[tid].get("ok") else results[tid].get("error") or "failed"))
                         for k, (tid, pages) in enumerate(chunks[i])]
                row.update(convert.merge_chunks(doc_dirs[i][0], doc_dirs[i][1], parts))
            cov = [c for x in rs for c in (x.get("coverage") or [])]
            if cov:                                            # share of the PDF text layer's words found in the Markdown
                row["text_coverage"] = round(sum(c["found"] * c["words"] for c in cov) / max(1, sum(c["words"] for c in cov)), 3)
                row["low_coverage_pages"] = [c["page"] for c in cov if c["low"]]
            if any(x.get("described_figures") for x in rs):
                row["figures_described"] = sum(x.get("described_figures") or 0 for x in rs)
            add_review(row, [x.get("page_review") for x in rs])
            add_stages(row, [x.get("stage_seconds") for x in rs])
            notes = sorted({x["forced_ocr"] for x in rs if x.get("forced_ocr")})
            if notes:
                row["forced_ocr"] = notes
            rej = {}
            for x in rs:
                for k, v in (x.get("marker_llm_rejected") or {}).items():
                    rej[k] = rej.get(k, 0) + v
            if rej:
                row["marker_llm_rejected"] = rej
            skipped = {}
            for x in rs:
                for k, v in (x.get("tables_skipped") or {}).items():
                    skipped[k] = skipped.get(k, 0) + v
            if skipped:
                row["tables_skipped"] = skipped
            dskip = {}
            for x in rs:
                for k, v in (x.get("drawn_formulas_skipped") or {}).items():
                    dskip[k] = dskip.get(k, 0) + v
            if dskip:
                row["drawn_formulas_skipped"] = dskip
            errs = sorted({x["tables_error"] for x in rs if x.get("tables_error")})
            if errs:
                row["tables_error"] = errs
            audits = [x.get("audit") for x in rs if x.get("audit")]
            if audits:                                         # accurate: what the line-by-line audit proposed and did
                total = {}
                for st in audits:
                    for k, v in st.items():
                        if isinstance(v, int):
                            total[k] = total.get(k, 0) + v
                total["examples"] = [e for st in audits for e in st.get("examples", [])][:12]
                row["audit"] = total
            qs = sorted({x.get("quality") for x in rs if x.get("quality")})
            if qs:
                row["quality"] = qs[0] if len(qs) == 1 else qs
            for key in ("tables_realigned", "table_pictures", "drawn_formulas", "layer_fixes"):
                if any(x.get(key) for x in rs):
                    row[key] = sum(x.get(key) or 0 for x in rs)
            if any(x.get("indic_words_fixed") for x in rs):
                row["indic_words_fixed"] = sum(x.get("indic_words_fixed") or 0 for x in rs)
            if not row["ok"]:
                row["error"] = r.get("error")
            else:
                row["output_dir"] = out_rel[i]
            per_doc.append(row)
        pages = sum(d["pages"] or 0 for d in per_doc if d["ok"])
        warnings += [e for e in POOL.mem_events if any(e.startswith(dd[1] + ":") for dd in docs)]
        resources = sampler.summary()                         # GPU busy %, CPU %, lowest free RAM while this job ran
        for row in per_doc:
            row["box_resources"] = resources
        report = {"mode": mode, "llm": (o["LLM_MODEL"] or "gemini") if llm.get("USE_LLM") else None, "documents": per_doc, "warnings": warnings,
                  "wall_s": round(time.time() - t_start), "pages": pages, "resources": resources}
        (out / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        bundle = shutil.make_archive(str(jdir / f"{tag}.markdown"), "zip", out)
        n_ok = sum(d["ok"] for d in per_doc)
        return {"mode": mode, "docs": per_doc, "n_ok": n_ok, "n_failed": n - n_ok, "warnings": warnings, "pages": pages, "bundle": bundle,
                "files": sorted(str(p.relative_to(out)) for p in out.rglob("*") if p.is_file() and p.suffix in (".md", ".json", ".html"))[:200],
                "wall_s": round(time.time() - t_start), "partial": n_ok < n}
    finally:
        sampler.done.set()                                       # a failed or cancelled job stops sampling too
        STEALABLE.pop(job["id"], None)
        shutil.rmtree(inp, ignore_errors=True)

In [ ]:
%%writefile mdbox/server.py
"""Document -> Markdown box API (runs on the Kaggle GPU machine, exposed through a Cloudflare tunnel).

Every request needs `Authorization: Bearer <BOX_TOKEN>` (except GET /ping).
Cloudflare's free tunnel rejects request bodies over ~100 MB and cuts requests after ~100 s, so a document is uploaded in parts:

  POST   /jobs                 {"name", "size", "options"?}         -> {"job_id", "part_size"}
  PUT    /jobs/{id}/parts/{n}  raw bytes (header X-Sha256 optional)  -> {"part", "bytes"}
  POST   /jobs/{id}/start      {"parts": N}                          -> job status (queued)
  GET    /jobs/{id}                                                  -> job status / progress
  GET    /jobs/{id}/bundle.zip   everything in one zip (<name>/<name>.md + images, report.json)
  GET    /jobs/{id}/files/{path} one output file
  DELETE /jobs/{id}            cancel (running) or delete (finished)
  POST   /files {"name","size"} -> {"file_id","part_size"};  PUT /files/{fid}/parts/{n};  POST /files/{fid}/finish {"parts"}
         -> {"pages","pdf"}  store one document once (office/DjVu rendered to PDF);  DELETE /files/{fid}
  POST   /jobs {"file_id", "name", "options": {"PAGE_RANGE": ...}}  a job on a stored file (one batch of pages), queued at once
  POST   /jobs/{id}/steal      {"n"}  -> {"chunks": [...]}  page chunks the client converts on its own workers
  *      /vlm/...  /llm/...    this box's Surya VLM / helper vision LLM (OpenAI API), for the client's own workers
  GET    /jobs      GET /health      GET /ping (no auth)
A job is one document (see pipeline.DOC_EXT: pdf, images, Word/PowerPoint/Excel old and new, OpenDocument, RTF, DjVu, html, epub, txt/md/csv)
or a .zip of documents. Up to RUNNERS jobs run at the same time."""
import asyncio, hashlib, hmac, os, queue, re, shutil, subprocess, threading, time, traceback, uuid
from pathlib import Path

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import FileResponse, JSONResponse, PlainTextResponse, Response

import pipeline

TOKEN = os.environ["BOX_TOKEN"].strip()
if len(TOKEN) < 16:
    raise SystemExit("BOX_TOKEN must be at least 16 characters")
ROOT = Path(os.environ.get("MD_ROOT", "jobs")).resolve()
UPLOADS = Path(os.environ.get("MD_UPLOADS", str(pipeline.TMP_ROOT / "md_uploads"))).resolve()
DEADLINE = float(os.environ.get("BOX_DEADLINE", "0") or 0) or time.time() + 10 * 3600
PART_SIZE = 16 * 1024 * 1024
MAX_UPLOAD = int(float(os.environ.get("MD_MAX_UPLOAD_GB", "3")) * 2 ** 30)
HAS_GEMINI = bool(os.environ.get("GOOGLE_API_KEY"))
RUNNERS = int(os.environ.get("MD_RUNNERS", "8"))       # jobs at once; their documents/pages share the worker queue
ABANDON_S = float(os.environ.get("MD_ABANDON_MIN", "15")) * 60     # a job no client has asked about for this long is cancelled
                                                       # (its client was killed: it would hold the workers for nothing); 0 = never
VLM_BASE = pipeline.DEFAULTS["VLM_URL"].rstrip("/").removesuffix("/v1")
LLM_BASE = pipeline.DEFAULTS["LLM_URL"].rstrip("/").removesuffix("/v1")
HAS_LLM = bool(LLM_BASE) or HAS_GEMINI
ROOT.mkdir(parents=True, exist_ok=True); UPLOADS.mkdir(parents=True, exist_ok=True)

JOBS, ORDER, Q, LOCK = {}, [], queue.Queue(), threading.Lock()
STARTED = time.time()
LAST_ACTIVITY = [STARTED]
app = FastAPI(title="markdown box", docs_url=None, redoc_url=None, openapi_url=None)

OPT_TYPES = {"MODE": str, "OUTPUT_FORMAT": str, "FORCE_OCR": bool, "STRIP_EXISTING_OCR": bool, "REDO_INLINE_MATH": bool, "PAGINATE": bool,
             "KEEP_HEADERS_FOOTERS": bool, "IMAGES": bool, "PAGE_RANGE": str, "USE_LLM": bool, "OFFICE": str,
             "FIGURE_FIX": bool, "DESCRIBE_FIGURES": bool, "PAGE_REVIEW": bool, "QUALITY": str, "AUDIT": bool}


def clean_options(raw):
    out = {}
    for k, v in (raw or {}).items():
        k = k.upper()
        if k not in OPT_TYPES:
            raise HTTPException(400, f"unknown option {k!r}; allowed: {sorted(OPT_TYPES)}")
        if v is None:
            continue
        if OPT_TYPES[k] is bool and not isinstance(v, bool):
            raise HTTPException(400, f"option {k} must be true or false")
        if OPT_TYPES[k] is str and not isinstance(v, str):
            raise HTTPException(400, f"option {k} must be a string")
        if k == "MODE" and v not in pipeline.MODES:
            raise HTTPException(400, f"MODE must be one of {list(pipeline.MODES)}")
        if k == "OFFICE" and v not in pipeline.convert.OFFICE_MODES:
            raise HTTPException(400, f"OFFICE must be one of {list(pipeline.convert.OFFICE_MODES)}")
        if k == "OUTPUT_FORMAT" and v not in ("markdown", "html", "json", "chunks"):
            raise HTTPException(400, "OUTPUT_FORMAT must be markdown, html, json or chunks")
        if k == "PAGE_RANGE" and not re.fullmatch(r"[0-9]+(-[0-9]+)?(,[0-9]+(-[0-9]+)?)*", v.replace(" ", "")):
            raise HTTPException(400, "PAGE_RANGE looks like 0,5-10,20 (page numbers start at 0)")
        if k == "USE_LLM" and v and not HAS_LLM:
            raise HTTPException(400, "USE_LLM needs a local LLM on this box (LLM_LOCAL in the notebook) or the Kaggle secret GOOGLE_API_KEY")
        out[k] = v.replace(" ", "") if k == "PAGE_RANGE" else v
    return out


@app.middleware("http")
async def auth(request: Request, call_next):
    if request.url.path == "/ping" and request.method == "GET":
        return await call_next(request)
    if not hmac.compare_digest(request.headers.get("authorization", "").encode(), f"Bearer {TOKEN}".encode()):
        return JSONResponse({"error": "unauthorized"}, status_code=401)
    return await call_next(request)


@app.get("/ping")
async def ping():
    return PlainTextResponse("ok")


def gpus():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=5).stdout
        rows = [[c.strip() for c in l.split(",")] for l in out.strip().splitlines() if l.strip()]
        return [{"index": int(r[0]), "name": r[1], "mem_used_mb": int(r[2]), "mem_total_mb": int(r[3]), "util_pct": int(r[4])} for r in rows]
    except Exception:
        return []


PUBLIC_KEYS = ("id", "name", "size", "status", "stage", "pct", "msg", "created", "started", "finished", "options", "error",
               "mode", "docs", "n_ok", "n_failed", "warnings", "pages", "partial", "files", "wall_s")


def public(j):
    d = {k: j.get(k) for k in PUBLIC_KEYS}
    if j["status"] == "queued":
        d["queue_position"] = sum(1 for i in ORDER if JOBS[i]["status"] in ("queued", "running") and JOBS[i]["created"] < j["created"]) + 1
    return d


def get_job(jid):
    j = JOBS.get(jid)
    if not j:
        raise HTTPException(404, "no such job")
    return j


def mem_status():
    d, G = pipeline.memory_detail(), 2 ** 30
    return {"limit_gb": round(d["limit"] / G, 1), "used_gb": round(d["used"] / G, 1), "free_gb": round((d["limit"] - d["used"]) / G, 1),
            "file_cache_gb": round(d["cache"] / G, 1), "paused": (pipeline.POOL.dir / "pause").exists(),
            "top_processes_mb": pipeline.processes(8)}


def idle_seconds():
    """0 while anything is uploading/queued/running, else the time since the last job activity (the notebook stops the box when this gets long)"""
    if any(j["status"] in ("uploading", "queued", "running") for j in JOBS.values()):
        return 0
    return round(time.time() - LAST_ACTIVITY[0])


@app.get("/health")
async def health():
    now = time.time()
    return {"ok": True, "box": "markdown", "gpus": gpus(), "uptime_s": round(now - STARTED), "remaining_s": round(DEADLINE - now),
            "workers": pipeline.POOL.status(), "chunk_pages": int(pipeline.DEFAULTS["CHUNK_MAX_PAGES"]), "vlm": pipeline.POOL.vlm_ok, "vlm_slots": int(pipeline.DEFAULTS["VLM_SLOTS"]),
            "waiting_tasks": pipeline.POOL.waiting() + sum(1 for j in JOBS.values() if j["status"] == "queued" or j.get("stage") == "preparing"), "default_mode": "balanced" if pipeline.POOL.vlm_ok else "fast",
            "llm": HAS_LLM, "llm_model": pipeline.DEFAULTS["LLM_MODEL"] or ("gemini" if HAS_GEMINI else ""),
            "llm_local": bool(LLM_BASE), "llm_slots": int(pipeline.DEFAULTS["LLM_SLOTS"]), "llm_default": bool(pipeline.DEFAULTS["USE_LLM"]),
            "defaults": {k: pipeline.DEFAULTS[k] for k in ("REDO_INLINE_MATH", "DESCRIBE_FIGURES", "PAGE_REVIEW", "LLM_CTX", "QUALITY")},
            "memory": mem_status(), "libreoffice": bool(pipeline.convert.SOFFICE), "ddjvu": bool(pipeline.convert.DDJVU), "idle_s": idle_seconds(), "queued": sum(1 for j in JOBS.values() if j["status"] == "queued"),
            "running": [j["id"] for j in JOBS.values() if j["status"] == "running"], "max_upload_gb": MAX_UPLOAD / 2 ** 30,
            "extensions": list(pipeline.DOC_EXT) + [".zip"]}


@app.get("/jobs")
async def list_jobs():
    return [public(JOBS[i]) for i in ORDER]


@app.post("/jobs")
async def create_job(request: Request):
    body = await request.json()
    if body.get("file_id"):
        return await job_from_file(body)
    name, size = Path(str(body.get("name") or "document.pdf")).name, int(body.get("size") or 0)
    if size <= 0 or size > MAX_UPLOAD:
        raise HTTPException(413, f"size must be 1..{MAX_UPLOAD} bytes")
    if not name.lower().endswith(pipeline.DOC_EXT + (".zip",)):
        raise HTTPException(415, f"unsupported file type {Path(name).suffix!r}; supported: {', '.join(pipeline.DOC_EXT)} and .zip of those")
    if DEADLINE - time.time() < 600:
        raise HTTPException(503, "this box session is about to end; start a new one")
    jid = uuid.uuid4().hex[:12]
    j = {"id": jid, "name": name, "size": size, "status": "uploading", "stage": "uploading", "pct": 0.0, "msg": "", "created": time.time(),
         "options": clean_options(body.get("options")), "dir": str(ROOT / jid), "updir": str(UPLOADS / jid), "cancel": False}
    Path(j["dir"]).mkdir(parents=True); Path(j["updir"]).mkdir(parents=True)
    with LOCK:
        JOBS[jid] = j; ORDER.append(jid)
    LAST_ACTIVITY[0] = time.time()
    return {"job_id": jid, "part_size": PART_SIZE}


@app.put("/jobs/{jid}/parts/{n}")
async def put_part(jid: str, n: int, request: Request):
    j = get_job(jid)
    if j["status"] != "uploading":
        raise HTTPException(409, f"job is {j['status']}")
    j["seen"] = time.time()
    return await save_part(j["updir"], n, request, j["size"])


async def save_part(updir, n, request, limit=MAX_UPLOAD):
    """one part of an upload; never more bytes than the whole file was declared to have (the disk can't be filled)"""
    if not 0 <= n < 100000:
        raise HTTPException(400, "bad part number")
    dest = Path(updir) / f"{n:05d}.part"; tmp = dest.with_suffix(".tmp")
    h, size = hashlib.sha256(), 0
    with open(tmp, "wb") as f:
        async for chunk in request.stream():
            size += len(chunk)
            if size > limit:
                f.close(); tmp.unlink(missing_ok=True)
                raise HTTPException(413, "part larger than the declared file size")
            f.write(chunk); h.update(chunk)
    want = request.headers.get("x-sha256")
    if want and want.lower() != h.hexdigest():
        tmp.unlink(missing_ok=True)
        raise HTTPException(422, "checksum mismatch; resend this part")
    os.replace(tmp, dest)
    LAST_ACTIVITY[0] = time.time()
    return {"part": n, "bytes": size}


def assemble(j, n):
    up = Path(j["updir"]); parts = [up / f"{i:05d}.part" for i in range(n)]
    missing = [i for i, p in enumerate(parts) if not p.exists()]
    if missing:
        return missing, 0
    doc = up / "upload.bin"
    with open(doc, "wb") as out:
        for p in parts:
            with open(p, "rb") as f:
                shutil.copyfileobj(f, out, 8 * 1024 * 1024)
            p.unlink()
    return [], doc.stat().st_size


# ----------------------------------------------------------------- stored files (one upload, many page-range jobs)
FILES = {}


def get_file(fid):
    f = FILES.get(fid)
    if not f:
        raise HTTPException(404, "no such file on this box (it may have restarted): upload it again")
    return f


@app.post("/files")
async def create_file(request: Request):
    body = await request.json()
    name, size = Path(str(body.get("name") or "document.pdf")).name, int(body.get("size") or 0)
    if size <= 0 or size > MAX_UPLOAD:
        raise HTTPException(413, f"size must be 1..{MAX_UPLOAD} bytes")
    if not name.lower().endswith(pipeline.DOC_EXT):
        raise HTTPException(415, f"unsupported file type {Path(name).suffix!r}; supported: {', '.join(pipeline.DOC_EXT)}")
    fid = uuid.uuid4().hex[:12]
    d = UPLOADS / f"file_{fid}"; d.mkdir(parents=True)
    FILES[fid] = {"id": fid, "name": name, "size": size, "dir": str(d), "status": "uploading"}
    LAST_ACTIVITY[0] = time.time()
    return {"file_id": fid, "part_size": PART_SIZE}


@app.put("/files/{fid}/parts/{n}")
async def put_file_part(fid: str, n: int, request: Request):
    f = get_file(fid)
    if f["status"] != "uploading":
        raise HTTPException(409, f"file is {f['status']}")
    return await save_part(f["dir"], n, request, f["size"])


def finish_file(f, n, office):
    missing, size = assemble({"updir": f["dir"]}, n)
    if missing:
        return {"error": f"missing parts: {missing[:20]}"}
    if size != f["size"]:
        return {"error": f"assembled {size} bytes but {f['size']} were announced"}
    src = Path(f["dir"]) / pipeline.safe_part(f["name"], 120)
    os.replace(Path(f["dir"]) / "upload.bin", src)
    warnings = []
    path = pipeline.pre_render(src, 0, {"OFFICE": office}, warnings)      # office / DjVu -> PDF once, here
    pdf = path.suffix.lower() == ".pdf"
    f.update(status="ready", path=str(path), pdf=pdf, pages=pipeline.pdf_pages(path) if pdf else None)
    return {"file_id": f["id"], "pages": f["pages"], "pdf": pdf, "warnings": warnings}


@app.post("/files/{fid}/finish")
async def finish(fid: str, request: Request):
    f = get_file(fid)
    if f["status"] != "uploading":
        raise HTTPException(409, f"file is {f['status']}")
    body = await request.json()
    office = clean_options({"OFFICE": (body.get("options") or {}).get("OFFICE")}).get("OFFICE") or pipeline.DEFAULTS["OFFICE"]
    res = await asyncio.get_running_loop().run_in_executor(None, finish_file, f, int(body["parts"]), office)
    if res.get("error"):
        raise HTTPException(409, res["error"])
    LAST_ACTIVITY[0] = time.time()
    return res


@app.delete("/files/{fid}")
async def delete_file(fid: str):
    f = get_file(fid)
    shutil.rmtree(f["dir"], ignore_errors=True)
    FILES.pop(fid, None)
    return {"deleted": fid}


async def job_from_file(body):
    f = get_file(str(body["file_id"]))
    if f["status"] != "ready":
        raise HTTPException(409, f"file is {f['status']}")
    name = Path(str(body.get("name") or Path(f["path"]).name)).name
    if f["pdf"] and not name.lower().endswith(".pdf"):
        name = Path(name).stem + ".pdf"                                   # it was rendered to PDF on upload
    if DEADLINE - time.time() < 600:
        raise HTTPException(503, "this box session is about to end; start a new one")
    jid = uuid.uuid4().hex[:12]
    j = {"id": jid, "name": name, "size": Path(f["path"]).stat().st_size, "status": "queued", "stage": "queued", "pct": 0.0,
         "msg": "waiting for a free slot", "created": time.time(), "options": clean_options(body.get("options")),
         "dir": str(ROOT / jid), "updir": str(UPLOADS / jid), "cancel": False, "file_id": f["id"]}
    Path(j["dir"]).mkdir(parents=True); Path(j["updir"]).mkdir(parents=True)
    os.link(f["path"], Path(j["updir"]) / "upload.bin")                    # the job consumes a link; the stored file stays
    with LOCK:
        JOBS[jid] = j; ORDER.append(jid)
    Q.put(jid)
    LAST_ACTIVITY[0] = time.time()
    return {"job_id": jid, **public(j)}


@app.post("/jobs/{jid}/start")
async def start_job(jid: str, request: Request):
    j = get_job(jid)
    if j["status"] != "uploading":
        raise HTTPException(409, f"job is {j['status']}")
    n = int((await request.json())["parts"])
    missing, size = await asyncio.get_running_loop().run_in_executor(None, assemble, j, n)
    if missing:
        raise HTTPException(409, f"missing parts: {missing[:20]}")
    if size != j["size"]:
        raise HTTPException(422, f"assembled {size} bytes but {j['size']} were announced")
    j.update(status="queued", stage="queued", msg="waiting for a free slot")
    Q.put(jid)
    return public(j)


@app.get("/jobs/{jid}")
async def job_status(jid: str):
    j = get_job(jid)
    j["seen"] = time.time()                                # its client is there (polls every 0.5-2 s)
    return public(j)


def abandoned(j):
    """no client has asked about this job for ABANDON_S: its client was stopped or lost (a killed client's full-suite job once
    ran 27 min beside the next one)"""
    return ABANDON_S > 0 and time.time() - (j.get("seen") or j.get("created") or time.time()) > ABANDON_S


@app.get("/jobs/{jid}/bundle.zip")
async def bundle(jid: str):
    j = get_job(jid)
    if j["status"] != "done" or not j.get("bundle"):
        raise HTTPException(409, f"job is {j['status']}")
    return FileResponse(j["bundle"], media_type="application/zip", filename=Path(j["bundle"]).name)


@app.get("/jobs/{jid}/files/{name:path}")
async def one_file(jid: str, name: str):
    j = get_job(jid)
    base = (Path(j["dir"]) / "out").resolve()
    p = (base / name).resolve()
    if j["status"] != "done" or base not in p.parents or not p.is_file():
        raise HTTPException(404, "no such file")
    return FileResponse(p, filename=p.name)


@app.delete("/jobs/{jid}")
async def delete_job(jid: str):
    j = get_job(jid)
    if j["status"] in ("running", "queued", "uploading"):
        j["cancel"] = True
        if j["status"] != "running":
            j.update(status="cancelled", stage="cancelled", finished=time.time())
            shutil.rmtree(j["updir"], ignore_errors=True)
        return {"cancelled": jid}
    shutil.rmtree(j["dir"], ignore_errors=True)
    with LOCK:
        JOBS.pop(jid, None)
        if jid in ORDER:
            ORDER.remove(jid)
    return {"deleted": jid}


@app.post("/jobs/{jid}/steal")
async def steal(jid: str, request: Request):
    j = get_job(jid)
    j["seen"] = time.time()
    n = int((await request.json()).get("n") or 1)
    if j["status"] != "running" or n < 1:
        return {"chunks": []}
    return {"chunks": pipeline.steal(jid, min(n, 16))}


PROXY_CLIENT = []


async def gpu_proxy(base, what, path, request):
    """the client's own workers use this box's GPU servers through here (the token is checked by the middleware)"""
    if not base:
        raise HTTPException(503, f"no {what} server is running on this box")
    import httpx
    if not PROXY_CLIENT:
        PROXY_CLIENT.append(httpx.AsyncClient(timeout=httpx.Timeout(900.0, connect=10.0), limits=httpx.Limits(max_connections=128)))
    headers = {k: v for k, v in request.headers.items() if k.lower() in ("content-type", "accept")}
    try:
        r = await PROXY_CLIENT[0].request(request.method, f"{base}/{path}", params=request.query_params, content=await request.body(), headers=headers)
    except httpx.HTTPError as e:
        raise HTTPException(502, f"{what} server unreachable: {type(e).__name__}")
    LAST_ACTIVITY[0] = time.time()
    return Response(content=r.content, status_code=r.status_code, media_type=r.headers.get("content-type"))


@app.api_route("/vlm/{path:path}", methods=["GET", "POST"])
async def vlm_proxy(path: str, request: Request):
    return await gpu_proxy(VLM_BASE, "VLM", path, request)


@app.api_route("/llm/{path:path}", methods=["GET", "POST"])
async def llm_proxy(path: str, request: Request):
    return await gpu_proxy(LLM_BASE, "LLM", path, request)


def runner():
    while True:
        jid = Q.get()
        j = JOBS.get(jid)
        if not j or j["status"] != "queued":
            continue
        if abandoned(j):
            j.update(status="cancelled", stage="cancelled", msg="cancelled: no client asked about it", finished=time.time())
            shutil.rmtree(j["updir"], ignore_errors=True)
            continue
        j.update(status="running", started=time.time(), stage="starting", pct=0.0)

        def upd(stage, pct=None, msg="", j=j):
            j["stage"], j["msg"] = stage, msg
            if pct is not None:
                j["pct"] = round(float(pct), 1)
        try:
            res = pipeline.run_job({"id": jid, "name": j["name"], "src": str(Path(j["updir"]) / "upload.bin"), "dir": j["dir"]},
                                   j["options"], DEADLINE, upd, lambda j=j: j["cancel"] or abandoned(j))
            j.update(res, status="done", stage="done", pct=100.0, msg=f"{res['n_failed']} of {res['n_ok'] + res['n_failed']} documents failed (see report.json)" if res["n_failed"] else "")
        except pipeline.Cancelled:
            j.update(status="cancelled", stage="cancelled", msg="cancelled" if j["cancel"] else "cancelled: no client asked about it")
        except Exception as e:
            traceback.print_exc()
            j.update(status="error", stage="error", error=f"{type(e).__name__}: {e}"[:1500])
        finally:
            j["finished"] = time.time(); LAST_ACTIVITY[0] = time.time()
            shutil.rmtree(j["updir"], ignore_errors=True)


def memory_watch():
    """runs all the time, not only while a job polls: RAM can fill between two progress ticks"""
    while True:
        try:
            with pipeline.POOL.mtx:
                pipeline.POOL.guard_memory()
        except Exception:
            traceback.print_exc()
        time.sleep(0.5)


pipeline.start_pool()
threading.Thread(target=memory_watch, daemon=True).start()
for _ in range(RUNNERS):
    threading.Thread(target=runner, daemon=True).start()

## 6. Preflight: GPUs, internet, secrets

In [ ]:
import re, json, hashlib, hmac, threading, urllib.request, urllib.error
T0 = time.time(); DEADLINE = T0 + SESSION_HOURS * 3600
def secret(name):
    v = os.environ.get(name)
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        except Exception: v = None
    return v
BOX_TOKEN = (secret("BOX_TOKEN") or "").strip()
if len(BOX_TOKEN) < 16: raise RuntimeError("Secret BOX_TOKEN is missing or shorter than 16 chars: Kaggle -> Add-ons -> Secrets, tick it for this notebook.")
for s in ("HF_TOKEN", "GOOGLE_API_KEY"):
    if secret(s): os.environ[s] = secret(s)
out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout
gpus = [l for l in out.strip().splitlines() if l.strip()]
print("GPUs:", *gpus, sep="\n   ")
if USE_VLM and VLM_BIN and len(gpus) < 2: print("WARNING: fewer than 2 GPUs (Settings -> Accelerator -> 'GPU T4 x2'). It still works, just slower.")
print(f"session ends in {SESSION_HOURS} h | idle stop {IDLE_STOP_MIN} min | token length {len(BOX_TOKEN)} (never printed) | Gemini key: {'yes' if os.environ.get('GOOGLE_API_KEY') else 'no'}")

## 7. Start the GPU servers (Surya + the vision LLM, each tested with a real request), the load balancers and the API

In [ ]:
LOGS = Path("logs"); LOGS.mkdir(exist_ok=True); PROCS = {}
def start(name, cmd, env=None, cwd=None):
    stop(name)
    log = open(LOGS / f"{name}.log", "a"); log.write(f"\n==== start {time.strftime('%H:%M:%S')}: {cmd}\n"); log.flush()
    PROCS[name] = subprocess.Popen(["bash", "-lc", cmd], stdout=log, stderr=subprocess.STDOUT, env=env, cwd=cwd, start_new_session=True)
def alive(name):
    p = PROCS.get(name); return p is not None and p.poll() is None
def stop(name):
    p = PROCS.pop(name, None)
    if p and p.poll() is None:
        try: os.killpg(p.pid, 15); p.wait(15)
        except Exception:
            try: os.killpg(p.pid, 9)
            except Exception: pass
def log_tail(name, n=25):
    f = LOGS / f"{name}.log"; return "\n".join(f.read_text(errors="replace").splitlines()[-n:]) if f.exists() else "(no log)"
def wait_http(url, timeout=60, every=2, ok=(200,)):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "box"}), timeout=5) as r:
                if r.status in ok: return True
        except Exception: pass
        time.sleep(every)
    return False

# llama.cpp releases daily and renames options (e.g. --no-mmap became --load-mode none): ask this build which it knows,
# and leave out optional flags it does not, instead of failing every start with "invalid argument"
LLAMA_HELP = (subprocess.run(f"{VLM_ENV} {VLM_BIN} --help", shell=True, capture_output=True, text=True, timeout=120).stdout
              if VLM_BIN else "")
def has_flag(flag):
    return not LLAMA_HELP or re.search(r"(^|[\s,`|])" + re.escape(flag) + r"([\s,=`|]|$)", LLAMA_HELP, re.M) is not None
NO_MMAP = "--load-mode none" if has_flag("--load-mode") else ("--no-mmap" if has_flag("--no-mmap") else "")   # weights go to the GPU, not RAM
OPTIONAL = ("--load-mode", "--no-mmap", "-fa", "--flash-attn", "-ctk", "-ctv", "--cache-type-k", "--cache-type-v", "--spec-type",
            "--image-max-tokens", "--reasoning-budget", "--threads", "--cache-ram", "--ctx-checkpoints")
NO_HOST_CACHE = "--cache-ram 0 --ctx-checkpoints 0"   # llama.cpp keeps up to 8 GB of old prompts in RAM per server by default: 4 servers filled the box
DROPPED = set()
def opt(flags):
    """the optional flags this llama-server build supports (each with its value); the others are left out and reported once"""
    out, toks, i = [], flags.split(), 0
    while i < len(toks):
        j = i + 1
        while j < len(toks) and not toks[j].startswith("-"):
            j += 1
        if toks[i] in OPTIONAL and not has_flag(toks[i]):
            if toks[i] not in DROPPED:
                DROPPED.add(toks[i]); print(f"   this llama-server has no {toks[i]}: left out")
        else:
            out += toks[i:j]
        i = j
    return " ".join(out)
def bad_args(name):
    """True if the server refused its command line: retrying with fewer slots cannot help"""
    return "invalid argument" in log_tail(name, 4) or "unknown argument" in log_tail(name, 4)
print("llama-server:", "help read," if LLAMA_HELP else "no --help output (all flags kept),", f"no-mmap via [{NO_MMAP or 'not available'}]")

SLOTS, VLM_PORTS = {}, []
def start_vlm(g, slots):
    port = 8081 + g
    while slots >= 1:
        ctx = slots * 12288                       # Surya's recommended per-slot context
        cmd = (f"CUDA_VISIBLE_DEVICES={g} {VLM_ENV} {VLM_BIN} -m {GGUF[0]} --mmproj {GGUF[1]} -ngl 99 {opt(NO_MMAP + ' --threads 1 ' + NO_HOST_CACHE)} "
               f"--host 127.0.0.1 --port {port} --parallel {slots} --ctx-size {ctx} --alias {VLM_ALIAS} --jinja --special")
        start(f"vlm{g}", cmd)
        t0 = time.time()
        while time.time() - t0 < 600 and alive(f"vlm{g}"):
            if wait_http(f"http://127.0.0.1:{port}/health", timeout=6, every=3): SLOTS[g] = slots; return port
        print(f"GPU {g}: server did not come up with {slots} slots:\n" + log_tail(f"vlm{g}", 6))
        if bad_args(f"vlm{g}"): stop(f"vlm{g}"); return None          # a flag problem: fewer slots will not help
        stop(f"vlm{g}"); slots //= 2
    return None
def start_vlms():
    ports, th = {}, []
    for g in range(len(gpus)):
        t = threading.Thread(target=lambda g=g: ports.__setitem__(g, start_vlm(g, VLM_SLOTS_PER_GPU))); t.start(); th.append(t)
    for t in th: t.join()
    return [p for _, p in sorted(ports.items()) if p]
def warm_vlm(port):
    """one OCR-like request (a page-sized image): the GPU kernels get compiled now, not during the first job"""
    import base64, io
    from PIL import Image, ImageDraw
    im = Image.new("RGB", (1240, 1754), "white"); d = ImageDraw.Draw(im)
    for k in range(30):
        d.text((80, 60 + 50 * k), f"Line {k + 1}: warm-up text", fill="black")
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=85)
    body = {"model": VLM_ALIAS, "temperature": 0, "max_tokens": 32, "messages": [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()}},
        {"type": "text", "text": "OCR this image."}]}]}
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
        urllib.request.urlopen(req, timeout=300).read()
    except Exception as e:
        print(f"   Surya warm-up on port {port} skipped: {str(e)[:120]}")

VLM_URL = ""
if USE_VLM and VLM_BIN and GGUF and gpus:
    t0 = time.time(); VLM_PORTS = start_vlms()
    th = [threading.Thread(target=warm_vlm, args=(p,)) for p in VLM_PORTS]
    for t in th: t.start()
    for t in th: t.join()
    if VLM_PORTS:
        start("lb", f"{sys.executable} mdbox/lb.py {VLM_LB_PORT} {' '.join(map(str, VLM_PORTS))}")
        if wait_http(f"http://127.0.0.1:{VLM_LB_PORT}/health", timeout=60):
            VLM_URL = f"http://127.0.0.1:{VLM_LB_PORT}/v1"
    print(f"VLM servers: {len(VLM_PORTS)} GPU(s), slots {SLOTS} in {time.time()-t0:.0f}s" if VLM_URL else "!!! no VLM server is running: CPU-only fast mode")

LLM_URL, LLM_SERVERS, LLM_RUN = "", {}, {}      # LLM_RUN[gpu] = (model path, mmproj path, flags, slots) that passed the test
def llm_ask(port, content, props, max_tokens):
    body = {"model": LLM_NAME, "temperature": 0, "max_tokens": max_tokens, "messages": [{"role": "user", "content": content}],
            "response_format": {"type": "json_schema", "json_schema": {"name": "answer", "strict": True, "schema": {
                "type": "object", "properties": {p: {"type": "string"} for p in props}, "required": list(props), "additionalProperties": False}}}}
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    r = json.load(urllib.request.urlopen(req, timeout=900))
    return json.loads(r["choices"][0]["message"]["content"]), (r.get("timings") or {}).get("predicted_per_second") or 0.0
def llm_test(port):
    """like Marker's requests (image + JSON schema). The first call also warms up (the driver JIT-compiles the GPU kernels);
    the answer must be exactly right, and the speed is measured on a warm, longer answer. Returns tokens/s or raises."""
    import base64, io
    from PIL import Image, ImageDraw
    im = Image.new("RGB", (220, 70), "white"); ImageDraw.Draw(im).text((20, 25), "TOTAL = 42", fill="black")
    buf = io.BytesIO(); im.save(buf, "PNG")
    img = [{"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()}},
           {"type": "text", "text": "Which number is written in the image? Answer with the digits only."}]
    for _ in range(2):
        ans, _ = llm_ask(port, img, ["number"], 32)
        if ans.get("number", "").strip() != "42":
            raise ValueError(f"wrong answer {ans!r} (expected 42)")
    page = Image.new("RGB", (1240, 1754), "white"); d = ImageDraw.Draw(page)    # a page-sized image too: its GPU kernels are
    for k in range(40):                                                          # compiled now, not during the first job
        d.text((80, 60 + 40 * k), f"Line {k + 1}: warm-up text for the vision model", fill="black")
    buf = io.BytesIO(); page.save(buf, "JPEG", quality=85)
    llm_ask(port, [{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()}},
                   {"type": "text", "text": "What is written on the first line?"}], ["text"], 24)
    _, tps = llm_ask(port, [{"type": "text", "text": "Write the numbers from 1 to 80, separated by commas."}], ["numbers"], 300)
    if tps < LLM_MIN_TOK_S:
        raise ValueError(f"too slow: {tps:.1f} tok/s < LLM_MIN_TOK_S={LLM_MIN_TOK_S}")
    return tps
def start_llm(g, model, mmproj, flags, slots):
    """one LLM server on GPU g; returns its port if it starts and passes llm_test"""
    port = 8181 + g
    cache = f"--cache-ram {int(LLM_HOST_CACHE_MB)} --ctx-checkpoints 0" if int(LLM_HOST_CACHE_MB or 0) > 0 else NO_HOST_CACHE
    extra = opt(f"{NO_MMAP} {cache} {flags} --threads 1 --image-max-tokens {LLM_IMAGE_MAX_TOKENS} --reasoning-budget 0")   # no thinking: Marker wants JSON
    start(f"llm{g}", f"CUDA_VISIBLE_DEVICES={g} {VLM_ENV} {VLM_BIN} -m {model} --mmproj {mmproj} -ngl 99 {extra} "
                     f"--host 127.0.0.1 --port {port} --parallel {slots} --ctx-size {slots * LLM_CTX_PER_SLOT} --alias {LLM_NAME} --jinja")
    t0 = time.time()
    while time.time() - t0 < 1200 and alive(f"llm{g}"):
        if wait_http(f"http://127.0.0.1:{port}/health", timeout=6, every=3):
            free = int(subprocess.run(["nvidia-smi", f"--id={g}", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                                      capture_output=True, text=True).stdout.strip() or 0)
            if free < LLM_MIN_FREE_MB:                # it runs, but Surya on the same GPU could run out of memory under load:
                print(f"   GPU {g}: {slots} slots leave {free} MiB GPU memory (< LLM_MIN_FREE_MB={LLM_MIN_FREE_MB}): trying fewer")
                stop(f"llm{g}"); return None          # (checked before the speed test: no time lost on a variant that can't stay)
            try:
                tps = llm_test(port)
                print(f"   GPU {g}: {LLM_NAME} {os.path.basename(model)} ok | {slots} slots, {tps:.1f} tok/s warm, {free} MiB free | {flags or 'plain'}")
                LLM_RUN[g] = (model, mmproj, flags, slots); return port
            except Exception as e:
                print(f"   GPU {g}: test failed: {str(e)[:200]}"); break
    print(f"   GPU {g}: not usable with [{flags or 'plain'}], {slots} slots:\n" + log_tail(f"llm{g}", 6)); stop(f"llm{g}")
    return None
def setup_llm(g):
    """best file x fastest variant that answers exactly and fast enough, on GPU g"""
    for f in LLM_FILES:
        try:
            model, mmproj = (hf_hub_download(LLM_REPO, x) for x in (f, LLM_MMPROJ))
        except Exception as e:
            print(f"   {f}: download failed:", str(e)[:300]); continue
        for flags, slots in LLM_VARIANTS:
            port = start_llm(g, model, mmproj, flags, slots)
            if port: return port
    return None
if LLM_LOCAL and VLM_BIN and gpus:
    from huggingface_hub import hf_hub_download
    t0 = time.time()
    th = [threading.Thread(target=lambda g=g: LLM_SERVERS.__setitem__(g, setup_llm(g))) for g in range(len(gpus))]
    for t in th: t.start()
    for t in th: t.join()
    LLM_SERVERS = {g: p for g, p in LLM_SERVERS.items() if p}
    if LLM_SERVERS:
        start("lb_llm", f"{sys.executable} mdbox/lb.py {LLM_LB_PORT} {' '.join(map(str, LLM_SERVERS.values()))}")
        if wait_http(f"http://127.0.0.1:{LLM_LB_PORT}/health", timeout=60): LLM_URL = f"http://127.0.0.1:{LLM_LB_PORT}/v1"
    print(f"vision LLM {LLM_NAME}: GPUs {sorted(LLM_SERVERS)} ready in {time.time()-t0:.0f}s" if LLM_URL
          else "!!! vision LLM not running: the LLM passes only work with a GOOGLE_API_KEY secret")

sys.path.insert(0, str(Path("mdbox").resolve()))
def api_env():
    env = os.environ.copy()
    defaults = dict(JOB_DEFAULTS, VLM_URL=VLM_URL, VLM_SLOTS=sum(SLOTS.values()) if VLM_URL else 0,
                    LLM_URL=LLM_URL, LLM_MODEL=LLM_NAME if LLM_URL else "", LLM_SLOTS=sum(r[3] for r in LLM_RUN.values()) if LLM_URL else 0, LLM_CTX=LLM_CTX_PER_SLOT)
    env.update(BOX_TOKEN=BOX_TOKEN, BOX_DEADLINE=str(DEADLINE), MD_ROOT=str(Path(RESULTS_DIR).resolve()), MD_MAX_UPLOAD_GB=str(MAX_UPLOAD_GB),
               MD_DEFAULTS=json.dumps(defaults), PYTHONPATH=str(Path("mdbox").resolve()))
    return env
def start_api():
    start("api", f"{sys.executable} -m uvicorn server:app --host 127.0.0.1 --port {API_PORT} --log-level warning", env=api_env(), cwd=str(Path("mdbox").resolve()))
    if not wait_http(f"http://127.0.0.1:{API_PORT}/ping", timeout=90): raise RuntimeError("API did not start:\n" + log_tail("api"))
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{API_PORT}/health", timeout=5); raise RuntimeError("API answered without a token!")
    except urllib.error.HTTPError as e: assert e.code == 401, e.code
def api_get(path):
    req = urllib.request.Request(f"http://127.0.0.1:{API_PORT}{path}", headers={"Authorization": f"Bearer {BOX_TOKEN}"})
    return json.loads(urllib.request.urlopen(req, timeout=10).read())
start_api()
t0 = time.time()
while time.time() - t0 < 180:
    w = api_get("/health")["workers"]
    if w and all(v == "ready" for v in w.values()): break
    time.sleep(3)
print("API up | token required | workers:", w)

## 8. Cloudflare tunnel (only the API is exposed, and it needs the token)

In [ ]:
def cloudflare_up(port=API_PORT, timeout=150):
    if not shutil.which("cloudflared"):
        subprocess.run('curl -fsSL "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb" -o /tmp/cf.deb && dpkg -i /tmp/cf.deb', shell=True, check=True, capture_output=True)
    start("cloudflared", f"cloudflared tunnel --no-autoupdate --protocol http2 --edge-ip-version 4 --url http://127.0.0.1:{port}")
    t0, url, log = time.time(), None, ""
    while time.time() - t0 < timeout:
        f = LOGS / "cloudflared.log"
        log = f.read_text(errors="replace").rsplit("==== start", 1)[-1] if f.exists() else ""
        urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
        if urls:
            url = urls[-1]
            if wait_http(url + "/ping", timeout=40, every=3): return url
        if not alive("cloudflared"): break
        time.sleep(2)
    raise RuntimeError(f"Cloudflare tunnel did not become reachable (url seen: {url}):\n" + log_tail("cloudflared"))
def announce(url):
    topic = "mbox-" + hashlib.sha256(("announce:" + BOX_TOKEN).encode()).hexdigest()[:40]
    t = int(time.time())
    msg = {"box": BOX_NAME, "url": url, "t": t, "deadline": int(DEADLINE), "sig": hmac.new(BOX_TOKEN.encode(), f"{BOX_NAME}|{url}|{t}".encode(), "sha256").hexdigest()}
    try:
        urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{topic}", data=json.dumps(msg).encode()), timeout=15).close()
        print("   address announced: your PC will find it automatically")
    except Exception as e: print(f"   could not announce ({type(e).__name__}); pass the url below with --url")
def network_up():
    url = cloudflare_up()
    if ANNOUNCE: announce(url)
    return url
PUBLIC_URL = network_up()
h = api_get("/health")
print("\n================ BOX READY ================")
print("  box   :", BOX_NAME, "| default mode:", h["default_mode"], "| LLM passes:", h["llm_model"] or ("Gemini" if h["llm"] else "off"))
print("  url   :", PUBLIC_URL)
print("  PC    : python md_client.py book.pdf          (add --local auto to also convert on your PC's CPUs)")
print("          (or add  --url", PUBLIC_URL, " if auto-find does not work)")
print("===========================================")

## 9. Serve until the deadline or until idle
Checks every 30 s, restarts the API, GPU servers or tunnel if they die, and stops the box after `IDLE_STOP_MIN` minutes without jobs so the weekly GPU quota is not wasted. Stop early with **Cancel run**.

In [ ]:
def api_ok(): return alive("api") and wait_http(f"http://127.0.0.1:{API_PORT}/ping", timeout=15)
def net_ok(): return alive("cloudflared") and wait_http(PUBLIC_URL + "/ping", timeout=20)
fails = {"api": 0, "net": 0}; last_beat = 0
try:
    while time.time() < DEADLINE:
        for name, ok in (("api", api_ok), ("net", net_ok)): fails[name] = 0 if ok() else fails[name] + 1
        for g in range(len(VLM_PORTS)):
            if not alive(f"vlm{g}"):
                print(time.strftime("%H:%M:%S"), f"GPU server {g} died; restarting\n" + log_tail(f"vlm{g}", 6)); start_vlm(g, SLOTS.get(g, 2))
        for g in LLM_SERVERS:
            if not alive(f"llm{g}"):
                print(time.strftime("%H:%M:%S"), f"vision LLM on GPU {g} died; restarting\n" + log_tail(f"llm{g}", 6)); start_llm(g, *LLM_RUN[g])
        if LLM_URL and not alive("lb_llm"): start("lb_llm", f"{sys.executable} mdbox/lb.py {LLM_LB_PORT} {' '.join(map(str, LLM_SERVERS.values()))}")
        if VLM_URL and not alive("lb"): start("lb", f"{sys.executable} mdbox/lb.py {VLM_LB_PORT} {' '.join(map(str, VLM_PORTS))}")
        if fails["api"] >= 2: print(time.strftime("%H:%M:%S"), "API down; restarting (job history in memory is lost)\n" + log_tail("api", 8)); start_api(); fails["api"] = 0
        if fails["net"] >= 2: print(time.strftime("%H:%M:%S"), "tunnel down; restarting"); PUBLIC_URL = network_up(); print("   new url:", PUBLIC_URL); fails["net"] = 0
        try: h = api_get("/health")
        except Exception: h = {}
        if IDLE_STOP_MIN and h.get("idle_s", 0) > IDLE_STOP_MIN * 60:
            print(f"{time.strftime('%H:%M:%S')} idle for {IDLE_STOP_MIN} min: stopping to save GPU quota (run the notebook again when needed)"); break
        if time.time() - last_beat > 300:
            last_beat = time.time()
            try: print(f"{time.strftime('%H:%M:%S')} alive | {(DEADLINE-time.time())/3600:.1f} h left | idle {h.get('idle_s')} s | RAM {h.get('memory', {}).get('used_gb')}/{h.get('memory', {}).get('limit_gb')} GB | jobs:", [(j['id'], j['status'], f"{j['pct']:.0f}%") for j in api_get('/jobs')[-5:]] or "none")
            except Exception as e: print("heartbeat:", repr(e)[:100])
        time.sleep(30)
finally:
    for n in ["cloudflared", "api", "lb", "lb_llm"] + [f"vlm{g}" for g in range(len(gpus))] + [f"llm{g}" for g in range(len(gpus))]: stop(n)
    print("box stopped after %.1f h" % ((time.time() - T0) / 3600))